# AI Technologies (Basic Course) · Notebook 07
## Topic 4. Machine learning and neural networks in the recognition of military objects

| Lesson | Type, duration | Title |
|---|---|---|
| **4.3** | Practical class, 2 h | Classification of thermal signatures by HOG + SVM: target — background |
| **4.4** | Practical class, 2 h | Transfer learning of MobileNet for the recognition of military vehicles and export of the model to ONNX |

Course: *AI Technologies (Basic Course)*, AIT-BC · Content module 2 · Kruty Heroes Military Institute of Telecommunications and Information Technology, Kyiv, 2026.

**How to work with this notebook**

1. Run the cells from top to bottom: *Runtime → Run all*, or `Shift+Enter` cell by cell. Section 0 is always run first; after a restart of the runtime it is run again.
2. Every block begins with a text cell: the purpose, the formulas, the parameters of the form, the expected result and the control question. The answers are written in the text cell **Report** at the end of each lesson.
3. The cells marked **(code shown)** are the practical skills of the lesson: read the code, change it and run it again. The blocks of a lesson use the results of the previous blocks of the same lesson.
4. Lesson 4.3 runs on the processor in 1–3 minutes per block. Lesson 4.4 trains a neural network: with a GPU (*Runtime → Change runtime type → T4 GPU*) one epoch takes seconds, on the processor of Colab minutes; block 4.4.3 prints the estimated time after the first epoch.
5. The results are saved in the course project on Google Drive (`outputs`, `models`, `project.json`), created in Notebook 02.

**Data.** Only real images are used, with the annotation of the authors of the data sets:
- **HIT-UAV** — thermal frames of a reconnaissance UAV (DJI Matrice M210 V2, camera Zenmuse XT2, 60–130 m, camera angle 30–90°, day and night) with boxes of persons and vehicles; the flight conditions are written in the file names;
- **Military vehicles** — 9444 photographs of 24 types of military vehicles (Notebook 06), joined into 7 groups by purpose.

**Contents**

- **0. Preparation** — 0.1 runtime, course library and data sets · 0.2 course project
- **Lesson 4.3 (practical class)**
  - 4.3.1 Thermal frames and candidates of the classical detector
  - 4.3.2 HOG features of thermal signatures
  - 4.3.3 Division into the training and the test part
  - 4.3.4 Training of the SVM and the threshold of the decision
  - 4.3.5 Analysis of the errors
  - 4.3.6 Saving and checking the verifier
- **Lesson 4.4 (practical class)**
  - 4.4.1 Preparation of the data and augmentation
  - 4.4.2 Stage 1: frozen network and a new output layer
  - 4.4.3 Stage 2: fine-tuning of the last blocks
  - 4.4.4 Evaluation of the model and Grad-CAM
  - 4.4.5 Export to ONNX and INT8 quantisation
  - 4.4.6 Recognition of a new image by the ONNX model
- **Summary**: self-check of Topic 4, data sources and licences

---
# 0. Preparation

### Block 0.1. Runtime, course library and data sets

**Purpose.** To prepare the runtime: the libraries, the course library `aitbc` (version 2.6.0 adds the candidates of the thermal detector, square fragments and Grad-CAM) and the data sets of the lessons.

**Parameters.**
- `CACHE_ON_DRIVE` — keep the downloaded files on Google Drive (they remain after the end of the session); otherwise they are kept on the disk of the session.
- `PREFETCH_ALL` — download now the two parquet files of military vehicles (166 MB) for Lesson 4.4; the thermal frames of Lesson 4.3 (about 30 MB) are downloaded in block 4.3.1.

**Output.** A table of versions, the computing device and the size of the cache.

In [ ]:
#@title 0.1 Runtime, course library and data sets (run once) { display-mode: "form" }
CACHE_ON_DRIVE = False #@param {type:"boolean"}
PREFETCH_ALL = True #@param {type:"boolean"}

import sys, os, platform, time, json, math, importlib, shutil, warnings
import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display
import sklearn
import skimage
import torch
import torchvision
pd.set_option("display.max_colwidth", 200)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
WORK_DIR = "/content" if IN_COLAB else os.getcwd()

LIB_SOURCE = r'''"""
aitbc - course library of "AI Technologies (Basic Course)" (AIT-BC).

Kruty Heroes Military Institute of Telecommunications and Information Technology,
Kyiv, 2026.

All demonstrations of the course use REAL open data sets recorded by UAVs and
ground vehicles. Only the files that a block needs are downloaded: large archives
are read selectively with HTTP range requests, and every file is cached in
DATA_ROOT (a Google Drive folder keeps the cache between sessions).

Data sets
---------
HIT-UAV   2898 thermal (LWIR) images 640x512 and 5 thermal videos. DJI Matrice
          M210 V2 with a Zenmuse XT2 camera (25 mm lens, field of view 25 x 20 deg),
          flight altitude 60-130 m, camera angle 30-90 deg, day and night. Boxes of
          Person, Car, Bicycle, OtherVehicle, DontCare.
          Suo J. et al. Scientific Data 10, 227 (2023). Licence CC BY 4.0.
Kust4K    4024 pixel-aligned visible (RGB) and thermal (TIR) image pairs 640x512
          from a hexacopter over urban roads, day and night; pixel labels of 8 classes.
          Scientific Data 12, 1701 (2025). Licence CC BY 4.0.
VisDrone  RGB images of drones over cities (VisDrone2019-DET validation part,
          548 images) with boxes of pedestrians, vehicles and other objects.
          Zhu P. et al. IEEE TPAMI 44(11), 7380-7399 (2022).
GOOSE     camera images of a ground vehicle on roads, field tracks, in forests and on
          a training area, with pixel labels of 64 classes (validation part, 962 images).
          IEEE ICRA 2024. Licence CC BY-SA 4.0.
M3OT      synchronised visible and thermal image sequences of two DJI Mavic 3T drones
          (100-120 m, 10 fps, 640x512, day, dusk, night) with vehicle tracks.
          Nie Z. et al. Scientific Data 12, 1927 (2025). Licence CC BY 4.0.

TII-RATM  frames 640x480 (125 fps), IMU 500 Hz and motion capture of an FPV racing quadrotor
          flown by human pilots in an indoor arena with racing gates (wide-angle camera
          155 x 115 deg). Bosello M. et al. IEEE RA-L 9(4), 3799-3806 (2024). Licence CC BY 4.0.
MilVeh    9444 photographs of 24 types of military vehicles collected from web sites
          (Kricheli J. S. et al., arXiv:2407.15192, 2024; Hugging Face
          leibnitz-lab/military_vehicles, licence declared: MIT).
SULAND    33771 frames 1440x1920 of video sequences of a camera over grass, leaves and
          gravel with surrogates of the surface landmines PFM-1 and PMA-2 (Italy and USA);
          Vivoli E. et al. Remote Sensing 16(4), 677 (2024); refined annotation SULAND_v2,
          Lekhak S. et al., arXiv:2607.28996 (2026). Licence CC BY 4.0.

Units: metres, seconds, degrees. Images are NumPy arrays H x W (grey) or H x W x 3 (RGB).
"""

__version__ = "2.6.0"

import csv
import io
import json
import math
import os
import struct
import time
import urllib.error
import urllib.request
import zlib
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import cv2

DATA_ROOT = os.environ.get("AITBC_DATA") or (
    "/content/aitbc_data" if os.path.isdir("/content") else os.path.abspath("aitbc_data"))


def set_data_root(path):
    """Set the folder of the download cache (for example a folder on Google Drive)."""
    global DATA_ROOT
    DATA_ROOT = os.path.abspath(path)
    os.makedirs(DATA_ROOT, exist_ok=True)
    return DATA_ROOT


def _cache(*parts):
    p = os.path.join(DATA_ROOT, *parts)
    os.makedirs(os.path.dirname(p), exist_ok=True)
    return p


def cache_size():
    """Number of files and size of the download cache, MB."""
    n, s = 0, 0
    for dp, _, fn in os.walk(DATA_ROOT):
        for f in fn:
            n += 1
            s += os.path.getsize(os.path.join(dp, f))
    return n, s / 1e6


# =============================================================================
# 1. Downloading
# =============================================================================
class _HTTPSRedirect(urllib.request.HTTPRedirectHandler):
    """Follow redirects, keep the Range header and upgrade http:// targets to https://."""

    def redirect_request(self, req, fp, code, msg, headers, newurl):
        if newurl.startswith("http://"):
            newurl = "https://" + newurl[len("http://"):]
        new = super().redirect_request(req, fp, code, msg, headers, newurl)
        if new is not None and req.has_header("Range"):
            new.add_header("Range", req.get_header("Range"))
        return new


_OPENER = urllib.request.build_opener(_HTTPSRedirect)
_OPENER.addheaders = [("User-Agent", "aitbc-course/2.0")]


def _urlopen(url, headers=None, timeout=60, retries=4):
    """Open url with retries; a new Request object is built for every attempt."""
    err = None
    for k in range(retries):
        try:
            return _OPENER.open(urllib.request.Request(url, headers=dict(headers or {})), timeout=timeout)
        except urllib.error.HTTPError as e:
            if e.code in (400, 401, 403, 404, 416):
                raise
            err = e
        except Exception as e:                      # network error: retry after a pause
            err = e
        time.sleep(1.5 * (k + 1))
    raise err


def _retry(transfer, retries=5):
    """Run a complete transfer (request and reading of the whole body) with retries: a broken connection,
    a read timeout or an incomplete body repeats the transfer after a pause; client errors are not repeated."""
    err = None
    for k in range(retries):
        try:
            return transfer()
        except urllib.error.HTTPError as e:
            if e.code in (400, 401, 403, 404, 416):
                raise
            err = e
        except Exception as e:                      # timeout, reset connection, incomplete read
            err = e
        time.sleep(2.0 * (k + 1))
    raise err


def _read_url(url, headers=None, timeout=60, min_bytes=0, retries=5):
    """Whole body of a response (e.g. of a range request) with retries; a body shorter than min_bytes is
    an incomplete transfer and is repeated."""
    def transfer():
        with _OPENER.open(urllib.request.Request(url, headers=dict(headers or {})), timeout=timeout) as r:
            data = r.read()
        if len(data) < min_bytes:
            raise IOError(f"incomplete transfer: {len(data)} of at least {min_bytes} bytes")
        return data
    return _retry(transfer, retries)


def fetch(url, path, timeout=120, retries=5):
    """Download url to path with retries; nothing is done when the file is already in the cache.
    The file appears under its name only after a complete transfer."""
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    tmp = path + ".part"

    def transfer():
        with _OPENER.open(urllib.request.Request(url), timeout=timeout) as r, open(tmp, "wb") as fh:
            expected = int(r.headers.get("Content-Length") or 0)
            n = 0
            while True:
                b = r.read(1 << 20)
                if not b:
                    break
                fh.write(b)
                n += len(b)
        if expected and n < expected:
            raise IOError(f"incomplete download of {url}: {n} of {expected} bytes")
        os.replace(tmp, path)
        return path
    return _retry(transfer, retries)


def fetch_many(jobs, threads=8):
    """jobs: list of (url, path); parallel download, returns the list of paths."""
    with ThreadPoolExecutor(threads) as ex:
        return list(ex.map(lambda j: fetch(*j), jobs))


class _RangeFile(io.RawIOBase):
    """Seekable read-only file over HTTP range requests (reads the directory of a remote zip)."""

    def __init__(self, url, chunk=1 << 17, timeout=60):
        self.url, self.chunk, self.timeout = url, chunk, timeout
        r = _urlopen(url, {"Range": "bytes=0-0"}, timeout)
        cr = r.headers.get("Content-Range")
        r.read()
        if r.status != 206 or not cr:
            raise IOError(f"the server of {url} does not support range requests")
        self.size = int(cr.split("/")[-1])
        self.pos, self.buf_start, self.buf = 0, 0, b""

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, off, whence=0):
        self.pos = off if whence == 0 else (self.pos + off if whence == 1 else self.size + off)
        return self.pos

    def read(self, n=-1):
        if n is None or n < 0:
            n = self.size - self.pos
        n = min(n, self.size - self.pos)
        if n <= 0:
            return b""
        if not (self.buf_start <= self.pos and self.pos + n <= self.buf_start + len(self.buf)):
            start = self.pos
            end = min(self.size, start + max(n, self.chunk)) - 1
            self.buf = _read_url(self.url, {"Range": f"bytes={start}-{end}"}, self.timeout, min_bytes=end - start + 1)
            self.buf_start = start
        off = self.pos - self.buf_start
        out = self.buf[off:off + n]
        self.pos += len(out)
        return out

    def readinto(self, b):
        d = self.read(len(b))
        b[:len(d)] = d
        return len(d)


class RemoteZip:
    """Selective reading of members of a large remote zip archive: only the central
    directory and the requested members are transferred; members are cached on disk."""

    def __init__(self, url, cache_name):
        import zipfile
        self.url, self.cache_name = url, cache_name
        idx_path = _cache("_index", cache_name + ".json")
        if os.path.exists(idx_path):
            with open(idx_path) as fh:
                self.index = json.load(fh)
        else:
            zf = zipfile.ZipFile(_RangeFile(url))
            self.index = {i.filename: (i.header_offset, i.compress_size, i.file_size, i.compress_type)
                          for i in zf.infolist() if not i.filename.endswith("/")}
            with open(idx_path, "w") as fh:
                json.dump(self.index, fh)

    def namelist(self):
        return list(self.index)

    def _fetch_member(self, name):
        path = _cache(self.cache_name, name)
        if os.path.exists(path):
            return path
        off, csize, usize, ctype = self.index[name]
        raw = _read_url(self.url, {"Range": f"bytes={off}-{off + 30 + 1024 + csize}"}, min_bytes=30 + csize)
        if raw[:4] != b"PK\x03\x04":
            raise IOError(f"bad local header of {name}")
        n, m = struct.unpack("<HH", raw[26:30])
        data = raw[30 + n + m: 30 + n + m + csize]
        if ctype == 8:
            data = zlib.decompressobj(-15).decompress(data)
        elif ctype != 0:
            raise IOError(f"unsupported compression {ctype} in {name}")
        if len(data) != usize:
            raise IOError(f"incomplete member {name}")
        with open(path + ".part", "wb") as fh:
            fh.write(data)
        os.replace(path + ".part", path)
        return path

    def get(self, names, threads=8):
        """Download members (a name or a list of names); returns local path(s)."""
        single = isinstance(names, str)
        names = [names] if single else list(names)
        with ThreadPoolExecutor(threads) as ex:
            paths = list(ex.map(self._fetch_member, names))
        return paths[0] if single else paths


def _rgb(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise IOError(f"cannot read {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


# =============================================================================
# 2. Data sets
# =============================================================================
class HITUAV:
    """HIT-UAV: thermal images and videos of a DJI Matrice M210 V2 with a Zenmuse XT2.

    File name T_HHH_AA_W_NNNNN: T = 0 day / 1 night, HHH = flight altitude, m,
    AA = camera angle, deg (90 = vertical view), W = 0 sunny / 1 rainy.
    Images are 8-bit white-hot (warm = bright) after the automatic gain control
    of the camera: brightness is a relative, not a calibrated, temperature."""

    PALETTE = "white-hot"
    BASE = "https://raw.githubusercontent.com/suojiashun/HIT-UAV-Infrared-Thermal-Dataset/main/"
    CLASSES = ("Person", "Car", "Bicycle", "OtherVehicle", "DontCare")
    COLORS = {"Person": (0, 229, 255), "Car": (255, 235, 59), "Bicycle": (118, 255, 3),
              "OtherVehicle": (255, 152, 0), "DontCare": (190, 190, 190)}
    CAMERA = dict(model="DJI Zenmuse XT2, uncooled VOx microbolometer", width=640, height=512,
                  focal_mm=25.0, pixel_pitch_um=17.0, hfov_deg=25.0, vfov_deg=20.0, band_um="7.5-13.5")
    EXAMPLES = ('1_60_30_0_00047.jpg', '0_60_60_0_01734.jpg', '1_70_90_0_07776.jpg', '0_100_50_0_08162.jpg', '1_120_90_0_06161.jpg', '0_130_40_0_05603.jpg', '1_90_70_0_04990.jpg', '0_110_80_0_08518.jpg')
    VIDEOS = {"60m-30_1.mov": "60 m, camera angle 30 deg: people on a sports ground",
              "60m-40_3.mov": "60 m, 40 deg: cars on a parking lot",
              "70m-90_1.mov": "70 m, 90 deg (vertical view): people on a square",
              "120m-30_3.mov": "120 m, 30 deg: road along buildings",
              "130m-60_3.mov": "130 m, 60 deg: parking lot and roads"}

    def __init__(self, splits=("train", "val", "test")):
        import pandas as pd
        imgs, objs = [], []
        jobs = [(self.BASE + f"normal_json/annotations/{sp}.json", _cache("hituav", "annotations", sp + ".json"))
                for sp in splits]
        for sp, path in zip(splits, fetch_many(jobs, 3)):
            with open(path) as fh:
                d = json.load(fh)
            cat = {c["id"]: c["name"] for c in d["categories"]}
            id2f = {}
            for im in d["images"]:
                fn = im.get("filename") or im.get("file_name")
                id2f[im["id"]] = fn
                t, h, a, w = fn.split("_")[:4]
                imgs.append(dict(file=fn, split=sp, night=t == "1", altitude=int(h), angle=int(a), rainy=w == "1"))
            for an in d.get("annotation", d.get("annotations", [])):
                x, y, w, h = an["bbox"]
                objs.append(dict(ann_id=an["id"], file=id2f[an["image_id"]], cls=cat[an["category_id"]],
                                 x1=float(x), y1=float(y), x2=float(x + w), y2=float(y + h)))
        # oriented boxes [xc, yc, w, h, angle]: true width and length of rotated objects
        rjobs = [(self.BASE + f"rotate_json/annotations/{sp}.json", _cache("hituav", "annotations", "rot_" + sp + ".json"))
                 for sp in splits]
        rot = {}
        for path in fetch_many(rjobs, 3):
            with open(path) as fh:
                d = json.load(fh)
            for an in d.get("annotations", d.get("annotation", [])):
                b = an["bbox"]
                rot[an["id"]] = (min(b[2], b[3]), max(b[2], b[3]), b[4] if len(b) > 4 else 0.0)
        self.images = pd.DataFrame(imgs).sort_values("file").reset_index(drop=True)
        o = pd.DataFrame(objs)
        o["w"], o["h"] = o.x2 - o.x1, o.y2 - o.y1
        o["width_px"] = o.ann_id.map(lambda i: rot.get(i, (np.nan,) * 3)[0])     # oriented box: short side
        o["length_px"] = o.ann_id.map(lambda i: rot.get(i, (np.nan,) * 3)[1])    # oriented box: long side
        self.objects = o.merge(self.images[["file", "night", "altitude", "angle"]], on="file", how="left")
        self._split = dict(zip(self.images.file, self.images.split))
        self._groups = {f: g for f, g in self.objects.groupby("file")}
        counts = self.objects.pivot_table(index="file", columns="cls", values="x1", aggfunc="count", fill_value=0)
        self.images = self.images.merge(counts, left_on="file", right_index=True, how="left").fillna(0)

    def _url(self, f):
        return self.BASE + f"normal_json/{self._split[f]}/{f}"

    def path(self, file):
        return fetch(self._url(file), _cache("hituav", self._split[file], file))

    def prefetch(self, files, threads=8):
        return fetch_many([(self._url(f), _cache("hituav", self._split[f], f)) for f in files], threads)

    def load(self, file):
        """Thermal image (H x W uint8) and the list of annotated objects."""
        img = cv2.imread(self.path(file), cv2.IMREAD_GRAYSCALE)
        g = self._groups.get(file)
        objs = [] if g is None else [dict(cls=r.cls, box=[r.x1, r.y1, r.x2, r.y2]) for r in g.itertuples()]
        return img, objs

    def video(self, name):
        return fetch(self.BASE + "video_sample/" + name, _cache("hituav", "video", name))

    @staticmethod
    def video_geometry(name):
        """Flight altitude, m, and camera angle, deg, from a video name such as '60m-30_1.mov'."""
        alt, rest = name.split("m-")
        return int(alt), int(rest.split("_")[0])

    def gsd(self, altitude, angle=90):
        """Ground sampling distance at the image centre, m/px (formula of Lesson 1.1)."""
        rng = altitude / max(math.sin(math.radians(angle)), 1e-3)
        return 2 * rng * math.tan(math.radians(self.CAMERA["hfov_deg"]) / 2) / self.CAMERA["width"]


class Kust4K:
    """Kust4K: pixel-aligned RGB and thermal (TIR) pairs 640x512 of a hexacopter
    HET-3021C over urban roads, day (id ...D) and night (id ...N), with pixel labels.

    The thermal images are pseudo-colour (palette from purple = cold to yellow = warm);
    tir_gray is their luminance, a relative thermal brightness. The data set contains
    frames with SIMULATED failures (visible channel set to zero, thermal channel
    blurred) made by its authors; they are excluded from the curated lists below."""

    PALETTE = "pseudo-colour (purple = cold, yellow = warm)"
    URL = "https://ndownloader.figshare.com/files/{}"
    FILES = {"rgb": 55979147, "tir": 55979165, "seg": 55979159}
    CLASSES = ("unlabelled", "road", "building", "motorcycle", "car", "truck", "tree", "person", "traffic facilities")
    COLORS = np.array([(0, 0, 0), (150, 150, 150), (229, 57, 53), (255, 152, 0), (255, 235, 59),
                       (236, 64, 122), (46, 160, 67), (0, 229, 255), (30, 136, 229)], np.uint8)
    OBJECTS = (3, 4, 5, 7)
    CAMERA = dict(platform="hexacopter HET-3021C", tir_focal_mm=9.1, tir_pixel_um=12.0,
                  rgb="1920x1080, f = 8 mm, registered to the thermal image and resized to 640x512")
    DAY = ("00720D", "00719D", "00031D", "01944D", "00851D", "00580D", "00990D", "00355D", "02094D", "02293D")
    NIGHT = ("02566N", "02807N", "02921N", "03135N", "03685N", "03895N", "03780N", "04004N", "02867N", "02599N")

    def __init__(self):
        self._z = {}

    def _zip(self, key):
        if key not in self._z:
            self._z[key] = RemoteZip(self.URL.format(self.FILES[key]), "kust4k_" + key)
        return self._z[key]

    def prefetch(self, ids, threads=6):
        names = [i + ".png" for i in ids]
        with ThreadPoolExecutor(3) as ex:
            list(ex.map(lambda k: self._zip(k).get(names, threads), ("rgb", "tir", "seg")))

    def pair(self, fid):
        """dict(id, night, rgb, tir, tir_gray, labels) for a frame id such as '00720D'."""
        self.prefetch([fid])
        name = fid + ".png"
        rgb = _rgb(self._zip("rgb").get(name))
        tir = _rgb(self._zip("tir").get(name))
        lab = cv2.imread(self._zip("seg").get(name), cv2.IMREAD_UNCHANGED)
        return dict(id=fid, night=fid.endswith("N"), rgb=rgb, tir=tir,
                    tir_gray=cv2.cvtColor(tir, cv2.COLOR_RGB2GRAY), labels=lab)


class VisDrone:
    """VisDrone2019-DET validation part: RGB images of drones with object boxes.
    Annotation line: x, y, w, h, score, category, truncation, occlusion."""

    URL = "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip"
    CLASSES = ("ignored region", "pedestrian", "people", "bicycle", "car", "van", "truck", "tricycle",
               "awning-tricycle", "bus", "motor", "others")
    COLORS = {"pedestrian": (0, 229, 255), "people": (0, 176, 255), "bicycle": (118, 255, 3), "car": (255, 235, 59),
              "van": (255, 193, 7), "truck": (255, 152, 0), "tricycle": (186, 104, 200),
              "awning-tricycle": (156, 39, 176), "bus": (244, 67, 54), "motor": (236, 64, 122)}

    CURATED = ('0000213_03920_d_0000243', '0000291_03801_d_0000887', '0000117_00112_d_0000087', '0000001_08414_d_0000013', '0000155_00401_d_0000001', '0000193_01705_d_0000112', '0000295_02400_d_0000033', '0000289_04601_d_0000834', '0000215_00000_d_0000256', '0000327_04001_d_0000731')

    def __init__(self):
        self.z = RemoteZip(self.URL, "visdrone_val")

    def names(self):
        return sorted(n.split("/")[-1][:-4] for n in self.z.namelist() if n.endswith(".jpg"))

    def _members(self, name):
        return [f"VisDrone2019-DET-val/images/{name}.jpg", f"VisDrone2019-DET-val/annotations/{name}.txt"]

    def prefetch(self, names, threads=8):
        self.z.get([m for n in names for m in self._members(n)], threads)

    def load(self, name):
        """RGB image and the list of objects (ignored regions and 'others' are skipped)."""
        pi, pa = self.z.get(self._members(name))
        objs = []
        with open(pa) as fh:
            for line in fh:
                v = line.strip().strip(",").split(",")
                if len(v) < 8:
                    continue
                x, y, w, h, _, c, trunc, occ = (int(float(s)) for s in v[:8])
                if c in (0, 11):
                    continue
                objs.append(dict(cls=self.CLASSES[c], box=[x, y, x + w, y + h], truncation=trunc, occlusion=occ))
        return _rgb(pi), objs


class M3OT:
    """M3OT: synchronised visible (RGB) and thermal (IR) image sequences of two DJI Mavic 3T
    drones flying at 100-120 m (10 frames per second, 640 x 512 px), day, dusk and night, with
    multi-object tracking annotation of vehicles (MOT format: frame, id, x, y, w, h, flag, class,
    confidence). The track identifiers are the same in the RGB and IR sequences of one drone.
    The two channels were aligned by the authors with an affine transform; a residual spatial
    offset of several pixels and a time offset of up to several frames remain (Lesson 1.4).

    The IR frames are 8-bit grey images in the BLACK-HOT palette (warm = dark). The data
    descriptor does not state the palette; it is established from the scene: in all nine
    sequences of drone 1 the road surface is darker than vegetation by day, at dusk and at night,
    while asphalt is, as a rule, the warmer surface (Lesson 1.3, task 1.3.4). Vehicles are
    bright, i.e. cooler than the road. to_white_hot() gives the white-hot convention of HIT-UAV."""

    PALETTE = "black-hot"
    URL = "https://ndownloader.figshare.com/files/52023875"
    CAMERA = dict(platform="DJI Mavic 3T", altitude_m="100-120", fps=10, width=640, height=512,
                  rgb="telephoto camera, f = 29.85 mm (35 mm equivalent 162 mm), frames registered to IR",
                  ir="uncooled thermal camera 640 x 512, f = 9.1 mm (35 mm equivalent 40 mm)")
    SEQUENCES = {"1-01": "day, suburban road", "1-02": "night, city crossroads", "1-03": "day, city road",
                 "1-04": "day, suburban highway", "1-05": "day, crossroads", "1-06": "dusk, highway",
                 "1-07": "day, crossroads", "1-08": "dusk, crossroads", "1-09": "day, city street with parked cars"}

    def __init__(self):
        self.z = RemoteZip(self.URL, "m3ot")
        self._seq = {}
        for n in self.z.namelist():
            parts = n.split("/")        # M3OT/<drone>/<rgb|ir>/<split>/<sequence>/img1/<frame>.PNG
            if len(parts) == 7 and parts[5] == "img1":
                self._seq.setdefault((parts[4].rstrip("T"), parts[2]), []).append(n)
        for k in self._seq:
            self._seq[k].sort()

    def sequences(self):
        return sorted({k[0] for k in self._seq})

    def n_frames(self, seq, channel="rgb"):
        return len(self._seq[(seq, channel)])

    def frame_names(self, seq, channel="rgb", start=0, count=None, step=1):
        names = self._seq[(seq, channel)]
        end = len(names) if count is None else min(len(names), start + count * step)
        return names[start:end:step]

    def prefetch(self, seq, channel="rgb", start=0, count=None, step=1, threads=8):
        return self.z.get(self.frame_names(seq, channel, start, count, step), threads)

    def frames(self, seq, channel="rgb", start=0, count=None, step=1):
        """List of RGB images (H x W x 3) of a sequence; IR frames are grey images in 3 equal channels."""
        return [_rgb(p) for p in self.prefetch(seq, channel, start, count, step)]

    def tracks(self, seq, channel="rgb"):
        """Annotation table: frame (from 1), id, x, y, w, h, cx, cy."""
        import pandas as pd
        names = self._seq[(seq, channel)]
        gt = names[0].rsplit("/img1/", 1)[0] + "/gt/gt.txt"
        g = pd.read_csv(self.z.get(gt), header=None, names=["frame", "id", "x", "y", "w", "h", "flag", "cls", "conf"])
        g["cx"], g["cy"] = g.x + g.w / 2, g.y + g.h / 2
        return g[["frame", "id", "x", "y", "w", "h", "cx", "cy"]]

    def objects(self, table, frame):
        """Objects of one frame in the format of draw_boxes."""
        t = table[table.frame == frame]
        return [dict(cls="vehicle", id=int(r.id), box=[r.x, r.y, r.x + r.w, r.y + r.h]) for r in t.itertuples()]


class GOOSE:
    """GOOSE: front camera of a ground vehicle (2048x1000) with pixel labels of 64 classes."""

    URL = "https://goose-dataset.de/storage/goose_2d_val.zip"
    TRAVERSABLE = ("asphalt", "cobble", "gravel", "soil", "low_grass", "moss", "leaves", "snow",
                   "road_marking", "pedestrian_crossing", "sidewalk", "bikeway")
    CONDITIONAL = ("high_grass", "crops", "debris", "tree_root")
    IGNORE = ("undefined", "sky", "ego_vehicle", "outlier")
    CURATED = (
        'labels/val/2022-07-22_flight/2022-07-22_flight__0078_1658494246011438339_labelids.png',
        'labels/val/2022-07-22_flight/2022-07-22_flight__0158_1658494465090456964_labelids.png',
        'labels/val/2022-08-30_siegertsbrunn_feldwege/2022-08-30_siegertsbrunn_feldwege__0528_1661860582736903436_labelids.png',
        'labels/val/2022-08-30_siegertsbrunn_feldwege/2022-08-30_siegertsbrunn_feldwege__0551_1661860697315889089_labelids.png',
        'labels/val/2022-09-21_garching_uebungsplatz_2/2022-09-21_garching_uebungsplatz_2__0002_1663755194931038715_labelids.png',
        'labels/val/2022-09-21_garching_uebungsplatz_2/2022-09-21_garching_uebungsplatz_2__0038_1663755353192423465_labelids.png',
        'labels/val/2023-03-03_garching_2/2023-03-03_garching_2__0114_1677850404544550420_labelids.png',
        'labels/val/2023-03-03_garching_2/2023-03-03_garching_2__0156_1677850669766430912_labelids.png',
        'labels/val/2022-12-07_aying_hills/2022-12-07_aying_hills__0007_1670420722571435228_labelids.png',
        'labels/val/2022-12-07_aying_hills/2022-12-07_aying_hills__0057_1670421449778652057_labelids.png',
        'labels/val/2022-12-07_aying_hills/2022-12-07_aying_hills__0037_1670421230548919820_labelids.png',
        'labels/val/2023-05-15_neubiberg_rain/2023-05-15_neubiberg_rain__0650_1684157928693659670_labelids.png',
    )

    def __init__(self):
        self.z = RemoteZip(self.URL, "goose_val")
        self.names, self.colors = {}, {}
        with open(self.z.get("goose_label_mapping.csv"), newline="") as fh:
            for row in csv.DictReader(fh):
                k = int(row["label_key"])
                h = row["hex"].lstrip("#")
                self.names[k] = row["class_name"]
                self.colors[k] = tuple(int(h[i:i + 2], 16) for i in (0, 2, 4))
        self.lut = np.zeros((256, 3), np.uint8)
        for k, c in self.colors.items():
            self.lut[k] = c
        self.id_of = {v: k for k, v in self.names.items()}

    def frames(self):
        return sorted(n for n in self.z.namelist() if n.endswith("_labelids.png"))

    @staticmethod
    def _image_member(label_member):
        return label_member.replace("labels/", "images/").replace("_labelids.png", "_windshield_vis.png")

    def prefetch(self, members, threads=6):
        self.z.get([m for lm in members for m in (self._image_member(lm), lm)], threads)

    def load(self, label_member, scale=0.5):
        """RGB image and label map (both resized by `scale`)."""
        pi, pl = self.z.get([self._image_member(label_member), label_member])
        img, lab = _rgb(pi), cv2.imread(pl, cv2.IMREAD_UNCHANGED)
        if scale != 1:
            w, h = int(img.shape[1] * scale), int(img.shape[0] * scale)
            img = cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA)
            lab = cv2.resize(lab, (w, h), interpolation=cv2.INTER_NEAREST)
        return img, lab

    def traversability(self, lab):
        """0 = ignored (sky, own vehicle), 1 = traversable, 2 = conditionally traversable, 3 = obstacle."""
        lut = np.full(256, 3, np.uint8)
        for group, v in ((self.TRAVERSABLE, 1), (self.CONDITIONAL, 2), (self.IGNORE, 0)):
            for n in group:
                if n in self.id_of:
                    lut[self.id_of[n]] = v
        return lut[lab]


# =============================================================================
# 3. Camera geometry
# =============================================================================
def gsd_nadir(altitude, hfov_deg, width_px):
    """Ground sampling distance of a vertical view, m/px: GSD = 2 H tan(alpha/2) / N."""
    return 2 * altitude * math.tan(math.radians(hfov_deg) / 2) / width_px


def hfov_from_sensor(focal_mm, pixel_um, width_px):
    """Horizontal field of view, deg, from the focal length and the pixel pitch."""
    return math.degrees(2 * math.atan(width_px * pixel_um * 1e-3 / 2 / focal_mm))


# =============================================================================
# 4. Degradations of the video signal (applied to real frames)
# =============================================================================
def fractal_noise(shape, rng, octaves=5, start=3, persistence=0.5):
    """Smooth random field in [0, 1] (sum of up-sampled random grids)."""
    h, w = shape
    out = np.zeros((h, w), np.float32)
    amp, m = 1.0, max(h, w)
    for o in range(octaves):
        n = start * (2 ** o)
        grid = rng.random((max(2, int(round(n * h / m)) + 1), max(2, int(round(n * w / m)) + 1)), dtype=np.float32)
        out += amp * cv2.resize(grid, (w, h), interpolation=cv2.INTER_CUBIC)
        amp *= persistence
    out -= out.min()
    return out / max(float(out.max()), 1e-6)


def _f(img):
    return img.astype(np.float32)


def _u8(x):
    return np.clip(x, 0, 255).astype(np.uint8)


def motion_blur(img, length=9.0, angle=0.0):
    """Linear motion blur: the point spread function is a line of `length` px at `angle` deg."""
    if length < 1.5:
        return img.copy()
    size = int(math.ceil(length)) | 1
    k = np.zeros((size * 4, size * 4), np.uint8)
    c = size * 2
    dx = math.cos(math.radians(angle)) * (length - 1) * 2
    dy = -math.sin(math.radians(angle)) * (length - 1) * 2
    cv2.line(k, (int(c - dx), int(c - dy)), (int(c + dx), int(c + dy)), 255, 4, cv2.LINE_AA)
    k = cv2.resize(k.astype(np.float32), (size, size), interpolation=cv2.INTER_AREA)
    return cv2.filter2D(img, -1, k / max(k.sum(), 1e-6), borderType=cv2.BORDER_REFLECT)


def defocus(img, sigma=1.5):
    return img.copy() if sigma <= 0 else cv2.GaussianBlur(img, (0, 0), sigma)


def add_noise(img, sigma=10.0, seed=None):
    rng = np.random.default_rng(seed)
    return _u8(_f(img) + rng.normal(0, sigma, img.shape))


def low_light(img, level=0.15, gain_max=8.0, seed=None):
    """Dark scene and automatic gain: light reduced by `level`, photon and read noise added."""
    if level >= 0.999:
        return img.copy()
    rng = np.random.default_rng(seed)
    lin = (_f(img) / 255.0) ** 2.2 * level
    full_well = 3000.0
    noisy = lin + rng.standard_normal(lin.shape).astype(np.float32) * np.sqrt(lin / full_well + (4.0 / full_well) ** 2)
    gain = float(np.clip(0.16 / max(float(lin.mean()), 1e-6), 1.0, gain_max))
    return _u8(np.clip(noisy * gain, 0, 1) ** (1 / 2.2) * 255)


def haze(img, density=0.4, airlight=200.0):
    """Uniform haze: I = J t + A (1 - t), transmission t = 1 - density."""
    t = 1.0 - float(np.clip(density, 0, 0.98))
    return _u8(_f(img) * t + airlight * (1 - t))


def smoke(img, density=0.5, seed=0, color=170.0, opacity=0.92):
    """Smoke: spatially variable veil."""
    if density <= 0:
        return img.copy()
    n = fractal_noise(img.shape[:2], np.random.default_rng(seed))
    m = np.clip((n - (1.0 - density)) * 3.0, 0, 1) * opacity
    if img.ndim == 3:
        m = m[..., None]
    return _u8(_f(img) * (1 - m) + color * m)


def jpeg(img, quality=30):
    """JPEG compression artefacts."""
    if quality >= 100:
        return img.copy()
    x = img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    _, buf = cv2.imencode(".jpg", x, [cv2.IMWRITE_JPEG_QUALITY, int(quality)])
    y = cv2.imdecode(buf, cv2.IMREAD_UNCHANGED)
    return y if img.ndim == 2 else cv2.cvtColor(y, cv2.COLOR_BGR2RGB)


def downscale(img, factor=0.5):
    """Loss of resolution: down-sample by `factor`, then up-sample back."""
    if factor >= 0.999:
        return img.copy()
    h, w = img.shape[:2]
    small = cv2.resize(img, (max(1, int(w * factor)), max(1, int(h * factor))), interpolation=cv2.INTER_AREA)
    return cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)


def rolling_shutter(img, amplitude=6.0, cycles=1.5, phase=0.0):
    """Rolling-shutter distortion: every row is shifted by a sinusoid of the row number."""
    if amplitude <= 0:
        return img.copy()
    h, w = img.shape[:2]
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    shift = amplitude * np.sin(2 * np.pi * cycles * yy / h + phase)
    return cv2.remap(img, xx + shift, yy, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)


def analog_interference(img, strength=0.5, seed=None):
    """Analog FPV video link with a weak or jammed signal: colour loss, snow, line
    noise, horizontal tearing (loss of synchronisation) and a rolling brightness bar."""
    if strength <= 0:
        return img.copy()
    rng = np.random.default_rng(seed)
    h = img.shape[0]
    x = _f(img)
    if x.ndim == 3:
        gray = x.mean(axis=2, keepdims=True)
        x = gray + (x - gray) * (1 - 0.85 * min(strength * 1.3, 1))
    x += rng.normal(0, 45 * strength, x.shape)
    for r in rng.integers(0, h, int(h * 0.06 * strength)):
        x[r] += rng.normal(70, 40) * strength
    if strength > 0.25:
        y0, band = rng.integers(0, h), int(h * 0.25 * strength)
        offs = np.cumsum(rng.normal(0, 3.0 * strength, band))
        for i in range(band):
            r = (y0 + i) % h
            x[r] = np.roll(x[r], int(offs[i] + 30 * strength), axis=0)
    bar = 1.0 + 0.3 * strength * np.sin(2 * np.pi * (np.arange(h, dtype=np.float32)[:, None] / h * 1.3 + rng.random()))
    return _u8(x * (bar if x.ndim == 2 else bar[..., None]))


def packet_loss(img, rate=0.05, block=16, seed=None):
    """Digital link with lost packets: damaged macroblock slices are filled with a smear
    of the row above or with a grey block."""
    if rate <= 0:
        return img.copy()
    rng = np.random.default_rng(seed)
    out = img.copy()
    gh, gw = img.shape[0] // block, img.shape[1] // block
    lost = rng.random((gh, gw)) < rate
    for r in range(gh):
        c = 0
        while c < gw:
            if lost[r, c]:
                run = int(rng.integers(1, max(2, gw // 3)))
                lost[r, c:c + run] = True
                c += run
            c += 1
    for r, c in zip(*np.nonzero(lost)):
        ys, xs = slice(r * block, (r + 1) * block), slice(c * block, (c + 1) * block)
        if r > 0 and rng.random() < 0.6:
            out[ys, xs] = out[r * block - 1:r * block, xs]
        else:
            out[ys, xs] = 128 if img.ndim == 2 else (60, 128, 60)
    return out


def degrade(img, sensor="visible", motion_px=0.0, motion_angle=0.0, defocus_sigma=0.0, light=1.0, noise=0.0,
            haze_density=0.0, smoke_density=0.0, rolling=0.0, scale=1.0, jpeg_quality=100, analog=0.0,
            packet=0.0, seed=0):
    """Chain of degradations in physical order: atmosphere -> optics and motion -> sensor ->
    resolution -> compression -> radio link. For the thermal sensor haze and smoke are
    attenuated (long-wave infrared passes thin smoke and haze) and low light does not apply."""
    x, ir = img, sensor == "thermal"
    if haze_density > 0:
        x = haze(x, haze_density * (0.25 if ir else 1.0), airlight=128 if ir else 200)
    if smoke_density > 0:
        x = smoke(x, smoke_density, seed=seed, color=128 if ir else 170, opacity=0.18 if ir else 0.92)
    if motion_px > 0:
        x = motion_blur(x, motion_px, motion_angle)
    if defocus_sigma > 0:
        x = defocus(x, defocus_sigma)
    if rolling > 0:
        x = rolling_shutter(x, rolling)
    if light < 1.0 and not ir:
        x = low_light(x, light, seed=seed)
    if noise > 0:
        x = add_noise(x, noise, seed=seed + 1)
    if scale < 1.0:
        x = downscale(x, scale)
    if jpeg_quality < 100:
        x = jpeg(x, jpeg_quality)
    if analog > 0:
        x = analog_interference(x, analog, seed=seed + 2)
    if packet > 0:
        x = packet_loss(x, packet, seed=seed + 3)
    return x


# =============================================================================
# 5. Measures, classical detection, tracking
# =============================================================================
def gray(img):
    return img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)


def psnr(ref, test):
    """Peak signal-to-noise ratio, dB."""
    mse = float(np.mean((_f(ref) - _f(test)) ** 2))
    return float("inf") if mse == 0 else 10 * math.log10(255.0 ** 2 / mse)


def ssim(ref, test):
    """Structural similarity index of grey levels (Gaussian window 11 x 11, sigma 1.5)."""
    a, b = _f(gray(ref)), _f(gray(test))
    c1, c2 = (0.01 * 255) ** 2, (0.03 * 255) ** 2
    blur = lambda z: cv2.GaussianBlur(z, (11, 11), 1.5)
    ma, mb = blur(a), blur(b)
    va, vb, cab = blur(a * a) - ma * ma, blur(b * b) - mb * mb, blur(a * b) - ma * mb
    return float((((2 * ma * mb + c1) * (2 * cab + c2)) / ((ma * ma + mb * mb + c1) * (va + vb + c2))).mean())


def sharpness(img):
    """Variance of the Laplacian: a measure of focus (small values = blurred image)."""
    return float(cv2.Laplacian(gray(img), cv2.CV_64F).var())


def to_white_hot(img, palette):
    """Thermal frame in the white-hot convention (warm = bright) as a 3-channel uint8 image.
    palette: "white-hot" (HIT-UAV), "black-hot" (M3OT, inverted here) or a pseudo-colour palette
    whose luminance grows with the temperature (Kust4K: purple = cold, yellow = warm)."""
    g = gray(img)
    if palette.startswith("black"):
        g = 255 - g
    return np.dstack([g, g, g])


def vegetation_mask(rgb, exg_min=0.12, v_min=35, erode_px=15):
    """Vegetation of a visible RGB frame by the excess-green index ExG = (2G - R - B) / (R + G + B).
    The mask is eroded by erode_px so that a residual offset between the visible and the thermal
    channel does not move it onto neighbouring surfaces."""
    f = rgb.astype(np.float32)
    r, g, b = f[..., 0], f[..., 1], f[..., 2]
    exg = (2 * g - r - b) / (r + g + b + 1e-6)
    m = ((exg > exg_min) & (f.max(axis=2) > v_min)).astype(np.uint8)
    if erode_px > 1:
        m = cv2.erode(m, np.ones((erode_px, erode_px), np.uint8))
    return m.astype(bool)


def surroundings_mask(shape, boxes, scale=1.0, gap=3):
    """Background ring around annotated objects (e.g. the road surface around vehicles): every box
    [x1, y1, x2, y2] is expanded by scale x its longer side; the boxes themselves, expanded by gap px,
    are excluded."""
    ring = np.zeros(shape[:2], np.uint8)
    core = np.zeros(shape[:2], np.uint8)
    for x1, y1, x2, y2 in boxes:
        s = int(round(scale * max(x2 - x1, y2 - y1)))
        cv2.rectangle(ring, (int(x1) - s, int(y1) - s), (int(x2) + s, int(y2) + s), 1, -1)
        cv2.rectangle(core, (int(x1) - gap, int(y1) - gap), (int(x2) + gap, int(y2) + gap), 1, -1)
    return (ring > 0) & (core == 0)


def palette_check(thermal, road, vegetation, margin=15, min_pixels=200):
    """Palette of a thermal frame established from the scene. In a road scene the asphalt is, as a
    rule, warmer than vegetation (by day it absorbs most of the solar radiation, at night it gives off
    the stored heat), therefore a road darker than vegetation indicates the black-hot palette
    (warm = dark) and a brighter road the white-hot convention (warm = bright). A shaded or wet road
    and sunlit tree crowns can reverse the order in single frames: check several frames."""
    g = gray(thermal)
    road = road & ~vegetation
    nr, nv = int(road.sum()), int(vegetation.sum())
    r = float(np.median(g[road])) if nr >= min_pixels else float("nan")
    v = float(np.median(g[vegetation])) if nv >= min_pixels else float("nan")
    if np.isnan(r) or np.isnan(v):
        verdict = "not determined: too few road or vegetation pixels"
    elif r < v - margin:
        verdict = "road darker than vegetation: black-hot (warm = dark)"
    elif r > v + margin:
        verdict = "road brighter than vegetation: white-hot (warm = bright)"
    else:
        verdict = f"difference below {margin} grey levels: not determined"
    return dict(road_median=r, vegetation_median=v, road_pixels=nr, vegetation_pixels=nv, verdict=verdict)


def signal_to_clutter(img, mask, exclude=None, gap=2, ring=8):
    """Signal-to-clutter ratio of an object: |mean of object - mean of background ring| /
    standard deviation of the background ring (grey levels)."""
    g = _f(gray(img))
    m = mask.astype(np.uint8)
    k1 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * gap + 1, 2 * gap + 1))
    k2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * (gap + ring) + 1, 2 * (gap + ring) + 1))
    bg = (cv2.dilate(m, k2) > 0) & (cv2.dilate(m, k1) == 0)
    if exclude is not None:
        bg &= ~exclude
    t, b = g[m > 0], g[bg]
    if t.size < 3 or b.size < 10:
        return float("nan")
    return abs(float(t.mean()) - float(b.mean())) / max(float(b.std()), 1.0)


def box_iou(a, b):
    """Intersection over union of two boxes [x1, y1, x2, y2]."""
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def boxes_from_labels(labels, class_ids, names, min_area=20):
    """Boxes of connected regions of the given classes in a label map."""
    out = []
    for c in class_ids:
        n, cc, stats, _ = cv2.connectedComponentsWithStats((labels == c).astype(np.uint8), connectivity=8)
        for i in range(1, n):
            x, y, w, h, a = stats[i]
            if a >= min_area:
                out.append(dict(cls=names[c], class_id=int(c), box=[float(x), float(y), float(x + w), float(y + h)],
                                area=int(a)))
    return out


def patch_labels(labels, tile, object_ids, names, min_px=10):
    """Patch classification from a label map: every tile receives the object class with
    the largest number of pixels in it, or 'background'."""
    rows, cols = labels.shape[0] // tile, labels.shape[1] // tile
    grid = np.empty((rows, cols), object)
    for r in range(rows):
        for c in range(cols):
            t = labels[r * tile:(r + 1) * tile, c * tile:(c + 1) * tile]
            n, k = max((int((t == k).sum()), k) for k in object_ids)
            grid[r, c] = names[k] if n >= min_px else "background"
    return grid


def detect_hotspots(img, k=5.0, se_px=21, min_area=4, max_area=5000):
    """Classical detector of small warm objects in a white-hot thermal image: white
    top-hat (local contrast) -> threshold mean + k std -> connected components.
    Returns boxes (N, 4) [x1, y1, x2, y2] and the binary mask."""
    g = cv2.GaussianBlur(gray(img), (3, 3), 0)
    se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (int(se_px) | 1, int(se_px) | 1))
    th = cv2.morphologyEx(g, cv2.MORPH_TOPHAT, se).astype(np.float32)
    mask = (th > th.mean() + k * th.std()).astype(np.uint8) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))
    n, _, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    boxes = [[x, y, x + w, y + h] for x, y, w, h, a in stats[1:] if min_area <= a <= max_area]
    return np.array(boxes, float).reshape(-1, 4), mask


def expected_size_px(gsd, size_m=0.45, margin_px=6.0):
    """Expected size of an object in an annotated box, px: size_m / GSD + margin_px.
    Defaults: nominal shoulder width of a person 0.45 m and a margin of 6 px (blur, thermal halo,
    annotation margin). For comparison, a least-squares fit to the HIT-UAV boxes of persons in
    vertical views gives d = 0.38 m and m = 6.6 px; block 1.1.1-B computes the fit for every class."""
    return size_m / gsd + margin_px


def detect_warm_objects(img, gsd, k=6.0, size_m=0.45, margin_px=6.0):
    """Hot-spot detector adapted to the ground sampling distance: the structuring element
    and the admissible blob areas follow the expected size of a person in pixels."""
    s = expected_size_px(gsd, size_m, margin_px)
    return detect_hotspots(img, k=k, se_px=int(max(7, round(2.5 * s))) | 1,
                           min_area=max(3, int((0.3 * s) ** 2)), max_area=int((2.0 * s) ** 2))


def match_detections(pred, objects, target_classes, pad=2.0):
    """A detection is correct when its centre lies inside a (padded) box of an object of a
    target class. Detections inside boxes of other annotated objects are counted apart.
    Returns dict(tp, fp, other, missed, recall, precision)."""
    tgt = [o for o in objects if o["cls"] in target_classes]
    oth = [o for o in objects if o["cls"] not in target_classes]
    used, tp, fp, other = set(), [], [], []
    inside = lambda o, cx, cy: (o["box"][0] - pad <= cx <= o["box"][2] + pad and o["box"][1] - pad <= cy <= o["box"][3] + pad)
    for b in pred:
        cx, cy = (b[0] + b[2]) / 2, (b[1] + b[3]) / 2
        hit = next((i for i, o in enumerate(tgt) if i not in used and inside(o, cx, cy)), None)
        if hit is not None:
            used.add(hit)
            tp.append(list(b))
        elif any(inside(o, cx, cy) for o in oth) or any(inside(o, cx, cy) for o in tgt):
            other.append(list(b))
        else:
            fp.append(list(b))
    n = len(tgt)
    return dict(tp=tp, fp=fp, other=other, missed=[o for i, o in enumerate(tgt) if i not in used],
                recall=len(tp) / n if n else float("nan"), precision=len(tp) / max(len(tp) + len(fp), 1))


def global_shift(prev, cur):
    """Translation of the image content from `prev` to `cur` (phase correlation), px."""
    (dx, dy), _ = cv2.phaseCorrelate(_f(gray(prev)), _f(gray(cur)))
    return dx, dy


class NearestNeighbourTracker:
    """Simple multi-object tracker: the detections of a frame are linked to the tracks of
    the previous frame by the smallest distance after compensation of the camera motion.
    A track is closed after `max_missed` frames without a detection."""

    def __init__(self, max_dist=12.0, max_missed=4):
        self.max_dist, self.max_missed = max_dist, max_missed
        self.tracks, self.closed, self.next_id = {}, {}, 1
        self.offset = np.zeros(2)                    # accumulated camera motion, px

    def update(self, centres, shift=(0.0, 0.0), frame=0):
        """centres: (N, 2) detections of the frame; shift: image motion since the previous frame.
        Returns the list of (track id, centre) of the frame."""
        self.offset += np.asarray(shift, float)
        pts = np.asarray(centres, float).reshape(-1, 2) - self.offset      # coordinates of the first frame
        ids = list(self.tracks)
        pairs = []
        for i, tid in enumerate(ids):
            last = self.tracks[tid]["hist"][-1][1]
            for j, p in enumerate(pts):
                d = float(np.hypot(*(p - last)))
                if d <= self.max_dist:
                    pairs.append((d, tid, j))
        pairs.sort()
        used_t, used_d, out = set(), set(), []
        for d, tid, j in pairs:
            if tid in used_t or j in used_d:
                continue
            used_t.add(tid)
            used_d.add(j)
            self.tracks[tid]["hist"].append((frame, pts[j]))
            self.tracks[tid]["missed"] = 0
            out.append((tid, pts[j] + self.offset))
        for tid in ids:
            if tid not in used_t:
                self.tracks[tid]["missed"] += 1
                if self.tracks[tid]["missed"] > self.max_missed:
                    self.closed[tid] = self.tracks.pop(tid)
        for j, p in enumerate(pts):
            if j not in used_d:
                self.tracks[self.next_id] = dict(hist=[(frame, p)], missed=0)
                out.append((self.next_id, p + self.offset))
                self.next_id += 1
        return out

    def history(self, tid):
        """Trajectory of a track in the coordinates of the current frame."""
        t = self.tracks.get(tid) or self.closed.get(tid)
        return np.array([p for _, p in t["hist"]]) + self.offset


def track_warm_objects(frames, gsd, fps, k=5.0, max_speed_ms=2.5, max_missed=4, min_len=12, min_disp_m=2.0):
    """Detection and tracking of warm objects in a thermal video:
    detect_warm_objects -> camera motion by phase correlation -> NearestNeighbourTracker.
    A track is reported as a moving object when it lasts at least `min_len` frames and its
    start and end points are at least `min_disp_m` apart on the ground.
    Returns (tracker, moving: {id: trajectory (K, 2) in the last frame}, rows: list of
    (frame, track id, x, y), detections per frame)."""
    tr = NearestNeighbourTracker(max_dist=max(5.0, max_speed_ms / gsd / max(fps, 1e-6) + 3.0), max_missed=max_missed)
    prev, rows, n_det = None, [], []
    for i, f in enumerate(frames):
        g = gray(f)
        b, _ = detect_warm_objects(g, gsd, k=k)
        c = np.c_[(b[:, 0] + b[:, 2]) / 2, (b[:, 1] + b[:, 3]) / 2] if len(b) else np.zeros((0, 2))
        shift = global_shift(prev, g) if prev is not None else (0.0, 0.0)
        for tid, p in tr.update(c, shift, i):
            rows.append((i, tid, float(p[0]), float(p[1])))
        n_det.append(len(b))
        prev = g
    moving = {}
    for tid, t in {**tr.closed, **tr.tracks}.items():
        pts = np.array([p for _, p in t["hist"]])
        if len(pts) >= min_len and float(np.hypot(*(pts[-1] - pts[0]))) * gsd >= min_disp_m:
            moving[tid] = pts + tr.offset
    return tr, moving, rows, n_det


def video_info(path):
    """Properties of a video file read by OpenCV."""
    cap = cv2.VideoCapture(path)
    info = dict(path=path, opened=cap.isOpened(), fps=cap.get(cv2.CAP_PROP_FPS),
                frames=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
                height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
                codec=int(cap.get(cv2.CAP_PROP_FOURCC)).to_bytes(4, "little").decode("latin-1", "replace"))
    cap.release()
    info["duration_s"] = info["frames"] / info["fps"] if info["fps"] else float("nan")
    info["size_mb"] = os.path.getsize(path) / 1e6 if os.path.exists(path) else float("nan")
    return info


def iter_video(path, every_n=1, max_frames=None, start=0):
    """Generator of (frame number, timestamp s, RGB frame) of every n-th frame of a video:
    frames are decoded one by one, so a long video does not fill the memory."""
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 1.0
    if start:
        cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    k, n_out = start, 0
    try:
        while True:
            ok, f = cap.read()
            if not ok or (max_frames is not None and n_out >= max_frames):
                break
            if (k - start) % every_n == 0:
                yield k, k / fps, cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
                n_out += 1
            k += 1
    finally:
        cap.release()


def write_video(path, frames, fps, codec="mp4v"):
    """Write RGB or grey frames to a video file; returns the path. Codec 'mp4v' (.mp4) works in
    Colab and in most OpenCV builds; 'MJPG' (.avi) stores every frame as JPEG."""
    h, w = frames[0].shape[:2]
    wr = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*codec), float(fps), (w, h))
    if not wr.isOpened():
        raise IOError(f"OpenCV cannot write the codec {codec}")
    for f in frames:
        f = to_rgb(f)
        if f.shape[:2] != (h, w):
            f = cv2.resize(f, (w, h))
        wr.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
    wr.release()
    return path


def contact_sheet(frames, cols=6, width=1600, labels=None, font=0.5):
    """Grid of frames for a quick visual review of a sequence."""
    n = len(frames)
    rows = int(math.ceil(n / cols))
    tw = width // cols
    th = int(round(tw * frames[0].shape[0] / frames[0].shape[1]))
    sheet = np.zeros((rows * th, cols * tw, 3), np.uint8)
    for i, f in enumerate(frames):
        r, c = divmod(i, cols)
        t = cv2.resize(to_rgb(f), (tw, th), interpolation=cv2.INTER_AREA)
        if labels is not None:
            cv2.rectangle(t, (0, 0), (tw, 18), (0, 0, 0), -1)
            cv2.putText(t, str(labels[i]), (4, 13), cv2.FONT_HERSHEY_SIMPLEX, font * 0.8, (255, 255, 0), 1, cv2.LINE_AA)
        sheet[r * th:(r + 1) * th, c * tw:(c + 1) * tw] = t
    return sheet


def frame_difference(a, b):
    """Mean absolute difference of the grey levels of two frames."""
    return float(np.mean(np.abs(_f(gray(a)) - _f(gray(b)))))


def grad_magnitude(img, sigma=1.5):
    """Normalised gradient magnitude; edges of a visible and a thermal image of the same scene
    are similar even when their brightness is not, so it is used to compare the two channels."""
    g = cv2.GaussianBlur(_f(gray(img)), (0, 0), sigma)
    m = np.hypot(cv2.Sobel(g, cv2.CV_32F, 1, 0), cv2.Sobel(g, cv2.CV_32F, 0, 1))
    return (m - m.mean()) / (m.std() + 1e-6)


def channel_shift(img_a, img_b):
    """Translation between two images of different sensors (phase correlation of gradient
    magnitudes): (dx, dy, response). The response is low when the images have little common structure."""
    a, b = grad_magnitude(img_a), grad_magnitude(img_b)
    win = cv2.createHanningWindow((a.shape[1], a.shape[0]), cv2.CV_32F)
    (dx, dy), resp = cv2.phaseCorrelate(a, b, win)
    return float(dx), float(dy), float(resp)


def track_offsets(table_a, table_b, lag=0):
    """Offsets of the centres of the same track (same id) in two annotated channels:
    channel b at frame t is compared with channel a at frame t + lag. Returns arrays dx, dy."""
    bb = table_b[["id", "frame", "cx", "cy"]].copy()
    bb["frame"] = bb["frame"] + lag
    m = bb.merge(table_a[["id", "frame", "cx", "cy"]], on=["id", "frame"], suffixes=("_b", "_a"))
    return (m.cx_b - m.cx_a).values, (m.cy_b - m.cy_a).values


def read_video(path, step=1, max_frames=None, start=0):
    """Frames of a video file (BGR from OpenCV converted to RGB) and the frame rate."""
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    frames, k = [], 0
    while True:
        ok, f = cap.read()
        if not ok or (max_frames and len(frames) >= max_frames):
            break
        if k % step == 0:
            frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
        k += 1
    cap.release()
    return frames, fps


# =============================================================================
# 6. Visualisation
# =============================================================================
def to_rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_GRAY2RGB) if img.ndim == 2 else img


def colorize(labels, colors):
    """Label map -> RGB image with the colour table `colors` (array K x 3)."""
    return np.asarray(colors, np.uint8)[np.clip(labels, 0, len(colors) - 1)]


def overlay(img, color_img, alpha=0.5, mask=None):
    base, col = _f(to_rgb(img)), _f(color_img)
    a = np.full(base.shape[:2], alpha, np.float32) if mask is None else mask.astype(np.float32) * alpha
    return _u8(base * (1 - a[..., None]) + col * a[..., None])


def draw_boxes(img, objects, colors=None, labels=True, scores=None, ids=None, thickness=1, font=0.4, color=None):
    """Draw boxes of objects (dicts with 'box' and 'cls')."""
    out = to_rgb(img).copy()
    for i, o in enumerate(objects):
        x1, y1, x2, y2 = [int(round(v)) for v in o["box"]]
        c = color if color is not None else (colors or {}).get(o["cls"], (255, 255, 0))
        c = tuple(int(v) for v in c)
        cv2.rectangle(out, (x1, y1), (x2, y2), c, thickness, cv2.LINE_AA)
        txt = []
        if labels:
            txt.append(str(o["cls"]))
        if ids is not None:
            txt.append(f"#{ids[i]}")
        if scores is not None:
            txt.append(f"{scores[i]:.2f}")
        if txt:
            s = " ".join(txt)
            (tw, th), _ = cv2.getTextSize(s, cv2.FONT_HERSHEY_SIMPLEX, font, 1)
            y0 = max(y1 - 3, th + 2)
            cv2.rectangle(out, (x1, y0 - th - 2), (x1 + tw + 2, y0 + 2), c, -1)
            cv2.putText(out, s, (x1 + 1, y0), cv2.FONT_HERSHEY_SIMPLEX, font, (0, 0, 0), 1, cv2.LINE_AA)
    return out


def legend(ax, names, colors, loc="lower right", fontsize=7, ncol=1):
    import matplotlib.patches as mpatches
    ax.legend(handles=[mpatches.Patch(color=np.array(c) / 255.0, label=n) for n, c in zip(names, colors)],
              loc=loc, fontsize=fontsize, ncol=ncol, framealpha=0.85)


def setup_plots():
    import matplotlib.pyplot as plt
    plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 10, "axes.labelsize": 9, "xtick.labelsize": 8,
                         "ytick.labelsize": 8, "legend.fontsize": 8, "figure.facecolor": "white"})


def catalogue():
    """Table of the data sets of the course."""
    return [
        ("HIT-UAV", "UAV DJI M210 V2, thermal camera Zenmuse XT2 (640x512, 25 x 20 deg)",
         "2898 thermal images, 5 thermal videos; boxes of persons and vehicles; 60-130 m, day and night",
         "CC BY 4.0", "Suo J. et al., Scientific Data 10, 227 (2023)",
         "https://github.com/suojiashun/HIT-UAV-Infrared-Thermal-Dataset"),
        ("Kust4K", "UAV hexacopter HET-3021C, visible + thermal cameras",
         "4024 pixel-aligned RGB-thermal pairs 640x512 over urban roads, day and night; pixel labels of 8 classes",
         "CC BY 4.0", "Scientific Data 12, 1701 (2025)", "https://doi.org/10.6084/m9.figshare.29476610"),
        ("VisDrone2019-DET (val)", "camera drones of different types, 14 Chinese cities",
         "548 RGB images; boxes of pedestrians, cars, vans, trucks, buses, motors and others",
         "research use, see the data set site", "Zhu P. et al., IEEE TPAMI 44(11), 7380-7399 (2022)",
         "https://github.com/VisDrone/VisDrone-Dataset"),
        ("GOOSE (val)", "ground vehicle, front RGB camera 2048x1000",
         "962 images of roads, field tracks, forests and a training area; pixel labels of 64 classes",
         "CC BY-SA 4.0", "IEEE ICRA 2024 (arXiv:2310.16788)", "https://goose-dataset.de"),
        ("Military vehicles", "photographs from web sites (ground, side and oblique views)",
         "9444 images of 24 types of military vehicles, 7 groups", "MIT (declared by the authors)",
         "Kricheli J. S. et al., arXiv:2407.15192 (2024)", "https://huggingface.co/datasets/leibnitz-lab/military_vehicles"),
        ("SULAND_v2", "handheld camera iPhone 13 moved over the ground (view of a ground robot)",
         "33771 frames 1440x1920 of 55 sequences; boxes of surrogates of the landmines PFM-1 and PMA-2",
         "CC BY 4.0", "Vivoli E. et al., Remote Sensing 16(4), 677 (2024); Lekhak S. et al., arXiv:2607.28996 (2026)",
         "https://huggingface.co/datasets/SagarLekhak/SULAND_v2_RGB_Surface_Landmine_Dataset"),
    ]


# =====================================================================================
# Topic 2. Digital image, video frame formation and preprocessing
# =====================================================================================
def quantize(img, bits):
    """Brightness quantised to `bits` bits per channel (2**bits levels), shown again in 0...255."""
    step = 256.0 / (2 ** int(bits))
    return np.clip(np.floor(img.astype(np.float32) / step) * step + step / 2, 0, 255).astype(np.uint8)


def entropy(img):
    """Entropy of the brightness histogram, bit (8 bit = all 256 levels used equally)."""
    hist = np.bincount(gray(img).ravel(), minlength=256).astype(float)
    p = hist[hist > 0] / hist.sum()
    return float(-(p * np.log2(p)).sum())


def rms_contrast(img):
    """RMS contrast: standard deviation of the grey levels."""
    return float(_f(gray(img)).std())


def average_gradient(img):
    """Average gradient: mean of sqrt((gx^2 + gy^2) / 2) of the grey levels (central differences)."""
    g = _f(gray(img))
    gx = cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=1)
    gy = cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=1)
    return float(np.sqrt((gx ** 2 + gy ** 2) / 2.0).mean())


def hsv_mask(rgb, h=(35, 85), s=(40, 255), v=(40, 255)):
    """Pixels whose hue (degrees, 0...360), saturation and value (0...255) lie in the given ranges.
    OpenCV stores the hue of 8-bit images as degrees / 2 (0...179); a range with h[0] > h[1]
    passes through 0 degrees (red)."""
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    hh = hsv[..., 0].astype(np.int32) * 2
    mh = ((hh >= h[0]) & (hh <= h[1])) if h[0] <= h[1] else ((hh >= h[0]) | (hh <= h[1]))
    return mh & (hsv[..., 1] >= s[0]) & (hsv[..., 1] <= s[1]) & (hsv[..., 2] >= v[0]) & (hsv[..., 2] <= v[1])


def mask_scores(pred, truth, valid=None):
    """IoU, precision and recall of a binary mask against a reference mask; pixels outside `valid` are ignored."""
    if valid is not None:
        pred, truth = pred & valid, truth & valid
    tp = int((pred & truth).sum())
    fp = int((pred & ~truth).sum())
    fn = int((~pred & truth).sum())
    return dict(iou=tp / max(tp + fp + fn, 1), precision=tp / max(tp + fp, 1), recall=tp / max(tp + fn, 1))


def box_mask(shape, box):
    """Binary mask of a box [x1, y1, x2, y2]."""
    m = np.zeros(shape[:2], bool)
    x1, y1, x2, y2 = (int(round(v)) for v in box)
    m[max(0, y1):max(0, y2), max(0, x1):max(0, x2)] = True
    return m


def colour_planes(rgb, thermal=None):
    """8-bit planes of the colour models of a frame: R, G, B, grey (luma Y = 0.299 R + 0.587 G + 0.114 B),
    H (degrees / 2), S, V and, if given, the thermal channel."""
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    planes = {"R": rgb[..., 0], "G": rgb[..., 1], "B": rgb[..., 2], "grey": cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY),
              "H": hsv[..., 0], "S": hsv[..., 1], "V": hsv[..., 2]}
    if thermal is not None:
        planes["thermal"] = gray(thermal)
    return planes


def wide_angle_lens(img, barrel=0.2, vignetting=0.0):
    """Wide-angle lens of an FPV camera: barrel distortion and vignetting. The output pixel at the
    normalised radius r (r = 1 in the corners) shows the point of the ideal image at r (1 + b r^2) / (1 + b):
    the corners stay in place, the centre is magnified by 1 + b, straight lines bend outwards.
    OpenCV describes the same distortion with a negative coefficient k1 of its camera model.
    Vignetting: the brightness is multiplied by 1 - v r^2."""
    h, w = img.shape[:2]
    cx, cy = (w - 1) / 2.0, (h - 1) / 2.0
    rmax = math.hypot(cx, cy)
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    dx, dy = (xx - cx) / rmax, (yy - cy) / rmax
    r2 = dx * dx + dy * dy
    f = (1 + barrel * r2) / (1 + barrel)
    out = cv2.remap(img, (cx + dx * f * rmax).astype(np.float32), (cy + dy * f * rmax).astype(np.float32),
                    cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    if vignetting > 0:
        v = np.clip(1 - vignetting * r2, 0, 1)
        out = _u8(_f(out) * (v if out.ndim == 2 else v[..., None]))
    return out


def free_space_loss_db(distance_m, freq_mhz):
    """Free-space path loss, dB: 20 lg(d, km) + 20 lg(f, MHz) + 32.44."""
    return 20 * math.log10(max(distance_m, 1.0) / 1000.0) + 20 * math.log10(freq_mhz) + 32.44


def link_budget(p_tx_mw, distance_m, freq_mhz=5800.0, g_tx_dbi=2.0, g_rx_dbi=2.0, extra_loss_db=0.0,
                bandwidth_mhz=20.0, noise_figure_db=6.0, interference_db=0.0):
    """Line-of-sight radio link: received power P_rx = P_tx + G_tx + G_rx - L_fs - L_add, dBm; noise power
    N = -174 + 10 lg(B, Hz) + NF, dBm, raised by `interference_db` (jamming); carrier-to-noise ratio CNR = P_rx - N."""
    p_tx = 10 * math.log10(p_tx_mw)
    fspl = free_space_loss_db(distance_m, freq_mhz)
    p_rx = p_tx + g_tx_dbi + g_rx_dbi - fspl - extra_loss_db
    noise = -174.0 + 10 * math.log10(bandwidth_mhz * 1e6) + noise_figure_db + interference_db
    return dict(p_tx_dbm=p_tx, fspl_db=fspl, p_rx_dbm=p_rx, noise_dbm=noise, cnr_db=p_rx - noise)


def analog_link_frame(img, cnr_db, threshold_db=10.0, fm_gain_db=25.0, seed=0):
    """Frame at the output of an analog FM video receiver (ILLUSTRATIVE model of the course, not a standard):
    above the FM threshold the picture has Gaussian noise with SNR = CNR + fm_gain_db; below the threshold
    impulse noise ("sparkles"), loss of colour and loss of line synchronisation grow with the deficit
    threshold - CNR; at a deficit of 6 dB and more the picture is replaced by noise."""
    rng = np.random.default_rng(seed)
    x = _f(img)
    x = x + rng.normal(0, 255.0 * 10 ** (-(cnr_db + fm_gain_db) / 20.0), x.shape)
    deficit = threshold_db - cnr_db
    if deficit >= 6:
        return _u8(rng.normal(128, 60, img.shape))
    if deficit > 0:
        imp = rng.random(x.shape[:2]) < min(0.5, 0.012 * deficit ** 2)
        vals = rng.choice(np.array([0.0, 255.0], np.float32), size=int(imp.sum()))
        x[imp] = vals[:, None] if x.ndim == 3 else vals
        return analog_interference(_u8(x), strength=min(1.0, deficit / 6.0), seed=seed)
    return _u8(x)


def jpeg_for_budget(img, bytes_per_frame):
    """Highest JPEG quality (5...95) whose file of this frame fits into `bytes_per_frame`: intra-frame
    equivalent of a digital video link with a given bit rate. Returns (decoded frame, quality, bytes)."""
    bgr = img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    best = None
    lo, hi = 5, 95
    while lo <= hi:
        q = (lo + hi) // 2
        ok, buf = cv2.imencode(".jpg", bgr, [cv2.IMWRITE_JPEG_QUALITY, q])
        if len(buf) <= bytes_per_frame:
            best, lo = (q, buf), q + 1
        else:
            hi = q - 1
    if best is None:
        ok, buf = cv2.imencode(".jpg", bgr, [cv2.IMWRITE_JPEG_QUALITY, 5])
        best = (5, buf)
    dec = cv2.imdecode(best[1], cv2.IMREAD_UNCHANGED)
    return (dec if dec.ndim == 2 else cv2.cvtColor(dec, cv2.COLOR_BGR2RGB)), best[0], len(best[1])


def digital_packet_loss_rate(cnr_db, required_db=8.0, width_db=0.4):
    """Packet loss rate of a digital link near its sensitivity (ILLUSTRATIVE logistic model of the "cliff
    effect"): 0.5 at CNR = required_db, 0.1 % at a margin of about 2.8 dB."""
    return float(1.0 / (1.0 + math.exp((cnr_db - required_db) / width_db)))


def noise_sigma(img, exclude_edges=0.1):
    """Standard deviation of the noise of a frame by the method of Immerkaer (1996): the frame is filtered with
    the mask [[1,-2,1],[-2,4,-2],[1,-2,1]], which suppresses the smooth structure of the scene, and
    sigma = sqrt(pi/2) / 6 * mean |filtered|. Edges and fine texture increase the estimate, therefore the
    share `exclude_edges` of the pixels with the strongest gradient of the smoothed frame is excluded.
    For a colour frame the estimate is the mean over the R, G, B channels (noise of one channel)."""
    if img.ndim == 3 and not (np.array_equal(img[..., 0], img[..., 1]) and np.array_equal(img[..., 1], img[..., 2])):
        return float(np.mean([noise_sigma(img[..., c], exclude_edges) for c in range(3)]))
    g = _f(gray(img))
    k = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], np.float32)
    r = np.abs(cv2.filter2D(g, -1, k))[1:-1, 1:-1]
    if exclude_edges > 0:
        sm = cv2.GaussianBlur(g, (0, 0), 2.0)           # edges of the scene, not of the noise
        gx = cv2.Sobel(sm, cv2.CV_32F, 1, 0, ksize=3)[1:-1, 1:-1]
        gy = cv2.Sobel(sm, cv2.CV_32F, 0, 1, ksize=3)[1:-1, 1:-1]
        gm = gx * gx + gy * gy
        r = r[gm <= np.quantile(gm, 1 - exclude_edges)]
    return float(math.sqrt(math.pi / 2) * r.mean() / 6.0)


def add_gaussian_noise(img, sigma, seed=0):
    """Additive Gaussian noise with the standard deviation sigma (grey levels)."""
    rng = np.random.default_rng(seed)
    return _u8(_f(img) + rng.normal(0, sigma, img.shape))


def add_impulse_noise(img, amount, seed=0):
    """Impulse ("salt and pepper") noise: the share `amount` of the pixels becomes black or white."""
    rng = np.random.default_rng(seed)
    out = img.copy()
    m = rng.random(img.shape[:2]) < amount
    vals = rng.choice(np.array([0, 255], np.uint8), size=int(m.sum()))
    out[m] = vals[:, None] if img.ndim == 3 else vals
    return out


def equalize(img, method="clahe", clip=2.0, tiles=8):
    """Global histogram equalisation (method "he") or CLAHE ("clahe": tiles x tiles regions, histogram of every
    region clipped at clip x its mean height) of a grey frame or of the lightness L* of a colour frame (CIE L*a*b*)."""
    def one(g):
        if method == "he":
            return cv2.equalizeHist(g)
        return cv2.createCLAHE(clipLimit=float(clip), tileGridSize=(int(tiles), int(tiles))).apply(g)
    if img.ndim == 2:
        return one(img)
    if np.array_equal(img[..., 0], img[..., 1]) and np.array_equal(img[..., 1], img[..., 2]):
        g = one(img[..., 0])
        return np.dstack([g, g, g])
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    lab[..., 0] = one(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


def letterbox(img, size=640, stride=32, auto=False, scaleup=True, color=114):
    """Letterbox transformation to the input of a detector: the frame is scaled by r = min(S / H, S / W)
    without a change of its aspect ratio and padded symmetrically with the grey value `color` to S x S
    (auto=True: only to the nearest multiple of `stride`). Returns the image, r and the padding (left, top)."""
    h, w = img.shape[:2]
    r = min(size / h, size / w)
    if not scaleup:
        r = min(r, 1.0)
    nw, nh = int(round(w * r)), int(round(h * r))
    dw, dh = size - nw, size - nh
    if auto:
        dw, dh = dw % stride, dh % stride
    dw, dh = dw / 2.0, dh / 2.0
    if (nw, nh) != (w, h):
        img = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
    left, right = int(round(dw - 0.1)), int(round(dw + 0.1))
    value = (color, color, color) if img.ndim == 3 else color
    return cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=value), r, (left, top)


def boxes_to_input(boxes, r, pad):
    """Boxes [x1, y1, x2, y2] of the frame mapped to the letterboxed input: x' = r x + left, y' = r y + top."""
    return [[b[0] * r + pad[0], b[1] * r + pad[1], b[2] * r + pad[0], b[3] * r + pad[1]] for b in boxes]


def boxes_from_input(boxes, r, pad):
    """Inverse mapping of boxes of the letterboxed input to the frame: x = (x' - left) / r, y = (y' - top) / r."""
    return [[(b[0] - pad[0]) / r, (b[1] - pad[1]) / r, (b[2] - pad[0]) / r, (b[3] - pad[1]) / r] for b in boxes]


def to_input_tensor(img):
    """Standard input of a neural network: RGB, float32 in 0...1, layout N x C x H x W (N = 1)."""
    x = to_rgb(img).astype(np.float32) / 255.0
    return np.ascontiguousarray(x.transpose(2, 0, 1)[None])


def time_call(fn, repeats=5, warmup=1):
    """Median execution time of fn() in milliseconds."""
    for _ in range(warmup):
        fn()
    t = []
    for _ in range(max(1, repeats)):
        t0 = time.perf_counter()
        fn()
        t.append((time.perf_counter() - t0) * 1000.0)
    return float(np.median(t))


def step_overlap(tracks, step):
    """Overlap and displacement of every annotated object between the frames t and t + step of a track table
    (columns frame, id, x, y, w, h): IoU of its two boxes, displacement of the centre (px) and box size."""
    import pandas as pd
    a = tracks[["frame", "id", "x", "y", "w", "h"]]
    b = a.copy()
    b["frame"] = b["frame"] - step
    m = a.merge(b, on=["id", "frame"], suffixes=("", "_n"))
    ix = (np.minimum(m.x + m.w, m.x_n + m.w_n) - np.maximum(m.x, m.x_n)).clip(lower=0)
    iy = (np.minimum(m.y + m.h, m.y_n + m.h_n) - np.maximum(m.y, m.y_n)).clip(lower=0)
    inter = ix * iy
    union = m.w * m.h + m.w_n * m.h_n - inter
    return pd.DataFrame({"frame": m.frame, "id": m.id, "iou": inter / union.clip(lower=1e-6),
                         "displacement_px": np.hypot(m.x_n + m.w_n / 2 - m.x - m.w / 2, m.y_n + m.h_n / 2 - m.y - m.h / 2),
                         "size_px": np.minimum(m.w, m.h)})


# =====================================================================================
# Topic 2, practical lessons 2.3-2.4: noise models, quality and suitability measures
# =====================================================================================
def add_poisson_noise(img, photons_per_level=2.0, seed=0):
    """Photon (shot) noise: every value I is replaced by Poisson(k I) / k with k photons per grey level,
    so the variance of the noise equals I / k and grows with the brightness (signal-dependent noise)."""
    rng = np.random.default_rng(seed)
    k = max(float(photons_per_level), 1e-3)
    return _u8(rng.poisson(_f(img) * k) / k)


def add_column_noise(img, sigma=8.0, seed=0, row_sigma=0.0):
    """Fixed-pattern noise of a sensor read out by columns (stripes typical of uncooled thermal
    cameras between two non-uniformity corrections): every column receives its own offset N(0, sigma^2),
    optionally every row an offset N(0, row_sigma^2)."""
    rng = np.random.default_rng(seed)
    h, w = img.shape[:2]
    off = rng.normal(0, sigma, (1, w)) + (rng.normal(0, row_sigma, (h, 1)) if row_sigma > 0 else 0)
    x = _f(img)
    return _u8(x + (off[..., None] if x.ndim == 3 else off))


def column_correction(img, smooth_px=15):
    """Correction of column stripes (scene-based non-uniformity correction, simplest form): the mean of
    every column is compared with a smoothed profile of the column means; the difference is the offset of
    the column and is subtracted. Smooth structures of the scene wider than smooth_px are kept."""
    x = _f(img)
    planes = [x] if x.ndim == 2 else [x[..., c] for c in range(x.shape[2])]
    out = []
    for p in planes:
        m = p.mean(axis=0)
        ms = cv2.GaussianBlur(m.reshape(1, -1).astype(np.float32), (0, 0), smooth_px).ravel()
        out.append(p - (m - ms)[None, :])
    return _u8(out[0] if x.ndim == 2 else np.dstack(out))


def edge_f1(ref, test, low=60, high=140, tol=1):
    """Agreement of the Canny edges of a test frame with the edges of a reference frame: an edge pixel is
    matched when the other map has an edge within tol pixels; returns dict(precision, recall, f1)."""
    e1 = cv2.Canny(gray(ref), low, high) > 0
    e2 = cv2.Canny(gray(test), low, high) > 0
    k = np.ones((2 * tol + 1, 2 * tol + 1), np.uint8)
    d1 = cv2.dilate(e1.astype(np.uint8), k) > 0
    d2 = cv2.dilate(e2.astype(np.uint8), k) > 0
    p = float((e2 & d1).sum()) / max(int(e2.sum()), 1)
    r = float((e1 & d2).sum()) / max(int(e1.sum()), 1)
    return dict(precision=p, recall=r, f1=2 * p * r / max(p + r, 1e-9))


def rank_correlation(a, b, sample=200000, seed=0):
    """Spearman rank correlation of the grey levels of two frames of the same scene (pixel by pixel).
    A global monotonic mapping of the brightness (stretching, gamma, global HE) gives 1 up to ties;
    a local mapping (CLAHE) changes the order of brightness between regions and gives less than 1."""
    x, y = gray(a).ravel().astype(np.float64), gray(b).ravel().astype(np.float64)
    if x.size > sample:
        idx = np.random.default_rng(seed).choice(x.size, sample, replace=False)
        x, y = x[idx], y[idx]
    def ranks(v):
        order = np.argsort(v, kind="mergesort")
        r = np.empty(v.size, np.float64)
        r[order] = np.arange(v.size)
        _, inv, cnt = np.unique(v, return_inverse=True, return_counts=True)
        sums = np.bincount(inv, weights=r)
        return (sums / cnt)[inv]                       # mean rank of tied values
    rx, ry = ranks(x), ranks(y)
    return float(np.corrcoef(rx, ry)[0, 1])


def hue_shift(ref, test, s_min=40, v_min=40):
    """Mean absolute change of the hue (degrees, circular) of the saturated pixels between two colour frames."""
    h1 = cv2.cvtColor(to_rgb(ref), cv2.COLOR_RGB2HSV)
    h2 = cv2.cvtColor(to_rgb(test), cv2.COLOR_RGB2HSV)
    m = (h1[..., 1] >= s_min) & (h1[..., 2] >= v_min)
    if not m.any():
        return float("nan")
    d = np.abs(h1[..., 0][m].astype(np.int32) - h2[..., 0][m].astype(np.int32)) * 2
    return float(np.minimum(d, 360 - d).mean())


def linear_stretch(img, p_low=1.0, p_high=99.0):
    """Global linear stretch of the brightness between two percentiles to 0...255 (the operation of the
    automatic gain control of a thermal camera); monotonic, keeps the order of brightness."""
    g = gray(img)
    lo, hi = np.percentile(g, [p_low, p_high])
    hi = max(hi, lo + 1)
    lut = np.clip((np.arange(256) - lo) * 255.0 / (hi - lo), 0, 255).astype(np.uint8)
    return cv2.LUT(img, lut)


def gamma_correction(img, gamma=0.6):
    """Global power-law mapping I' = 255 (I / 255)^gamma; gamma < 1 brightens the dark part of the range."""
    lut = np.clip(255.0 * (np.arange(256) / 255.0) ** float(gamma), 0, 255).astype(np.uint8)
    return cv2.LUT(img, lut)


def ambe(a, b):
    """Absolute mean brightness error |mean(b) - mean(a)| of the grey levels."""
    return abs(float(gray(b).mean()) - float(gray(a).mean()))


# =============================================================================
# 9. Topic 3: edges, binarisation, contours, ground geometry, FPV video
# =============================================================================

def label_boundaries(labels, ignore=(0,)):
    """Boundaries between regions of a label image (pixel labels of a data set): a pixel is a boundary
    pixel when one of its 4 neighbours has another label; pixels of the classes in `ignore` are excluded."""
    lab = np.asarray(labels)
    b = np.zeros(lab.shape, bool)
    b[:, 1:] |= lab[:, 1:] != lab[:, :-1]
    b[:, :-1] |= lab[:, 1:] != lab[:, :-1]
    b[1:, :] |= lab[1:, :] != lab[:-1, :]
    b[:-1, :] |= lab[1:, :] != lab[:-1, :]
    for c in ignore:
        b &= lab != c
    return b


def edge_map_scores(pred, ref, tol=2):
    """Agreement of two binary edge maps within `tol` pixels: precision (share of predicted edge pixels near a
    reference edge), recall (share of reference edge pixels near a predicted edge) and F1."""
    p, r = np.asarray(pred) > 0, np.asarray(ref) > 0
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * tol + 1, 2 * tol + 1))
    rd = cv2.dilate(r.astype(np.uint8), k) > 0
    pd_ = cv2.dilate(p.astype(np.uint8), k) > 0
    prec = float((p & rd).sum()) / max(int(p.sum()), 1)
    rec = float((r & pd_).sum()) / max(int(r.sum()), 1)
    return dict(precision=prec, recall=rec, f1=2 * prec * rec / max(prec + rec, 1e-9),
                edge_share=float(p.mean()))


def otsu(img, mask=None):
    """Otsu threshold computed by its definition: for every threshold t the between-class variance
    sigma_B^2(t) = w0(t) w1(t) (mu0(t) - mu1(t))^2 of the histogram is computed, and the threshold with the
    largest value is chosen. Returns (t, sigma_B^2 for t = 0...255, histogram)."""
    g = gray(img)
    v = g[mask > 0] if mask is not None else g.ravel()
    h = np.bincount(v.ravel(), minlength=256).astype(np.float64)
    p = h / max(h.sum(), 1)
    w0 = np.cumsum(p)
    mu = np.cumsum(p * np.arange(256))
    mt = mu[-1]
    w1 = 1 - w0
    with np.errstate(divide="ignore", invalid="ignore"):
        sb = (mt * w0 - mu) ** 2 / (w0 * w1)
    sb = np.nan_to_num(sb, nan=0.0, posinf=0.0)
    return int(np.argmax(sb)), sb, h


def oblique_gsd(altitude, angle, rows, height=512, vfov_deg=20.0, hfov_deg=25.0, width=640):
    """Ground sampling distance of a tilted camera in the image rows, m/px: (across, along).
    The depression angle of the line of sight of row y is delta = angle + (y - H/2)/H * vfov
    (angle = 90 deg for a vertical view); the slant range is h / sin(delta), the footprint of a pixel
    across the line of sight is range * IFOV and along it range * IFOV / sin(delta)."""
    rows = np.asarray(rows, float)
    dep = np.radians(np.clip(angle + (rows - height / 2.0) / height * vfov_deg, 1.0, 179.0))
    rng = altitude / np.sin(dep)
    across = rng * 2 * math.tan(math.radians(hfov_deg) / 2) / width
    return across, across / np.sin(dep)


def camera_matrix(width, height, hfov_deg, vfov_deg=None):
    """Pinhole camera matrix K from the size of the image and the field of view."""
    fx = width / 2.0 / math.tan(math.radians(hfov_deg) / 2)
    fy = fx if vfov_deg is None else height / 2.0 / math.tan(math.radians(vfov_deg) / 2)
    return np.array([[fx, 0, (width - 1) / 2.0], [0, fy, (height - 1) / 2.0], [0, 0, 1.0]])


def ground_homography(altitude, angle, K, size, gsd_out=None, max_range_m=None):
    """Homography that maps an image of a tilted camera to a top view of the flat ground (perspective
    transformation of Lesson 3.3). The camera is at the height h over the plane Z = 0 and looks forward
    with the depression angle `angle` (90 = vertical view); image x is the ground X axis.
    H_g = K [r1, r2, -h r3] maps the ground point (X, Y, 1) to the image; the returned matrix maps
    image pixels to pixels of the top view with the step gsd_out, m/px (forward = up).
    Returns (H, (width, height) of the top view, gsd_out, extent (Xmin, Xmax, Ymin, Ymax) in metres)."""
    th = math.radians(angle)
    R = np.array([[1, 0, 0], [0, -math.sin(th), -math.cos(th)], [0, math.cos(th), -math.sin(th)]], float)
    Hg = K @ np.column_stack([R[:, 0], R[:, 1], -altitude * R[:, 2]])
    Hi = np.linalg.inv(Hg)
    w, h = size
    corners = np.array([[0, 0, 1], [w - 1, 0, 1], [w - 1, h - 1, 1], [0, h - 1, 1]], float).T
    g = Hi @ corners
    X, Y = g[0] / g[2], g[1] / g[2]
    if max_range_m:
        Y = np.minimum(Y, max(float(max_range_m), float(Y.min()) + 20.0))    # at least 20 m beyond the nearest row
    if gsd_out is None:
        gsd_out = altitude / math.sin(th) / K[0, 0]          # GSD across the line of sight at the image centre
    x0, x1, y0, y1 = X.min(), X.max(), Y.min(), Y.max()
    W, Ht = int(math.ceil((x1 - x0) / gsd_out)), int(math.ceil((y1 - y0) / gsd_out))
    S = np.array([[1 / gsd_out, 0, -x0 / gsd_out], [0, -1 / gsd_out, y1 / gsd_out], [0, 0, 1]])
    return S @ Hi, (W, Ht), gsd_out, (x0, x1, y0, y1)


def transform_boxes(boxes, H):
    """Boxes [x1, y1, x2, y2] after the perspective transformation H (box of the four mapped corners)."""
    out = []
    for b in boxes:
        c = np.array([[b[0], b[1], 1], [b[2], b[1], 1], [b[2], b[3], 1], [b[0], b[3], 1]], float).T
        m = H @ c
        x, y = m[0] / m[2], m[1] / m[2]
        out.append([x.min(), y.min(), x.max(), y.max()])
    return np.array(out, float).reshape(-1, 4)


def thermal_mask(img, polarity="warm", method="top-hat + Otsu", se_px=31, k=4.0, block=51, offset=-5):
    """Binary mask of thermal signatures. polarity: 'warm' (brighter than the surroundings in a white-hot
    frame), 'cold' (darker) or 'both'. method: 'Otsu (global)', 'top-hat + Otsu', 'top-hat + mean + k*std',
    'adaptive (local mean)'. The white top-hat I - open(I) keeps bright details smaller than the structuring
    element se_px, the black top-hat close(I) - I keeps dark details."""
    g = cv2.GaussianBlur(gray(img), (3, 3), 0)
    parts = (["warm"] if polarity == "warm" else ["cold"] if polarity == "cold" else ["warm", "cold"])
    se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (int(se_px) | 1, int(se_px) | 1))
    out = np.zeros(g.shape, np.uint8)
    for pol in parts:
        src = g if pol == "warm" else 255 - g
        if method.startswith("Otsu"):
            _, m = cv2.threshold(src, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        elif method.startswith("adaptive"):
            m = cv2.adaptiveThreshold(src, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, int(block) | 1, offset)
        else:
            th = cv2.morphologyEx(src, cv2.MORPH_TOPHAT, se)
            if "Otsu" in method:
                _, m = cv2.threshold(th, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            else:
                t = th.astype(np.float32)
                m = ((t > t.mean() + k * t.std()) * 255).astype(np.uint8)
        out |= m
    return out


def contour_features(c, gsd_across=None, gsd_along=None):
    """Geometric characteristics of a contour (OpenCV point array): area and perimeter, px; box;
    length and width of the minimum-area rectangle; aspect ratio L/W; extent = area / box area;
    solidity = area / area of the convex hull; circularity 4 pi A / P^2 (1 for a circle).
    With the GSD of the row of the centroid (m/px across and along the line of sight) the contour is
    scaled to metres and its area, length and width in metres are added."""
    a = float(cv2.contourArea(c))
    p = float(cv2.arcLength(c, True))
    x, y, w, h = cv2.boundingRect(c)
    m = cv2.moments(c)
    cx, cy = (m["m10"] / m["m00"], m["m01"] / m["m00"]) if m["m00"] > 0 else (x + w / 2, y + h / 2)
    (_, _), (rw, rh), ang = cv2.minAreaRect(c)
    L, W = max(rw, rh), min(rw, rh)
    hull = float(cv2.contourArea(cv2.convexHull(c)))
    f = dict(cx=cx, cy=cy, box=[x, y, x + w, y + h], area_px=a, perimeter_px=p, length_px=L, width_px=W,
             aspect=L / max(W, 1e-6), extent=a / max(w * h, 1e-6), solidity=a / max(hull, 1e-6),
             circularity=4 * math.pi * a / max(p * p, 1e-6))
    if gsd_across is not None:
        pts = c.reshape(-1, 2).astype(np.float32) * np.float32([gsd_across, gsd_along])
        (_, _), (mw, mh), _ = cv2.minAreaRect(pts)
        f.update(area_m2=a * gsd_across * gsd_along, length_m=max(mw, mh), width_m=min(mw, mh))
    return f


class TIIRacing:
    """TII-RATM ("Race Against the Machine"): flights of an FPV racing quadrotor in an indoor arena with
    racing gates, piloted by human FPV pilots and autonomous, speeds above 21 m/s. Camera Arducam IMX219,
    640 x 480 px, 120 frames per second (125 measured from the time stamps), wide-angle lens with a field of
    view of 155 x 115 deg, tilted up by 30-50 deg; IMU 500 Hz (gyroscope, accelerometer), motion capture,
    labels of the gates. Bosello M. et al. IEEE Robotics and Automation Letters 9(4), 3799-3806 (2024).
    Licence CC BY 4.0.

    The frames of a flight are stored in random order in a zip archive inside the release archive. The first
    call of clip() downloads the camera archive of the flight once (about 200 MB, transferred in parallel
    parts) and keeps in the cache the frames of the requested interval and of the preset SEGMENTS."""

    RELEASE = "https://github.com/tii-racing/drone-racing-dataset/releases/download/v3.0.0/"
    RAW = "https://raw.githubusercontent.com/tii-racing/drone-racing-dataset/main/"
    # local header offset and compressed size of the members in the release archive piloted.zip
    FLIGHTS = {"flight-16p-trackRATM": dict(asset="piloted_zipchunk01", camera=(1574086012, 201903685),
                                            camera_csv=(1549423339, 33752), imu=(1546851674, 488328),
                                            meta=(1574085613, 258))}
    SEGMENTS = {"flight-16p-trackRATM": {"A: take-off and first gates (0.5-4.5 s)": (0.5, 4.5),
                                         "B: slalom between gates (15.0-19.0 s)": (15.0, 19.0)}}
    ROTATION_KEY = {50: "trackRATM", 40: "lemniscate", 60: "piloted"}

    def __init__(self, flight="flight-16p-trackRATM"):
        if flight not in self.FLIGHTS:
            raise ValueError(f"flight {flight} is not prepared; available: {list(self.FLIGHTS)}")
        self.flight = flight
        self.info = self.FLIGHTS[flight]
        self.url = self.RELEASE + self.info["asset"]

    def _member(self, key):
        off, csize = self.info[key]
        path = _cache("tii", self.flight, key + ".bin")
        if os.path.exists(path):
            return open(path, "rb").read()
        raw = _read_url(self.url, {"Range": f"bytes={off}-{off + 30 + 1024 + csize}"}, min_bytes=30 + csize)
        if raw[:4] != b"PK\x03\x04":
            raise IOError(f"bad local header of {key}")
        n, m = struct.unpack("<HH", raw[26:30])
        data = zlib.decompressobj(-15).decompress(raw[30 + n + m: 30 + n + m + csize])
        with open(path + ".part", "wb") as fh:
            fh.write(data)
        os.replace(path + ".part", path)
        return data

    def metadata(self):
        """Flight metadata (YAML of the data set) as a flat dict."""
        out, sect = {}, ""
        for line in self._member("meta").decode().splitlines():
            if not line.strip() or ":" not in line:
                continue
            key, val = line.split(":", 1)
            ind = len(key) - len(key.lstrip())
            key = key.strip()
            if ind == 0:
                sect = key
            full = key if ind == 0 else f"{sect}.{key}"
            val = val.strip()
            if val:
                try:
                    val = float(val) if "." in val else int(val)
                except ValueError:
                    pass
                out[full] = val
        return out

    def frame_table(self):
        """Table of the camera frames: index, file, timestamp_us, t (s from the first frame)."""
        import pandas as pd
        t = pd.read_csv(io.BytesIO(self._member("camera_csv")))
        t["index"] = np.arange(len(t))
        t["t"] = (t.timestamp - t.timestamp.iloc[0]) / 1e6
        return t.rename(columns={"filename": "file"})[["index", "file", "timestamp", "t"]]

    def imu(self):
        """IMU table: timestamp_us, t (s from the first camera frame), accel_x/y/z m/s^2, gyro_x/y/z rad/s
        (axes of the body of the drone)."""
        import pandas as pd
        d = pd.read_csv(io.BytesIO(self._member("imu")))
        d["t"] = (d.timestamp - self.frame_table().timestamp.iloc[0]) / 1e6
        return d

    def calibration(self):
        """Camera matrix K and distortion coefficients D (pinhole model with radial and tangential terms)
        of the piloted trackRATM flights."""
        path = fetch(self.RAW + "camera_calibration/calib_p-trackRATM.json", _cache("tii", "calib_p-trackRATM.json"))
        d = json.load(open(path))
        return np.array(d["mtx"], float), np.array(d["dist"], float).ravel()

    def camera_to_body(self):
        """Rotation matrix R (3 x 3) from the camera axes (x right, y down, z forward) to the body axes of
        the drone, for the camera angle of the flight (file drone_to_camera.json of the data set)."""
        path = fetch(self.RAW + "camera_calibration/drone_to_camera.json", _cache("tii", "drone_to_camera.json"))
        q = json.load(open(path))["rotation"][self.ROTATION_KEY.get(int(self.metadata().get("camera.angle", 50)), "trackRATM")]
        x, y, z, w = q["x"], q["y"], q["z"], q["w"]
        return np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
                         [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
                         [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])

    def _download_camera_zip(self, path, part=16 << 20, threads=6):
        off, csize = self.info["camera"]
        hdr = _read_url(self.url, {"Range": f"bytes={off}-{off + 29}"}, min_bytes=30)
        n, m = struct.unpack("<HH", hdr[26:30])
        start = off + 30 + n + m
        pieces = [(start + p, min(start + csize, start + p + part) - 1) for p in range(0, csize, part)]
        with open(path, "wb") as fh:
            fh.truncate(csize)

        def get(rg):
            a, b = rg
            data = _read_url(self.url, {"Range": f"bytes={a}-{b}"}, timeout=120, min_bytes=b - a + 1)
            with open(path, "r+b") as fh:
                fh.seek(a - start)
                fh.write(data)
            return len(data)

        with ThreadPoolExecutor(threads) as ex:
            return sum(ex.map(get, pieces))

    def _extract(self, wanted):
        """Stream the camera archive of the flight and keep the frames with the indices in `wanted`."""
        tmp = _cache("tii", self.flight, "_camera_deflate.part")
        self._download_camera_zip(tmp)
        d = zlib.decompressobj(-15)
        buf, kept = b"", 0
        with open(tmp, "rb") as fh:
            while True:
                chunk = fh.read(4 << 20)
                buf += d.decompress(chunk) if chunk else d.flush()
                while len(buf) >= 30 and buf[:4] == b"PK\x03\x04":
                    flag, comp = struct.unpack("<HH", buf[6:10])
                    csz, usz, nl, el = struct.unpack("<IIHH", buf[18:30])
                    need = 30 + nl + el + csz
                    if len(buf) < need:
                        break
                    name = buf[30:30 + nl].decode("utf-8", "replace")
                    data = buf[30 + nl + el:need]
                    buf = buf[need:]
                    base = name.rsplit("/", 1)[-1]
                    if base.endswith(".jpg") and int(base.split("_")[0]) in wanted:
                        if comp == 8:
                            data = zlib.decompressobj(-15).decompress(data)
                        with open(_cache("tii", self.flight, "frames", base), "wb") as out:
                            out.write(data)
                        kept += 1
                if not chunk or (len(buf) >= 4 and buf[:4] == b"PK\x01\x02"):
                    break
        os.remove(tmp)
        return kept

    def clip(self, t0, t1, step=1):
        """Frames (RGB, 480 x 640 x 3) of the interval t0...t1 s with every `step`-th frame, and their table."""
        tab = self.frame_table()
        sel = tab[(tab.t >= t0) & (tab.t < t1)].iloc[::max(1, int(step))]
        miss = [i for i, f in zip(sel["index"], sel.file) if not os.path.exists(_cache("tii", self.flight, "frames", f))]
        if miss:
            wanted = set(miss)
            for a, b in self.SEGMENTS.get(self.flight, {}).values():
                wanted |= set(tab[(tab.t >= a) & (tab.t < b)]["index"])
            self._extract(wanted)
        frames = [_rgb(_cache("tii", self.flight, "frames", f)) for f in sel.file]
        return frames, sel.reset_index(drop=True)


def undistort_maps(K, D, size, alpha=0.0):
    """Maps of cv2.remap that remove the lens distortion, and the new camera matrix.
    alpha = 0 keeps only valid pixels (the image is cropped), alpha = 1 keeps the whole field of view."""
    newK, _ = cv2.getOptimalNewCameraMatrix(K, D, size, alpha, size)
    m1, m2 = cv2.initUndistortRectifyMap(K, D, None, newK, size, cv2.CV_16SC2)
    return m1, m2, newK


def gyro_rotation(t, w, t1, t2):
    """Rotation vector (rad) of the body between the times t1 and t2 obtained by integration of the angular
    rate w (N x 3, rad/s) measured at the times t (s): R = prod exp([w_k dt_k])."""
    m = (t >= t1) & (t <= t2)
    idx = np.where(m)[0]
    if idx.size == 0:
        return np.zeros(3)
    tt = np.concatenate([[t1], t[idx], [t2]])
    ww = np.vstack([w[max(idx[0] - 1, 0)], w[idx], w[idx[-1]]])
    R = np.eye(3)
    for k in range(len(tt) - 1):
        dt = tt[k + 1] - tt[k]
        if dt <= 0:
            continue
        Rk, _ = cv2.Rodrigues((ww[k] * dt).reshape(3, 1))
        R = R @ Rk
    rv, _ = cv2.Rodrigues(R)
    return rv.ravel()


def rotation_from_homography(H, K):
    """Rotation of the camera between two frames from their homography, assuming a pure rotation:
    H = K R K^-1, so R = K^-1 H K scaled to det = 1 and projected on the nearest rotation matrix.
    Returns the rotation vector (rad) of R in the axes of the camera."""
    if H is None or not np.all(np.isfinite(H)):
        return np.full(3, np.nan)
    R = np.linalg.inv(K) @ H @ K
    d = np.linalg.det(R)
    if not np.isfinite(d) or abs(d) < 1e-12:
        return np.full(3, np.nan)                          # degenerate homography: no rotation can be computed
    R = R / np.cbrt(d)
    U, _, Vt = np.linalg.svd(R)
    R = U @ Vt
    if np.linalg.det(R) < 0:
        R = -R
    rv, _ = cv2.Rodrigues(R)
    return rv.ravel()


def smooth_rotations(rotvecs, sigma):
    """Smoothed orientations of the camera: rotvecs (N x 3) are absolute orientations; every orientation is
    replaced by the Gaussian-weighted mean of its neighbours (sigma in frames) computed in the tangent space
    of the orientation itself. Returns N x 3 rotation vectors."""
    Rs = [cv2.Rodrigues(np.asarray(r, float).reshape(3, 1))[0] for r in rotvecs]
    n = len(Rs)
    half = int(max(1, round(3 * sigma)))
    out = []
    for i in range(n):
        a, b = max(0, i - half), min(n, i + half + 1)
        wts = np.exp(-0.5 * ((np.arange(a, b) - i) / max(sigma, 1e-6)) ** 2)
        wts /= wts.sum()
        acc = np.zeros(3)
        for j, wj in zip(range(a, b), wts):
            d, _ = cv2.Rodrigues(Rs[i].T @ Rs[j])
            acc += wj * d.ravel()
        Rm = Rs[i] @ cv2.Rodrigues(acc.reshape(3, 1))[0]
        out.append(cv2.Rodrigues(Rm)[0].ravel())
    return np.array(out)


def interframe_psnr(frames, valid=None):
    """Inter-frame transformation fidelity (ITF): mean PSNR between consecutive frames, dB; a stabilised
    video has a higher ITF. `valid` (optional list of masks) limits the comparison to valid pixels."""
    vals = []
    for k in range(1, len(frames)):
        a, b = _f(gray(frames[k - 1])), _f(gray(frames[k]))
        m = np.ones(a.shape, bool) if valid is None else (valid[k - 1] & valid[k])
        if m.sum() < 100:
            continue
        mse = float(((a - b)[m] ** 2).mean())
        vals.append(10 * math.log10(255.0 ** 2 / max(mse, 1e-6)))
    return float(np.mean(vals)) if vals else float("nan")



# =====================================================================================
# Topic 4. Machine learning and neural networks in the recognition of military objects
# =====================================================================================
class MilitaryVehicles:
    """Photographs of 24 types of military vehicles: 9444 images (7823 for training, 1621 for
    testing) collected from web sites; Kricheli J. S. et al., arXiv:2407.15192 (2024);
    Hugging Face data set leibnitz-lab/military_vehicles, licence declared by the authors: MIT
    (the rights to the individual photographs are not stated by the authors).

    The parquet files hold every image twice: in the tree of 24 fine types (file names of
    1-3 digits, "0.jpg") and in the tree of the 7 coarse classes of the authors (4 digits,
    "0000.jpg"). The fine tree is used; the coarse class of the authors is attached to every
    image by its MD5 checksum (column `source_group`), the grouping of the course is in `group`."""

    URL = "https://huggingface.co/api/datasets/leibnitz-lab/military_vehicles/parquet/default/{}/0.parquet"
    GROUPS = {"T-14": "Tank", "T-62": "Tank", "T-64": "Tank", "T-72": "Tank", "T-80": "Tank", "T-90": "Tank",
              "BMP-1": "IFV", "BMP-2": "IFV", "BMP-T15": "IFV", "BMD": "IFV",
              "BTR-60": "APC/recon", "BTR-70": "APC/recon", "BTR-80": "APC/recon", "BRDM": "APC/recon",
              "MT_LB": "APC/recon", "2S19_MSTA": "Artillery", "D-30": "Artillery",
              "BM-30": "MLRS", "Tornado": "MLRS", "TOS-1": "MLRS",
              "Pantsir-S1": "Air defence/radar", "30N6E": "Air defence/radar",
              "Iskander": "Missile system", "Rs-24": "Missile system"}
    GROUP_ORDER = ("Tank", "IFV", "APC/recon", "Artillery", "MLRS", "Air defence/radar", "Missile system")
    TYPES = {"T-14": "main battle tank (Armata)", "T-62": "main battle tank", "T-64": "main battle tank",
             "T-72": "main battle tank", "T-80": "main battle tank", "T-90": "main battle tank",
             "BMP-1": "tracked infantry fighting vehicle", "BMP-2": "tracked infantry fighting vehicle",
             "BMP-T15": "heavy infantry fighting vehicle T-15 (Armata)", "BMD": "airborne infantry fighting vehicle",
             "BTR-60": "wheeled armoured personnel carrier", "BTR-70": "wheeled armoured personnel carrier",
             "BTR-80": "wheeled armoured personnel carrier", "BRDM": "wheeled armoured scout car",
             "MT_LB": "tracked multi-purpose armoured tractor", "2S19_MSTA": "152 mm self-propelled howitzer",
             "D-30": "122 mm towed howitzer", "BM-30": "300 mm multiple rocket launcher (Smerch)",
             "Tornado": "multiple rocket launcher (Tornado)", "TOS-1": "heavy thermobaric rocket launcher",
             "Pantsir-S1": "self-propelled anti-aircraft gun and missile system",
             "30N6E": "fire-control radar of the S-300 system", "Iskander": "tactical ballistic missile system",
             "Rs-24": "intercontinental ballistic missile system (Yars) on a wheeled launcher"}

    def __init__(self):
        import hashlib
        import pandas as pd
        import pyarrow.parquet as pq
        parts = []
        for sp in ("train", "test"):
            path = fetch(self.URL.format(sp), _cache("military_vehicles", sp + ".parquet"))
            t = pq.read_table(path)
            names = json.loads(t.schema.metadata[b"huggingface"])["info"]["features"]["label"]["names"]
            d = t.to_pandas()
            d["file"] = [x["path"] for x in d.image]
            d["bytes"] = [x["bytes"] for x in d.image]
            d["label_name"] = [names[i] for i in d.label]
            d["md5"] = [hashlib.md5(b).hexdigest() for b in d.bytes]
            d["split"] = sp
            parts.append(d.drop(columns=["image", "label"]))
        a = pd.concat(parts, ignore_index=True)
        fine = a.file.str.len() < 8
        coarse = a[~fine].groupby("md5").label_name.agg(lambda s: s.value_counts().index[0])
        t = a[fine].rename(columns={"label_name": "cls"}).reset_index(drop=True)
        t["source_group"] = t.md5.map(coarse)
        t["group"] = t.cls.map(self.GROUPS)
        hw = [cv2.imdecode(np.frombuffer(b, np.uint8), cv2.IMREAD_GRAYSCALE).shape for b in t.bytes]
        t["height"], t["width"] = [h for h, w in hw], [w for h, w in hw]
        tr = t[t.split == "train"]
        t["dup_in_split"] = t.duplicated(["split", "md5"])
        cls_train = tr.groupby("md5").cls.agg(lambda s: ", ".join(sorted(set(s))))
        t["in_train"] = (t.split == "test") & t.md5.isin(tr.md5)
        t["train_label"] = np.where(t.in_train, t.md5.map(cls_train), "")
        self.table = t[["split", "file", "cls", "group", "source_group", "width", "height", "md5",
                        "dup_in_split", "in_train", "train_label", "bytes"]]
        self.CLASSES = tuple(sorted(t.cls.unique()))

    def image(self, i, size=None, gray=False):
        """Image i of the table: RGB (or grey) array, optionally resized to size x size."""
        b = self.table.bytes.iat[i] if not isinstance(i, (bytes, bytearray)) else i
        img = cv2.imdecode(np.frombuffer(b, np.uint8), cv2.IMREAD_GRAYSCALE if gray else cv2.IMREAD_COLOR)
        if not gray:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if size:
            img = cv2.resize(img, (int(size), int(size)), interpolation=cv2.INTER_AREA)
        return img

    def hashes(self):
        """dHash (8 bytes) of every image of the table (computed once)."""
        if getattr(self, "_hashes", None) is None:
            self._hashes = np.stack([dhash(self.image(i, gray=True)) for i in range(len(self.table))])
        return self._hashes

    def clean(self, near=None):
        """Indices of the cleaned training and test parts: a copy of an image repeated inside a part is
        removed, and every test image that also occurs in the training part is removed. With near = d
        also the test images whose dHash differs from the hash of a training image by at most d bits
        (near duplicates: the same photograph re-compressed or resized) are removed."""
        t = self.table
        tr = t.index[(t.split == "train") & ~t.dup_in_split].values
        te = t.index[(t.split == "test") & ~t.dup_in_split & ~t.in_train].values
        if near is not None:
            h = self.hashes()
            d, _ = hamming_nearest(h[te], h[tr])
            te = te[d > int(near)]
        return tr, te

    def subset(self, idx, n_per_class, seed=0):
        """At most n_per_class indices of every fine type from idx (random, reproducible)."""
        rng = np.random.default_rng(seed)
        t = self.table.loc[idx]
        out = [rng.permutation(g.index.values)[:int(n_per_class)] for _, g in t.groupby("cls")]
        return np.sort(np.concatenate(out))


class SULAND:
    """SULAND: frames of video sequences of a camera moved over grass, fallen leaves and gravel
    with surrogates of two anti-personnel surface landmines, PFM-1 ("butterfly") and PMA-2
    ("starfish"); iPhone 13, 1440 x 1920 pixels (portrait), boxes in the YOLO format.
    In-distribution part recorded in Italy (data-iid: train 34, val 5, test 6 sequences) and
    out-of-distribution part recorded in the USA (data-ood: val 10 sequences).
    Vivoli E., Bertini M., Capineri L. Remote Sensing 16(4), 677 (2024); refined annotation
    SULAND_v2: Lekhak S. et al., arXiv:2607.28996 (2026). Licence CC BY 4.0.

    Only the annotation archive (7-10 MB) is downloaded in full; the frames are read one by one
    from the image archive of 16.7 GB with HTTP range requests. version="v1" gives the original
    annotation of the authors, including its errors (in data-ood the class numbers are inverted)."""

    BASE = "https://huggingface.co/datasets/SagarLekhak/SULAND_v2_RGB_Surface_Landmine_Dataset/resolve/main/"
    ANNOTATIONS = {"v2": "Annotation_files_for_SULAND_v2.zip", "v1": "Annotations_files_for_SULAND_v1.zip"}
    CLASSES = ("PFM-1", "PMA-2")
    NICKNAMES = {"PFM-1": "butterfly", "PMA-2": "starfish"}
    COLORS = {"PFM-1": (255, 64, 64), "PMA-2": (0, 229, 255)}
    _images = None

    def __init__(self, version="v2"):
        import re as _re
        import zipfile
        import pandas as pd
        self.version = version
        path = fetch(self.BASE + self.ANNOTATIONS[version], _cache("suland", self.ANNOTATIONS[version]))
        z = zipfile.ZipFile(path)
        frames, boxes = [], []
        for n in z.namelist():
            if not (n.endswith(".txt") and "/labels/" in n):
                continue
            p = n.split("/")
            i = p.index("labels")
            dom, split, seq, stem = p[i - 2], p[i - 1], p[i + 1], p[-1][:-4]
            fr = int(_re.search(r"(\d+)$", stem).group(1))
            lines = [l.split() for l in z.read(n).decode().strip().splitlines() if l.strip()]
            for l in lines:
                c = int(l[0])
                boxes.append((dom, split, seq, fr, c, self.CLASSES[c], *map(float, l[1:5])))
            frames.append((dom, split, seq, fr, stem, f"SULAND_images/{dom}/{split}/images/{seq}/{stem}.jpg",
                           len(lines), sum(int(l[0]) == 0 for l in lines), sum(int(l[0]) == 1 for l in lines)))
        self.frames = pd.DataFrame(frames, columns=["domain", "split", "seq", "frame", "stem", "member", "n_boxes",
                                                    "n_pfm1", "n_pma2"]).sort_values(["domain", "split", "seq", "frame"]).reset_index(drop=True)
        self.boxes = pd.DataFrame(boxes, columns=["domain", "split", "seq", "frame", "cls_id", "cls", "cx", "cy", "w", "h"])
        self._by_frame = {k: g for k, g in self.boxes.groupby(["seq", "frame"])}
        self._member = {(r.seq, r.frame): r.member for r in self.frames.itertuples()}

    def sequences(self):
        """Table of the sequences: part, number of frames, frames with mines, boxes of both classes."""
        f = self.frames
        g = f.groupby(["domain", "split", "seq"]).agg(frames=("frame", "size"), first=("frame", "min"), last=("frame", "max"),
                                                      with_mines=("n_boxes", lambda s: int((s > 0).sum())),
                                                      boxes=("n_boxes", "sum"), pfm1=("n_pfm1", "sum"), pma2=("n_pma2", "sum"))
        return g.reset_index()

    @classmethod
    def _zip(cls):
        if SULAND._images is None:
            SULAND._images = RemoteZip(cls.BASE + "SULAND_images.zip", "suland_images")
        return SULAND._images

    def path(self, seq, frame):
        return self._zip()._fetch_member(self._member[(seq, int(frame))])

    def prefetch(self, items, threads=8):
        """items: list of (sequence, frame); downloads the frames (about 0.5 MB each)."""
        return self._zip().get([self._member[(s, int(f))] for s, f in items], threads)

    def objects(self, seq, frame, width=1440, height=1920):
        """Boxes of a frame in pixels: list of dicts with 'cls' and 'box' [x1, y1, x2, y2]."""
        g = self._by_frame.get((seq, int(frame)))
        if g is None:
            return []
        return [dict(cls=r.cls, box=yolo_to_xyxy(r.cx, r.cy, r.w, r.h, width, height)) for r in g.itertuples()]

    def load(self, seq, frame, gray=False):
        """Frame (RGB or grey array) and its boxes in pixels."""
        img = cv2.imread(self.path(seq, frame), cv2.IMREAD_GRAYSCALE if gray else cv2.IMREAD_COLOR)
        if not gray:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return img, self.objects(seq, frame, img.shape[1], img.shape[0])

    def window(self, seq, length):
        """First frame of the run of `length` consecutive frames of a sequence with most frames with mines."""
        f = self.frames[self.frames.seq == seq].set_index("frame").n_boxes.gt(0).astype(int)
        s = f.rolling(int(length)).sum().shift(-(int(length) - 1))
        return int(s.idxmax())

    def crop_set(self, seqs, length=30, size=64, scale=1.5, n_background=2, seed=0, threads=8):
        """Crops of a run of `length` consecutive frames of every sequence: every box (square of
        scale x the longer side of the box around its centre) and n_background squares of the same
        size without mines from the same frame; grey crops size x size. Returns a DataFrame with
        seq, frame, label ('PFM-1', 'PMA-2', 'background') and img."""
        import pandas as pd
        rng = np.random.default_rng(seed)
        items = []
        for s in seqs:
            a = self.window(s, length)
            items += [(s, a + k) for k in range(int(length)) if (s, a + k) in self._member]
        self.prefetch(items, threads)
        rows = []
        for s, fr in items:
            img, objs = self.load(s, fr, gray=True)
            H, W = img.shape
            sq = []
            for o in objs:
                x1, y1, x2, y2 = o["box"]
                side = int(max(24, scale * max(x2 - x1, y2 - y1)))
                x0 = int(min(max((x1 + x2) / 2 - side / 2, 0), W - side))
                y0 = int(min(max((y1 + y2) / 2 - side / 2, 0), H - side))
                sq.append((x0, y0, side))
                rows.append(dict(seq=s, frame=fr, label=o["cls"],
                                 img=cv2.resize(img[y0:y0 + side, x0:x0 + side], (size, size), interpolation=cv2.INTER_AREA)))
            k = tries = 0
            while k < n_background and tries < 300:
                tries += 1
                side = int(rng.choice([q[2] for q in sq])) if sq else 120
                x0, y0 = int(rng.integers(0, W - side)), int(rng.integers(0, H - side))
                if all(x0 + side <= q[0] or q[0] + q[2] <= x0 or y0 + side <= q[1] or q[1] + q[2] <= y0 for q in sq):
                    rows.append(dict(seq=s, frame=fr, label="background",
                                     img=cv2.resize(img[y0:y0 + side, x0:x0 + side], (size, size), interpolation=cv2.INTER_AREA)))
                    k += 1
        return pd.DataFrame(rows)


def yolo_to_xyxy(cx, cy, w, h, width, height):
    """YOLO box (centre and size as fractions of the image) -> [x1, y1, x2, y2] in pixels."""
    return [(cx - w / 2) * width, (cy - h / 2) * height, (cx + w / 2) * width, (cy + h / 2) * height]


def xyxy_to_yolo(box, width, height):
    """[x1, y1, x2, y2] in pixels -> (cx, cy, w, h) as fractions of the image (YOLO format)."""
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2 / width, (y1 + y2) / 2 / height, (x2 - x1) / width, (y2 - y1) / height)


def xyxy_to_coco(box):
    """[x1, y1, x2, y2] -> [x, y, width, height] (COCO format, pixels)."""
    x1, y1, x2, y2 = box
    return [x1, y1, x2 - x1, y2 - y1]


def dhash(img, n=8):
    """Difference hash: n x n bits (sign of the difference of neighbouring pixels of the image reduced
    to (n + 1) x n); returned as n * n / 8 bytes (uint8). Close images have close hashes."""
    g = gray(img) if img.ndim == 3 else img
    r = cv2.resize(g, (n + 1, n), interpolation=cv2.INTER_AREA).astype(np.int16)
    return np.packbits((r[:, 1:] > r[:, :-1]).ravel())


def hamming_nearest(A, B, chunk=256):
    """For every hash of A (Na x k uint8) the smallest Hamming distance to the hashes of B and its index."""
    A, B = np.asarray(A, np.uint8), np.asarray(B, np.uint8)
    dist, idx = np.empty(len(A), int), np.empty(len(A), int)
    for s in range(0, len(A), chunk):
        x = np.bitwise_xor(A[s:s + chunk, None, :], B[None, :, :])
        d = np.unpackbits(x, axis=2).sum(2)
        dist[s:s + chunk], idx[s:s + chunk] = d.min(1), d.argmin(1)
    return dist, idx


def hog_length(size, cell=8, block=2, orientations=9):
    """Length of the HOG vector of a size x size image: blocks x block^2 x orientations."""
    nb = size // cell - block + 1
    return nb * nb * block * block * orientations


def hog_features(images, size=64, orientations=9, cell=8, block=2):
    """HOG vectors (N x D float32) of grey images resized to size x size (scikit-image)."""
    from skimage.feature import hog
    out = []
    for im in images:
        g = gray(im) if im.ndim == 3 else im
        if g.shape != (size, size):
            g = cv2.resize(g, (size, size), interpolation=cv2.INTER_AREA)
        out.append(hog(g, orientations=orientations, pixels_per_cell=(cell, cell), cells_per_block=(block, block)))
    return np.asarray(out, np.float32)


IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], np.float32)


def imagenet_batch(images, size=224):
    """RGB uint8 images -> float32 array N x 3 x size x size normalised as for ImageNet models."""
    x = np.stack([cv2.resize(im, (size, size), interpolation=cv2.INTER_AREA) for im in images]).astype(np.float32) / 255.0
    return ((x - IMAGENET_MEAN) / IMAGENET_STD).transpose(0, 3, 1, 2).copy()


def cnn_features(model, images, size=224, batch=64, device="cpu"):
    """Output of the convolutional part of a torchvision MobileNet (features + global average pooling):
    N x C float32. `images` is a list of RGB arrays or a function i -> image with len given by a range."""
    import torch
    model = model.eval().to(device)
    out = []
    with torch.no_grad():
        for s in range(0, len(images), batch):
            x = torch.from_numpy(imagenet_batch([images[i] for i in range(s, min(s + batch, len(images)))], size)).to(device)
            f = model.avgpool(model.features(x)).flatten(1)
            out.append(f.cpu().numpy())
    return np.concatenate(out)


def count_parameters(model):
    """Number of parameters of a torch model (all, trainable)."""
    return sum(p.numel() for p in model.parameters()), sum(p.numel() for p in model.parameters() if p.requires_grad)


def conv_params(k, c_in, c_out, separable=False):
    """Number of weights of a k x k convolution layer (without bias): standard k^2*C_in*C_out or
    depthwise separable k^2*C_in + C_in*C_out."""
    return k * k * c_in + c_in * c_out if separable else k * k * c_in * c_out


# ---- diagrams of networks -------------------------------------------------------------
_EDGE, _IN, _HID, _OUT, _OP = "#455a64", "#e3f2fd", "#fff3e0", "#e8f5e9", "#f3e5f5"


def _node(ax, x, y, r, fc, text=None, fs=8):
    import matplotlib.patches as mp
    ax.add_patch(mp.Circle((x, y), r, fc=fc, ec=_EDGE, lw=1.0, zorder=3))
    if text:
        ax.text(x, y, text, ha="center", va="center", fontsize=fs, zorder=4)


def _box(ax, x, y, w, h, fc, text, fs=8.5):
    import matplotlib.patches as mp
    ax.add_patch(mp.FancyBboxPatch((x - w / 2, y - h / 2), w, h, boxstyle="round,pad=0.02,rounding_size=0.05",
                                   fc=fc, ec=_EDGE, lw=1.0, zorder=3))
    ax.text(x, y, text, ha="center", va="center", fontsize=fs, zorder=4)


def _arrow(ax, a, b, lw=1.0, color=_EDGE):
    ax.annotate("", xy=b, xytext=a, arrowprops=dict(arrowstyle="-|>", lw=lw, color=color), zorder=2)


def draw_neuron(ax, inputs=("x1", "x2"), weights=("w1", "w2"), bias="b", activation="sigmoid", output="y"):
    """Diagram of an artificial neuron: inputs, weights, weighted sum with bias, activation, output."""
    n = len(inputs)
    ys = np.linspace(0.85, 0.15, n) if n > 1 else np.array([0.5])
    for y, xi, wi in zip(ys, inputs, weights):
        _node(ax, 0.08, y, 0.055, _IN, xi)
        _arrow(ax, (0.14, y), (0.43, 0.5 + (y - 0.5) * 0.15))
        ax.text(0.27, 0.5 + (y - 0.5) * 0.6 + 0.03, wi, fontsize=9, color="#1565c0", ha="center")
    _node(ax, 0.08, -0.05, 0.045, "#eeeeee", "1", 8)
    _arrow(ax, (0.13, -0.03), (0.43, 0.44))
    ax.text(0.26, 0.12, bias, fontsize=9, color="#1565c0")
    _node(ax, 0.5, 0.5, 0.08, _HID, "Σ", 13)
    _arrow(ax, (0.58, 0.5), (0.66, 0.5))
    _box(ax, 0.75, 0.5, 0.17, 0.2, _OP, "")
    z = np.linspace(-1, 1, 40)
    act = {"sigmoid": 1 / (1 + np.exp(-5 * z)), "tanh": (np.tanh(3 * z) + 1) / 2, "ReLU": np.maximum(z, 0),
           "step": (z > 0).astype(float)}.get(activation, z)
    ax.plot(0.75 + z * 0.065, 0.43 + act * 0.12, color="#6a1b9a", lw=1.5, zorder=5)
    ax.text(0.75, 0.64, f"φ: {activation}", ha="center", fontsize=8.5)
    _arrow(ax, (0.84, 0.5), (0.95, 0.5))
    ax.text(0.97, 0.5, output, va="center", fontsize=11)
    ax.text(0.5, 0.24, "z = Σ wᵢxᵢ + b", ha="center", fontsize=9)
    ax.set_xlim(0, 1.03)
    ax.set_ylim(-0.12, 1.0)
    ax.set_aspect("equal")
    ax.axis("off")


def draw_mlp(ax, sizes, names=None, max_neurons=7, input_names=None, output_names=None):
    """Graph of a fully connected network: layers of neurons (circles) with all connections; a layer
    larger than max_neurons is drawn with max_neurons circles and a gap marked by dots."""
    L = len(sizes)
    xs = np.linspace(0.08, 0.92, L)
    pos = []
    for x, n in zip(xs, sizes):
        m = min(n, max_neurons)
        ys = np.linspace(0.9, 0.1, m) if m > 1 else np.array([0.5])
        pos.append((x, ys, n > max_neurons))
    for (x0, y0, _), (x1, y1, _) in zip(pos[:-1], pos[1:]):
        for a in y0:
            for b in y1:
                ax.plot([x0, x1], [a, b], color="#b0bec5", lw=0.4, zorder=1)
    for k, ((x, ys, cut), n) in enumerate(zip(pos, sizes)):
        fc = _IN if k == 0 else (_OUT if k == L - 1 else _HID)
        for j, y in enumerate(ys):
            if cut and j == len(ys) // 2:
                ax.text(x, y, "⋮", ha="center", va="center", fontsize=14, zorder=4)
                continue
            ax.scatter([x], [y], s=150, c=fc, edgecolors=_EDGE, linewidths=1.0, zorder=3)
        label = (names[k] if names else "") + f"\n{n} neurons" if k else (names[k] if names else "input") + f"\n{n} values"
        ax.text(x, -0.02, label, ha="center", va="top", fontsize=8.5)
    if input_names:
        for y, t in zip(pos[0][1], input_names):
            ax.text(pos[0][0] - 0.04, y, t, ha="right", va="center", fontsize=7.5)
    if output_names:
        for y, t in zip(pos[-1][1], output_names):
            ax.text(pos[-1][0] + 0.04, y, t, ha="left", va="center", fontsize=7.5)
    ax.set_xlim(-0.12 if input_names else -0.02, 1.2 if output_names else 1.02)
    ax.set_ylim(-0.2, 1.0)
    ax.axis("off")


def draw_cnn(ax, stages, title=None):
    """Feature maps of a convolutional network: stages = list of (label, height, width, channels);
    every map is a box with depth proportional to log(channels), height to log(size)."""
    import matplotlib.patches as mp
    x = 0.0
    for k, (lab, h, w, c) in enumerate(stages):
        hh = 0.25 + 0.12 * math.log2(max(h, 1))
        dd = 0.08 + 0.05 * math.log2(max(c, 1))
        fc = _IN if k == 0 else (_OUT if k == len(stages) - 1 else _HID)
        if h > 1:
            for j in range(3):
                ax.add_patch(mp.Rectangle((x + j * dd / 3, -hh / 2 + j * 0.05), 0.12, hh, fc=fc, ec=_EDGE, lw=0.8, zorder=3 + j))
            ax.text(x + 0.1, -hh / 2 - 0.12, f"{h}×{w}×{c}", ha="center", va="top", fontsize=7.5)
            ax.text(x + 0.1, hh / 2 + 0.25 + 0.35 * (k % 2), lab, ha="center", va="bottom", fontsize=8)
            x_next = x + 0.12 + dd + 0.35
        else:
            ax.add_patch(mp.Rectangle((x, -0.6), 0.08, 1.2, fc=fc, ec=_EDGE, lw=0.8, zorder=3))
            ax.text(x + 0.04, -0.72, f"{c}", ha="center", va="top", fontsize=7.5)
            ax.text(x + 0.04, 0.85 + 0.35 * (k % 2), lab, ha="center", va="bottom", fontsize=8)
            x_next = x + 0.08 + 0.35
        if k < len(stages) - 1:
            _arrow(ax, (x_next - 0.3, 0), (x_next - 0.05, 0), lw=0.9)
        x = x_next
    ax.set_xlim(-0.1, x)
    ax.set_ylim(-1.6, 2.2)
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=9.5)


def draw_mobilenet_block(ax, c_in=24, expansion=88, c_out=24, k=3, se=False, stride=1):
    """Inverted residual block of MobileNet V2/V3: 1x1 expansion, k x k depthwise convolution,
    optional squeeze-and-excitation, 1x1 linear projection, residual connection when
    stride = 1 and C_in = C_out; the number of weights of every layer is written under it."""
    steps = [("input", f"{c_in} ch.", _IN, None),
             ("1×1 conv\nexpansion", f"{c_in}→{expansion}", _OP, c_in * expansion),
             (f"{k}×{k} depthwise\nconv, stride {stride}", f"{expansion} ch.", _OP, k * k * expansion)]
    if se:
        steps.append(("squeeze-\nexcitation", "channel weights", _OP, 2 * expansion * max(8, expansion // 4)))
    steps += [("1×1 conv\nprojection", f"{expansion}→{c_out}", _OP, expansion * c_out), ("output", f"{c_out} ch.", _OUT, None)]
    xs = np.linspace(0.07, 0.93, len(steps))
    for x, (t, sub, fc, n) in zip(xs, steps):
        _box(ax, x, 0.5, 0.13, 0.3, fc, t, 8)
        ax.text(x, 0.3, sub, ha="center", va="top", fontsize=7.5)
        if n is not None:
            ax.text(x, 0.2, f"{n:,} weights".replace(",", " "), ha="center", va="top", fontsize=7.5, color="#1565c0")
    for a, b in zip(xs[:-1], xs[1:]):
        _arrow(ax, (a + 0.065, 0.5), (b - 0.065, 0.5))
    if stride == 1 and c_in == c_out:
        ax.annotate("", xy=(xs[-1], 0.66), xytext=(xs[0], 0.66),
                    arrowprops=dict(arrowstyle="-|>", lw=1.0, color="#2e7d32", connectionstyle="arc3,rad=-0.12"))
        ax.text(0.5, 0.86, "residual connection: output = input + F(input)", ha="center", fontsize=8, color="#2e7d32")
    std = k * k * expansion * expansion
    ax.text(0.5, 0.02, (f"a standard {k}×{k} convolution {expansion}→{expansion} channels would need {std:,} weights; "
                        f"the depthwise one needs {k * k * expansion:,}").replace(",", " "), ha="center", fontsize=8)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.05, 1.0)
    ax.axis("off")



# ---- Topic 4, Lessons 4.3-4.4: verification of thermal candidates, Grad-CAM -------------------
def crop_square(img, cx, cy, side):
    """Square fragment side x side centred at (cx, cy); parts outside the frame are filled by
    reflection. The result is a new array (not a view of the frame)."""
    side = int(max(8, round(side)))
    x0, y0 = int(round(cx - side / 2)), int(round(cy - side / 2))
    H, W = img.shape[:2]
    xa, ya, xb, yb = max(0, x0), max(0, y0), min(W, x0 + side), min(H, y0 + side)
    return cv2.copyMakeBorder(img[ya:yb, xa:xb], ya - y0, y0 + side - yb, xa - x0, x0 + side - xb, cv2.BORDER_REFLECT)


def thermal_candidates(img, objects, gsd, size=48, scale=1.5, targets=("Person",), k=6.0):
    """Candidates of the classical warm-object detector of Topic 3 in a white-hot thermal frame,
    labelled by the annotation: 'target' (the centre lies in a box of a target class), 'background'
    (false alarm: outside all annotated boxes) or 'other object' (inside a box of another class).
    Every candidate gets a grey fragment of scale x the longer side of its box, size x size px.
    Returns a list of dicts with box, label, side (px) and chip."""
    boxes, _ = detect_warm_objects(img, gsd, k=k)
    if len(boxes) == 0:
        return []
    m = match_detections(boxes, objects, set(targets))
    lab = {tuple(b): "target" for b in m["tp"]}
    lab.update({tuple(b): "background" for b in m["fp"]})
    lab.update({tuple(b): "other object" for b in m["other"]})
    g = gray(img)
    out = []
    for b in boxes:
        side = max(b[2] - b[0], b[3] - b[1], 8)
        chip = crop_square(g, (b[0] + b[2]) / 2, (b[1] + b[3]) / 2, scale * side)
        out.append(dict(box=[float(v) for v in b], label=lab[tuple(b)], side=float(side),
                        chip=cv2.resize(chip, (size, size), interpolation=cv2.INTER_AREA)))
    return out


def grad_cam(model, x, layer, class_index=None):
    """Grad-CAM (Selvaraju et al., 2017): map of the regions of the input that raise the score of a
    class. The feature maps A_k of `layer` are weighted by the mean gradient of the class score over
    each map, summed and passed through ReLU. x: tensor 1 x 3 x H x W. Returns (map H x W in 0...1,
    class index, probabilities)."""
    import torch
    acts, grads = [], []
    h1 = layer.register_forward_hook(lambda m, i, o: acts.append(o))
    h2 = layer.register_full_backward_hook(lambda m, gi, go: grads.append(go[0]))
    model.eval()
    try:
        x = x.clone().requires_grad_(True)
        out = model(x)
        c = int(out.argmax(1)) if class_index is None else int(class_index)
        model.zero_grad()
        out[0, c].backward()
        a, g = acts[0][0].detach(), grads[0][0].detach()
        cam = torch.relu((g.mean((1, 2))[:, None, None] * a).sum(0)).cpu().numpy()
    finally:
        h1.remove()
        h2.remove()
    cam = cv2.resize(cam, (x.shape[3], x.shape[2]), interpolation=cv2.INTER_LINEAR)
    cam = cam / cam.max() if cam.max() > 0 else cam
    return cam, c, torch.softmax(out.detach(), 1)[0].cpu().numpy()
'''

with open(os.path.join(WORK_DIR, "aitbc.py"), "w", encoding="utf-8") as fh:
    fh.write(LIB_SOURCE)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
import aitbc
aitbc = importlib.reload(aitbc)
aitbc.setup_plots()

if os.environ.get("AITBC_DATA"):
    cache_dir = os.environ["AITBC_DATA"]
elif CACHE_ON_DRIVE and IN_COLAB:
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    cache_dir = "/content/drive/MyDrive/AITBC/data_cache"
else:
    cache_dir = os.path.join(WORK_DIR, "aitbc_data")
aitbc.set_data_root(cache_dir)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)

t0 = time.time()
hit = aitbc.HITUAV()                                   # thermal frames of a reconnaissance UAV
mv = aitbc.MilitaryVehicles() if PREFETCH_ALL else None
VERSIONS = {"Python": platform.python_version(), "NumPy": np.__version__, "OpenCV": cv2.__version__,
            "pandas": pd.__version__, "scikit-learn": sklearn.__version__, "scikit-image": skimage.__version__,
            "PyTorch": torch.__version__, "torchvision": torchvision.__version__, "aitbc": aitbc.__version__}
n_files, size_mb = aitbc.cache_size()
dev = torch.cuda.get_device_name(0) if DEVICE == "cuda" else f"CPU, {os.cpu_count()} logical cores"
display(pd.DataFrame({"Value": list(VERSIONS.values()) + ["Google Colab" if IN_COLAB else "local Jupyter", dev,
                                                           f"{aitbc.DATA_ROOT}: {n_files} files, {size_mb:.0f} MB",
                                                           f"{time.time() - t0:.0f} s"]},
                     index=list(VERSIONS) + ["environment", "computing device", "cache of the data sets", "preparation time"]))

### Block 0.2. Course project

**Purpose.** To connect the course project created in Notebook 02. The results of this notebook are saved in it: figures and tables in `outputs`, the verifier of Lesson 4.3 and the models of Lesson 4.4 (PyTorch, ONNX) in `models` and the parameters of the lessons in `project.json`.

**Parameters.** `USE_DRIVE` — the project is on Google Drive (Colab asks for access to the Google account); otherwise it is on the disk of the session. `PROJECT_PATH` — folder of the project; the same as in Notebooks 02–06.

**Output.** The folders of the project and the lessons already recorded in `project.json`.

In [ ]:
#@title 0.2 Course project { display-mode: "form" }
USE_DRIVE = True #@param {type:"boolean"}
PROJECT_PATH = "AITBC/AITBC_project" #@param {type:"string"}

if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    PROJECT_ROOT = os.path.join("/content/drive/MyDrive", PROJECT_PATH)
else:
    PROJECT_ROOT = os.path.join(WORK_DIR, PROJECT_PATH)
    if USE_DRIVE:
        print("Google Drive is available only in Google Colab; the project is used in", PROJECT_ROOT)
for d in ["data/images", "data/video", "data/thermal", "data/annotations", "frames/visible", "frames/thermal",
          "models", "outputs/figures", "outputs/video", "outputs/reports"]:
    os.makedirs(os.path.join(PROJECT_ROOT, d), exist_ok=True)
FIG_DIR = os.path.join(PROJECT_ROOT, "outputs", "figures")
REPORT_DIR = os.path.join(PROJECT_ROOT, "outputs", "reports")
MODEL_DIR = os.path.join(PROJECT_ROOT, "models")


def update_project_json(lesson, extra=None):
    path = os.path.join(PROJECT_ROOT, "project.json")
    meta = json.load(open(path, encoding="utf-8")) if os.path.exists(path) else {"project": os.path.basename(PROJECT_ROOT)}
    meta.setdefault("lessons", [])
    if lesson not in meta["lessons"]:
        meta["lessons"].append(lesson)
    meta["updated"] = time.strftime("%Y-%m-%d %H:%M:%S")
    meta["versions"] = VERSIONS
    meta.update(extra or {})
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(meta, fh, indent=2, ensure_ascii=False, default=str)
    print(f"project.json updated: lesson {lesson}" + (f", keys: {', '.join(extra)}" if extra else ""))


meta = json.load(open(os.path.join(PROJECT_ROOT, "project.json"), encoding="utf-8")) if os.path.exists(os.path.join(PROJECT_ROOT, "project.json")) else {}
print("Project:", PROJECT_ROOT)
print("Lessons recorded in project.json:", ", ".join(meta.get("lessons", [])) or "none (new project)")

---
# Lesson 4.3 (Practical class). Classification of thermal signatures by HOG + SVM: target — background

**Learning questions**
1. Formation of a data set of thermal signatures: the targets and the background.
2. Computation of the HOG features of fragments of thermal frames.
3. Division of the data into the training and the test part.
4. Training of the SVM classifier and the choice of the threshold of the decision.
5. Analysis of the errors of classification.

**Task of the lesson.** At night the thermal camera of a reconnaissance UAV shows a person as a small warm spot of 10–30 pixels. The classical detector of Topic 3 (white top-hat and threshold, Notebook 05) finds most such spots, but also many warm objects that are not persons: stones and roofs heated by the day, engines, lamps, animals. In this lesson a **verifier** is built: every candidate of the detector is cut out as a fragment, described by its HOG vector and classified by an SVM into two classes, **target** (person) and **background** (false alarm). The verifier keeps the targets and rejects most false alarms.

**Result of the lesson.** The trained verifier `models/thermal_person_verifier_hog_svm.joblib`, its average precision on flight conditions not used in the training, the precision and the recall of the detector before and after the verification, the galleries of the errors in `outputs`, the parameters in `project.json`.

## Task 4.3.1. Thermal frames and candidates of the classical detector

**Data.** The file name of a HIT-UAV frame `T_HHH_AA_W_NNNNN.jpg` gives the time of day (`T`: 0 — day, 1 — night), the flight altitude `HHH` in metres and the camera angle `AA` in degrees (90° — vertical view); `NNNNN` is the number of the frame. The frames of one **flight condition** (time of day, altitude, angle) are consecutive frames of the same flight. For every condition `FRAMES_PER_CONDITION` frames are taken at random.

**Candidates.** In every frame the warm-object detector of Topic 3 is run with the size of a person in pixels computed from the ground sampling distance of the centre row (formula of Notebook 05):
$$\text{GSD} = \frac{h}{\sin\delta}\cdot\frac{2\tan(\alpha_H/2)}{N},\qquad \delta = \text{camera angle},\ \alpha_H = 25°,\ N = 640 .$$
Every candidate (box of a warm spot) is labelled by the annotation of the authors: **target** — its centre lies in a box of a person; **background** — it lies outside all annotated boxes (false alarm); **other object** — it lies in a box of a vehicle or a bicycle (it is not used for the two classes). The fragment of a candidate is a square of 1.5 × the longer side of its box around its centre, stored as 48 × 48 px.

**Parameters.**
- `FRAMES_PER_CONDITION` — frames per flight condition (105 conditions; 6 frames — about 630 frames, 30 MB);
- `CONDITIONS` — day and night, night only or day only;
- `SHOW_FRAME` — number of the night frame shown.

**Expected result.** The table of the candidates by the time of day, the recall and the precision of the detector, a night frame with the candidates coloured by their label.

**What to observe.** In 630 frames of 105 conditions the detector gives 10 258 candidates. At night it finds 64 % of the persons (848 of 1319) with a precision of 0.18, i.e. about 4.7 false alarms for every found person; by day it finds only 24 % with a precision of 0.08: the ground heated by the sun gives many warm spots, and persons are less contrasting (Notebook 05). About one candidate in five lies on a vehicle or a bicycle (other object). The fragments of persons are small vertical warm spots; the false alarms are parts of roofs, edges of roads, lamps and spots of heated ground.

**Control question 1.** Why is the label of a candidate taken from the annotation and not given by hand, and which error of the annotation becomes an error of the data set of candidates?

In [ ]:
#@title 4.3.1 Frames of flight conditions and candidates of the detector (code shown)
FRAMES_PER_CONDITION = 6 #@param {type:"slider", min:2, max:15, step:1}
CONDITIONS = "day and night" #@param ["day and night", "night only", "day only"]
SHOW_FRAME = 0 #@param {type:"integer"}

frames = hit.images.copy()
frames["condition"] = np.where(frames.night, "night", "day") + " " + frames.altitude.astype(str) + " m " + frames.angle.astype(str) + "°"
if CONDITIONS != "day and night":
    frames = frames[frames.night == (CONDITIONS == "night only")]
FR = frames.sample(frac=1.0, random_state=0).groupby("condition").head(FRAMES_PER_CONDITION).sort_values("file").reset_index(drop=True)
t0 = time.time()
hit.prefetch(FR.file.tolist())
t_load = time.time() - t0
rows = []
t0 = time.time()
for r in FR.itertuples():
    g, objs = hit.load(r.file)
    gsd = float(aitbc.oblique_gsd(r.altitude, r.angle, [g.shape[0] / 2])[0][0])     # GSD of the centre row, m/px
    for c in aitbc.thermal_candidates(g, objs, gsd, size=48):                         # detector of Topic 3 + labels
        rows.append(dict(file=r.file, condition=r.condition, night=r.night, altitude=r.altitude, angle=r.angle,
                         split_authors=r.split, **c))
CAND = pd.DataFrame(rows)
t_cand = time.time() - t0
tab = CAND.pivot_table(index="night", columns="label", values="side", aggfunc="count", fill_value=0)
tab.index = ["day" if not v else "night" for v in tab.index]
tab["persons in the frames"] = [int(FR[FR.night == (k == "night")].Person.sum()) for k in tab.index]
tab["recall of the detector"] = (tab["target"] / tab["persons in the frames"]).round(3)
tab["precision of the detector"] = (tab["target"] / (tab["target"] + tab["background"])).round(3)
display(tab.style.set_caption(f"{len(FR)} frames of {FR.condition.nunique()} flight conditions: {len(CAND)} candidates "
                              f"(download {t_load:.0f} s, detection {t_cand:.0f} s)"))
night_files = FR[FR.night].file.tolist() or FR.file.tolist()
f_show = night_files[SHOW_FRAME % len(night_files)]
g, objs = hit.load(f_show)
vis = aitbc.to_rgb(g).copy()
for o in objs:
    x1, y1, x2, y2 = [int(v) for v in o["box"]]
    cv2.rectangle(vis, (x1, y1), (x2, y2), (255, 255, 255), 1)
COL = {"target": (0, 230, 0), "background": (255, 40, 40), "other object": (255, 200, 0)}
for c in CAND[CAND.file == f_show].itertuples():
    x1, y1, x2, y2 = [int(v) for v in c.box]
    cv2.rectangle(vis, (x1 - 2, y1 - 2), (x2 + 2, y2 + 2), COL[c.label], 2)
fig, ax = plt.subplots(1, 2, figsize=(17, 6.4), gridspec_kw={"width_ratios": [1.4, 1]})
ax[0].imshow(vis)
ax[0].set_title(f"{f_show}: annotation white; candidates: target green, false alarm red, other object yellow", fontsize=9)
ax[0].axis("off")
show = pd.concat([CAND[CAND.label == "target"].sample(min(8, (CAND.label == "target").sum()), random_state=1),
                  CAND[CAND.label == "background"].sample(min(8, (CAND.label == "background").sum()), random_state=1)])
tiles = np.vstack([np.hstack([np.pad(c, 2, constant_values=255) for c in show.chip.values[k * 8:(k + 1) * 8]]) for k in range(2)])
ax[1].imshow(tiles, cmap="gray")
ax[1].set_title("fragments 48 × 48: upper row targets (persons), lower row false alarms", fontsize=9)
ax[1].axis("off")
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "lesson43_candidates.png"), dpi=80)
plt.show()

## Task 4.3.2. HOG features of thermal signatures

**Method.** The fragment is reduced to $S \times S$ pixels and its HOG vector is computed (Lesson 4.1): gradient, histograms of orientations in cells of $c \times c$ pixels, normalisation in blocks of 2 × 2 cells. The length of the vector is
$$N = \left(\frac{S}{c} - 1\right)^2 \cdot 4 \cdot n_o;\qquad S = 32,\ c = 4,\ n_o = 9:\ N = 7^2 \cdot 4 \cdot 9 = 1764 .$$
A small cell is needed: a person of 10–30 px occupies only a few cells of 4 × 4 px of a fragment of 32 × 32 px.

**Parameters.** `HOG_SIZE` — $S$; `CELL` — $c$; `ORIENTATIONS` — $n_o$.

**Expected result.** The fragments of targets and false alarms with their HOG pictures, the mean HOG picture of both classes and the length of the vector.

**What to observe.** A person at night gives a compact bright spot with gradients of all orientations around it (a "ring" of the HOG); a false alarm is more often a part of a straight edge (a roof, a road, a wall) with one dominating orientation, or a spot without a clear shape. The mean pictures of the classes differ in the centre of the fragment.

**Control question 2.** What is the length of the HOG vector for $S = 24$, $c = 4$, $n_o = 9$, and why does a cell of 8 × 8 px describe a person of 12 px poorly?

In [ ]:
#@title 4.3.2 HOG of thermal fragments (code shown)
HOG_SIZE = 32 #@param [24, 32, 48] {type:"raw"}
CELL = 4 #@param [4, 8] {type:"raw"}
ORIENTATIONS = 9 #@param [6, 9, 12] {type:"raw"}

from skimage.feature import hog
t0 = time.time()
small = [cv2.resize(c, (HOG_SIZE, HOG_SIZE), interpolation=cv2.INTER_AREA) for c in CAND.chip]
XH = aitbc.hog_features(small, size=HOG_SIZE, cell=CELL, orientations=ORIENTATIONS)     # N x D
t_hog = time.time() - t0
print(f"HOG of {len(XH)} fragments: {XH.shape[1]} values each (formula: {aitbc.hog_length(HOG_SIZE, CELL, 2, ORIENTATIONS)}), {t_hog:.1f} s")
fig, ax = plt.subplots(3, 8, figsize=(17, 6.6))
for j, lab in enumerate(["target", "background"]):
    idx = CAND.index[CAND.label == lab][:4]
    for k, i in enumerate(idx):
        _, hi = hog(small[i], orientations=ORIENTATIONS, pixels_per_cell=(CELL, CELL), cells_per_block=(2, 2), visualize=True)
        ax[0, 4 * j + k].imshow(small[i], cmap="gray")
        ax[0, 4 * j + k].set_title(lab if lab == "target" else "false alarm", fontsize=8.5)
        ax[1, 4 * j + k].imshow(hi, cmap="gray")
for j, lab in enumerate(["target", "background"]):
    m_img = np.mean([hog(small[i], orientations=ORIENTATIONS, pixels_per_cell=(CELL, CELL), cells_per_block=(2, 2), visualize=True)[1]
                     for i in CAND.index[CAND.label == lab][:300]], axis=0)
    ax[2, 4 * j].imshow(np.mean([small[i] for i in CAND.index[CAND.label == lab][:300]], axis=0), cmap="gray")
    ax[2, 4 * j].set_title("mean fragment", fontsize=8.5)
    ax[2, 4 * j + 1].imshow(m_img, cmap="gray")
    ax[2, 4 * j + 1].set_title("mean HOG picture", fontsize=8.5)
for a in ax.ravel():
    a.axis("off")
fig.suptitle("left: targets (persons); right: false alarms of the detector", fontsize=10)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "lesson43_hog.png"), dpi=80)
plt.show()

## Task 4.3.3. Division into the training and the test part

**Method.** The candidates of the classes *target* and *background* form the data set of the verifier: $\mathbf x_i$ — HOG vector, $y_i = 1$ for a target and $y_i = 0$ for the background. The quality must be estimated on frames of flights that the classifier has not seen (Lesson 4.1, block 4.1.6). Two divisions are compared:
- **by flight condition** — a random share `TEST_SHARE` of the conditions (time of day, altitude, angle) goes completely into the test part (`GroupShuffleSplit`, the condition is the group);
- **official split of the authors** — the parts `train`/`val` and `test` of HIT-UAV, formed by random frames: the neighbouring frames of the same flight fall into different parts.

**Parameters.** `SPLIT`; `TEST_SHARE` — share of the conditions in the test part.

**Expected result.** The table of the two parts (frames, conditions, targets, false alarms), the number of conditions present in both parts, and the share of the test frames of the official split whose neighbouring frame (number ±1) is in the training part.

**What to observe.** With the division by flight condition the 26 test conditions (149 frames) do not occur in the training part (78 conditions, 431 frames). In the official split of the authors 33.7 % of the 579 test frames have the neighbouring frame of the same flight in the training part. About one candidate in seven is a target (0.13–0.14). In the check of the course the average precision of the verifier was not higher with the official split than with the division by condition (the test parts differ), because the verifier classifies small fragments of separate frames; the division by flight condition is still the one that estimates the work on a new flight.

**Control question 3.** Why is the share of targets among the candidates (about one in seven) important for the training and for the estimation of the classifier?

In [ ]:
#@title 4.3.3 Training and test part (code shown)
SPLIT = "by flight condition" #@param ["by flight condition", "official split of the authors (random frames)"]
TEST_SHARE = 0.25 #@param {type:"slider", min:0.15, max:0.4, step:0.05}

from sklearn.model_selection import GroupShuffleSplit
use = CAND.label.isin(["target", "background"]).values
D = CAND[use].reset_index(drop=True)
X = XH[use]
y = (D.label == "target").astype(int).values
if SPLIT.startswith("by flight"):
    tr_idx, te_idx = next(GroupShuffleSplit(1, test_size=TEST_SHARE, random_state=0).split(X, y, groups=D.condition))
else:
    tr_idx, te_idx = np.where(D.split_authors != "test")[0], np.where(D.split_authors == "test")[0]
parts = {}
for name, ii in (("training", tr_idx), ("test", te_idx)):
    parts[name] = {"frames": D.file.values[ii].tolist(), "conditions": set(D.condition.values[ii]),
                   "targets": int(y[ii].sum()), "false alarms": int((1 - y[ii]).sum()),
                   "night candidates": int(D.night.values[ii].sum())}
display(pd.DataFrame({k: {"frames": len(set(v["frames"])), "flight conditions": len(v["conditions"]), "targets": v["targets"],
                          "false alarms": v["false alarms"], "share of targets": round(v["targets"] / (v["targets"] + v["false alarms"]), 3),
                          "candidates at night": v["night candidates"]} for k, v in parts.items()}).T)
both = parts["training"]["conditions"] & parts["test"]["conditions"]
print(f"Flight conditions present in both parts: {len(both)} of {D.condition.nunique()}")
# neighbouring frames in the official split of the authors (all 2898 frames of HIT-UAV)
allf = hit.images.copy()
allf["num"] = allf.file.str.extract(r"_(\d+)\.jpg$")[0].astype(int)
allf["cond"] = allf.file.str.rsplit("_", n=1).str[0]
train_keys = set(zip(allf.cond[allf.split != "test"], allf.num[allf.split != "test"]))
te_all = allf[allf.split == "test"]
nb = np.mean([((c, n - 1) in train_keys) or ((c, n + 1) in train_keys) for c, n in zip(te_all.cond, te_all.num)])
print(f"Official split: {100 * nb:.1f} % of the {len(te_all)} test frames have a neighbouring frame (number ± 1) in the training part")

## Task 4.3.4. Training of the SVM and the threshold of the decision

**Method.** The SVM computes for a candidate the value of the decision function $f(\mathbf x)$ (Lesson 4.1): $f > 0$ — target, $f < 0$ — background; $|f|$ grows with the distance from the boundary. With unequal classes (about one target for seven false alarms) the weights of the errors are equalised by `class_weight="balanced"`: an error on a class of $n_k$ examples costs $n / (2 n_k)$. The decision is taken with a **threshold** $\tau$: the candidate is accepted when $f(\mathbf x) > \tau$. Moving $\tau$ exchanges recall for precision; the curve *precision — recall* over all thresholds and its area, the **average precision** (AP), describe the verifier independently of the threshold:
$$\text{AP} = \sum_k (R_k - R_{k-1})\, P_k ,$$
where $P_k, R_k$ are the precision and the recall at the $k$-th threshold. A random verifier has AP equal to the share of targets.

**Parameters.**
- `KERNEL` — RBF or linear SVM; `C`; `BALANCED` — equalise the classes;
- `SELECT_C` — choose $C$ by cross-validation with 3 folds of flight conditions inside the training part (4 values of $C$, 4–5 minutes on a processor);
- `THRESHOLD` — $\tau$.

**Expected result.** The time of the training, the number of support vectors, the AP at night and by day, the curves precision — recall with the point of the chosen threshold, the precision, recall and F1 at the threshold and the confusion matrix.

**What to observe.** The SVM with the RBF kernel is trained in about 20 s on about 6000 candidates and keeps about half of them (3074) as support vectors. The AP at night is 0.82 against 0.14 for a random verifier; by day only 0.43. At τ = 0 the precision is 0.85 and the recall 0.50: half of the persons are rejected. To keep 90 % of the targets the threshold must be lowered to about τ = −0.8, where the precision is 0.35: about two false alarms for every accepted person instead of about six among the candidates of the detector. The choice of C by cross-validation (`SELECT_C`, about 4–5 minutes) gives AP 0.61 for C = 0.1 and 0.70 for C = 1–100.

**Control question 4.** The verifier must not lose more than 10 % of the persons found by the detector. How is the threshold chosen, and on which part of the data?

In [ ]:
#@title 4.3.4 SVM verifier: training, precision-recall curve, threshold (code shown)
KERNEL = "RBF" #@param ["RBF", "linear"]
C = 10 #@param [0.1, 1, 10, 100] {type:"raw"}
BALANCED = True #@param {type:"boolean"}
SELECT_C = False #@param {type:"boolean"}
THRESHOLD = 0.0 #@param {type:"slider", min:-2, max:2, step:0.25}

from sklearn.svm import SVC
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold
from sklearn.metrics import precision_recall_curve, average_precision_score, confusion_matrix, precision_recall_fscore_support


def make_svm(c, n):
    cw = "balanced" if BALANCED else None
    if KERNEL == "RBF":
        return make_pipeline(StandardScaler(), SVC(C=c, gamma="scale", class_weight=cw))
    return make_pipeline(StandardScaler(), SGDClassifier(loss="hinge", alpha=1.0 / (c * n), class_weight=cw, max_iter=200, tol=1e-4, random_state=0))


Xtr, ytr, Xte, yte = X[tr_idx], y[tr_idx], X[te_idx], y[te_idx]
if SELECT_C:
    cv_rows = []
    for c in (0.1, 1, 10, 100):
        aps = []
        for a, b in GroupKFold(3).split(Xtr, ytr, D.condition.values[tr_idx]):
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                m_ = make_svm(c, len(a)).fit(Xtr[a], ytr[a])
            aps.append(average_precision_score(ytr[b], m_.decision_function(Xtr[b])))
        cv_rows.append({"C": c, "AP (cross-validation)": np.mean(aps), "±": np.std(aps)})
    cv_tab = pd.DataFrame(cv_rows).set_index("C")
    display(cv_tab.style.format(precision=3).set_caption("Choice of C on the training part (3 folds of flight conditions)"))
    C = float(cv_tab["AP (cross-validation)"].idxmax())
    print("Chosen C =", C)
t0 = time.time()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    VERIFIER = make_svm(C, len(Xtr)).fit(Xtr, ytr)
t_fit = time.time() - t0
s_te = VERIFIER.decision_function(Xte)
night_te = D.night.values[te_idx]
pred = (s_te > THRESHOLD).astype(int)
P_, R_, F_, _ = precision_recall_fscore_support(yte, pred, average="binary", zero_division=0)
rows = {"training candidates (targets / false alarms)": f"{len(ytr)} ({int(ytr.sum())} / {int(len(ytr) - ytr.sum())})",
        "time of the training": f"{t_fit:.1f} s",
        "support vectors": int(VERIFIER[-1].n_support_.sum()) if KERNEL == "RBF" else "— (linear model)",
        "AP at night / by day / all": " / ".join(f"{average_precision_score(yte[m], s_te[m]):.3f}" if yte[m].sum() else "—"
                                              for m in (night_te, ~night_te, np.ones(len(yte), bool))),
        "AP of a random verifier (share of targets)": f"{yte.mean():.3f}",
        f"precision / recall / F1 at τ = {THRESHOLD:+.2f}": f"{P_:.3f} / {R_:.3f} / {F_:.3f}"}
pc, rc, thc = precision_recall_curve(yte, s_te)
k90 = int(np.where(rc[:-1] >= 0.9)[0].max())            # largest threshold that keeps 90 % of the targets
rows["τ keeping 90 % of the targets (illustration on the test part): τ, precision"] = f"{thc[k90]:+.2f}, {pc[k90]:.3f}"
display(pd.DataFrame({"value": list(rows.values())}, index=list(rows)))
fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
for m, lab, col in ((night_te, "night", "#1565c0"), (~night_te, "day", "#ef6c00")):
    if yte[m].sum():
        p, r, th = precision_recall_curve(yte[m], s_te[m])
        ax[0].plot(r, p, color=col, label=f"{lab}: AP {average_precision_score(yte[m], s_te[m]):.3f}")
        k = np.searchsorted(th, THRESHOLD)
        if k < len(th):
            ax[0].plot(r[k], p[k], "o", color=col)
ax[0].set_xlabel("recall")
ax[0].set_ylabel("precision")
ax[0].set_title(f"verifier HOG + {KERNEL} SVM (C = {C:g}); points: τ = {THRESHOLD:+.2f}", fontsize=9)
ax[0].grid(alpha=0.3)
ax[0].legend()
cm = confusion_matrix(yte, pred)
ax[1].imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax[1].text(j, i, cm[i, j], ha="center", va="center", fontsize=12, color="w" if cm[i, j] > cm.max() / 2 else "k")
ax[1].set_xticks([0, 1])
ax[1].set_xticklabels(["background", "target"])
ax[1].set_yticks([0, 1])
ax[1].set_yticklabels(["false alarm", "person"])
ax[1].set_xlabel("decision of the verifier")
ax[1].set_ylabel("label of the candidate")
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "lesson43_precision_recall.png"), dpi=80)
plt.show()

## Task 4.3.5. Analysis of the errors

**Method.** At the threshold $\tau$ two kinds of errors are studied on the test part: **missed targets** (persons rejected by the verifier, false negatives) and **accepted false alarms** (false positives). The errors are grouped by the time of day, the flight altitude, the camera angle and the size of the candidate. Then the verifier is put after the detector: in every test frame the detector gives its candidates and the verifier keeps those with $f > \tau$; the recall, the precision and the number of false alarms per frame of the whole pipeline are compared with the detector alone.

**Parameters.** `SHOW_TEST_FRAME` — number of the night test frame shown with the decisions.

**Expected result.** The galleries of the missed targets and of the accepted false alarms, the table of the errors by groups, the table *detector — detector + verifier* for night and day, a test frame with the kept (green) and rejected (red) candidates and the persons missed by the pipeline (yellow).

**What to observe.** At τ = 0 the verifier rejects almost all false alarms: at night their number falls from 12.9 to 0.17 per frame and the precision of the pipeline rises from 0.20 to 0.91, but the recall falls from 0.66 to 0.37. With τ = −0.5 (official split of the authors, in the check of the course) the night recall falls only from 0.72 to 0.61 at a precision of 0.55 and 1.8 false alarms per frame instead of 13.2. Small candidates (up to 10 px) are rejected most often (recall 0.32 against 0.62 for 10–20 px): a fragment of a few pixels has almost no shape. By day the recall of the pipeline is below 0.1. Some accepted false alarms look like persons: they may be persons missing in the annotation.

**Control question 5.** Which errors of the verifier are caused by the data (small size, day, annotation) and which by the classifier, and how can each kind be reduced?

In [ ]:
#@title 4.3.5 Errors of the verifier and the effect on the whole detector (code shown)
SHOW_TEST_FRAME = 0 #@param {type:"integer"}

Dte = D.iloc[te_idx].reset_index(drop=True)
Dte["score"], Dte["accepted"] = s_te, s_te > THRESHOLD
fn = Dte[(Dte.label == "target") & ~Dte.accepted].sort_values("score")
fp = Dte[(Dte.label == "background") & Dte.accepted].sort_values("score", ascending=False)
fig, ax = plt.subplots(2, 8, figsize=(17, 5.2))
for row, (tab_, name) in enumerate(((fn, "missed person"), (fp, "accepted false alarm"))):
    for k in range(8):
        a = ax[row, k]
        a.axis("off")
        if k < len(tab_):
            r = tab_.iloc[k]
            a.imshow(r.chip, cmap="gray")
            a.set_title(f"{name}\n{r.condition}, f = {r.score:+.2f}", fontsize=7.5)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "lesson43_errors.png"), dpi=80)
plt.show()
Dte["size, px"] = pd.cut(Dte.side, [0, 10, 20, 40, 1000], labels=["≤ 10", "10–20", "20–40", "> 40"])
Dte["time of day"] = np.where(Dte.night, "night", "day")
Dte["altitude, m"] = pd.cut(Dte.altitude, [0, 80, 110, 200], labels=["60–80", "90–110", "120–130"])


def err_table(by):
    g_ = Dte.groupby(by, observed=True)
    return pd.DataFrame({"targets": g_.apply(lambda t: int((t.label == "target").sum())),
                         "recall of the verifier": g_.apply(lambda t: ((t.label == "target") & t.accepted).sum() / max(1, (t.label == "target").sum())),
                         "false alarms": g_.apply(lambda t: int((t.label == "background").sum())),
                         "accepted false alarms, %": g_.apply(lambda t: 100 * ((t.label == "background") & t.accepted).sum() / max(1, (t.label == "background").sum()))})


for by in ("time of day", "altitude, m", "size, px"):
    display(err_table(by).style.format({"recall of the verifier": "{:.3f}", "accepted false alarms, %": "{:.1f}"}).set_caption(f"Errors by {by}"))
# the whole pipeline on the test frames: detector alone and detector + verifier
te_frames = sorted(set(Dte.file))
ff = FR.set_index("file").loc[te_frames]
pipe = []
for night in (True, False):
    fs = ff.index[ff.night == night]
    if len(fs) == 0:
        continue
    t_ = Dte[Dte.file.isin(fs)]
    n_p = int(ff.loc[fs].Person.sum())
    tp0, fp0 = int((t_.label == "target").sum()), int((t_.label == "background").sum())
    tp1, fp1 = int(((t_.label == "target") & t_.accepted).sum()), int(((t_.label == "background") & t_.accepted).sum())
    pipe.append({"test frames": f"{'night' if night else 'day'}: {len(fs)}", "persons": n_p,
                 "detector: recall": tp0 / max(n_p, 1), "detector: precision": tp0 / max(tp0 + fp0, 1), "detector: false alarms per frame": fp0 / len(fs),
                 "+ verifier: recall": tp1 / max(n_p, 1), "+ verifier: precision": tp1 / max(tp1 + fp1, 1), "+ verifier: false alarms per frame": fp1 / len(fs)})
PIPE = pd.DataFrame(pipe).set_index("test frames")
display(PIPE.style.format(precision=3).set_caption(f"Detector of Topic 3 and detector + verifier, τ = {THRESHOLD:+.2f}"))
nf = [f for f in te_frames if ff.loc[f, "night"]] or te_frames
f_show = nf[SHOW_TEST_FRAME % len(nf)]
g, objs = hit.load(f_show)
vis = aitbc.to_rgb(g).copy()
kept_boxes = []
for c in Dte[Dte.file == f_show].itertuples():
    x1, y1, x2, y2 = [int(v) for v in c.box]
    cv2.rectangle(vis, (x1 - 2, y1 - 2), (x2 + 2, y2 + 2), (0, 230, 0) if c.accepted else (255, 40, 40), 2)
    if c.accepted:
        kept_boxes.append(c.box)
m_ = aitbc.match_detections(np.array(kept_boxes).reshape(-1, 4), objs, {"Person"})
for o in m_["missed"]:
    x1, y1, x2, y2 = [int(v) for v in o["box"]]
    cv2.rectangle(vis, (x1 - 3, y1 - 3), (x2 + 3, y2 + 3), (255, 230, 0), 1)
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(vis)
ax.set_title(f"{f_show}: kept green, rejected red, persons missed by the pipeline yellow", fontsize=9)
ax.axis("off")
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "lesson43_pipeline_frame.png"), dpi=80)
plt.show()
PIPE.to_csv(os.path.join(REPORT_DIR, "lesson43_detector_and_verifier.csv"))

## Task 4.3.6. Saving and checking the verifier

**Method.** The trained verifier is a chain *standardisation → SVM*. To be used on another computer it is saved with everything that defines its input: the size of the fragment and its scale relative to the box of the candidate, the parameters of the HOG, the threshold $\tau$ and the classes. The file is loaded again and its decisions on the test part are compared with those of the model in memory.

**Expected result.** The file `models/thermal_person_verifier_hog_svm.joblib` with its size, the agreement of the decisions of the loaded model (must be 1.000) and the record in `project.json`. The file of the RBF SVM is large (about 40 MB): it keeps all support vectors (about 3000 vectors of 1764 values), which are needed to compute $f(\mathbf x)$; the linear SVM keeps one vector of weights (a few kB).

**Control question 6.** The onboard computer computes the HOG with a cell of 8 px instead of 4 px. What happens with the decisions of the saved verifier, and how is such an error prevented?

In [ ]:
#@title 4.3.6 Saving the verifier (code shown)
import joblib
path = os.path.join(MODEL_DIR, "thermal_person_verifier_hog_svm.joblib")
bundle = {"model": VERIFIER, "threshold": float(THRESHOLD), "classes": ["background", "target (person)"],
          "fragment": {"scale_of_box": 1.5, "stored_size_px": 48, "resized_to_px": HOG_SIZE, "palette": "white-hot"},
          "hog": {"orientations": ORIENTATIONS, "pixels_per_cell": CELL, "cells_per_block": 2, "block_norm": "L2-Hys"},
          "detector": "aitbc.detect_warm_objects (Topic 3), GSD of the centre row",
          "data": "HIT-UAV (Suo et al., Scientific Data 10, 227, 2023), " + SPLIT,
          "kernel": KERNEL, "C": float(C), "balanced": BALANCED,
          "AP_test": float(average_precision_score(yte, s_te))}
joblib.dump(bundle, path)
loaded = joblib.load(path)
agree = float(np.mean((loaded["model"].decision_function(Xte) > loaded["threshold"]) == (s_te > THRESHOLD)))
print(f"Saved: {path} ({os.path.getsize(path) / 2 ** 20:.2f} MB); agreement of the decisions of the loaded verifier: {agree:.3f}")
update_project_json("4.3", {"lesson_4_3": {k: v for k, v in bundle.items() if k != "model"} | {
    "model_file": os.path.relpath(path, PROJECT_ROOT), "frames_per_condition": FRAMES_PER_CONDITION, "conditions": CONDITIONS,
    "pipeline": PIPE.round(3).reset_index().to_dict("records")}})

### Report of Lesson 4.3

Double-click this cell and write the answers.

1. Frames, flight conditions and candidates; recall and precision of the detector at night and by day; answer to control question 1:
2. Parameters of the HOG and the length of the vector; differences of the HOG of targets and false alarms; answer to control question 2:
3. Division into the parts; conditions in both parts; neighbouring frames of the official split; answer to control question 3:
4. Kernel, C, AP at night and by day, the chosen threshold and its precision and recall; answer to control question 4:
5. Errors by the time of day, altitude and size; the table detector — detector + verifier; answer to control question 5:
6. File of the verifier and the check after loading; answer to control question 6:

---
# Lesson 4.4 (Practical class). Transfer learning of MobileNet for the recognition of military vehicles and export to ONNX

**Learning questions**
1. Preparation of the data for the training of a neural network: training, validation and test parts, augmentation.
2. Transfer learning: a frozen network trained on ImageNet and a new output layer.
3. Fine-tuning of the last blocks of the network.
4. Evaluation of the model and visualisation of the regions that define its decision (Grad-CAM).
5. Export of the model to ONNX and the check of the inference for the onboard computer.

**Task of the lesson.** MobileNetV3-small, trained on 1.2 million images of ImageNet, is adapted to the recognition of the 7 groups of military vehicles of the course (tank, infantry fighting vehicle, armoured personnel carrier and scout car, artillery, multiple rocket launcher, air defence and radar, missile system) in two stages: first a new output layer is trained on the frozen network, then the last blocks of the network are trained together with it with a small learning rate. In Lesson 4.2 the same frozen network with a linear layer reached an accuracy of 0.63 on the 7 groups (block 4.2.5); the lesson measures what the augmentation and the fine-tuning add.

**Result of the lesson.** The model `models/military_vehicles_mobilenetv3_finetuned.pt` and its ONNX file with the file of labels, the curves of the training, the metrics on the test part and the Grad-CAM pictures in `outputs`, the parameters in `project.json`.

## Task 4.4.1. Preparation of the data and augmentation

**Parts of the data.** The cleaned training part of block 4.1.2 (without repeated copies) gives at most `N_PER_CLASS_44` images of every type; a share `VAL_SHARE` of them (stratified by class) forms the **validation part**, used to choose the number of epochs and to compare the stages. The **test part** is the cleaned test part of Lesson 4.1 (without copies and near duplicates of training images, 1502 images) and is used once, in block 4.4.4.

**Augmentation.** In every epoch every training image is changed at random: a crop of 60–100 % of its area resized to $S \times S$ (`RandomResizedCrop`), a horizontal flip, changes of brightness, contrast and saturation by up to ±30 % (`ColorJitter`). The network sees a new variant of the image in every epoch, which reduces the fitting to the training images. The validation and test images are only resized to $S \times S$. All images are normalised by the mean and the standard deviation of ImageNet:
$$x' = \frac{x/255 - \mu_c}{\sigma_c},\qquad \mu = (0.485,\ 0.456,\ 0.406),\ \sigma = (0.229,\ 0.224,\ 0.225).$$

**Parameters.** `LEVEL_44`; `N_PER_CLASS_44`; `VAL_SHARE`; `INPUT_SIZE` — $S$ (160 px is about twice faster than 224 px); `AUGMENT`.

**Expected result.** The table of the parts and eight augmented variants of one training image.

**Control question 7.** Why must the augmentation not be applied to the validation and test images, and which augmentation would be wrong for thermal frames or for frames of a vertical view?

In [ ]:
#@title 4.4.1 Parts of the data and augmentation (code shown)
LEVEL_44 = "7 groups of the course" #@param ["7 groups of the course", "24 types"]
N_PER_CLASS_44 = 150 #@param {type:"slider", min:50, max:400, step:25}
VAL_SHARE = 0.15 #@param {type:"slider", min:0.1, max:0.3, step:0.05}
INPUT_SIZE = 160 #@param [128, 160, 224] {type:"raw"}
AUGMENT = True #@param {type:"boolean"}

import torch.nn as nn
from torchvision.transforms import v2 as T
from sklearn.model_selection import train_test_split
if mv is None:
    mv = aitbc.MilitaryVehicles()
TRAIN_ALL, TEST_IDX = mv.clean(near=2)
col44 = "cls" if LEVEL_44 == "24 types" else "group"
LABELS = sorted(mv.table.cls.unique()) if col44 == "cls" else list(aitbc.MilitaryVehicles.GROUP_ORDER)
sub = mv.subset(TRAIN_ALL, N_PER_CLASS_44, seed=0)
TR_IDX, VA_IDX = train_test_split(sub, test_size=VAL_SHARE, random_state=0, stratify=mv.table[col44].values[sub])
label_of = lambda idx: torch.tensor([LABELS.index(v) for v in mv.table[col44].values[idx]])
IMG = {}                                           # decoded images, resized once to 1.15 S (for the random crops)


def image_for(i, s):
    if (i, s) not in IMG:
        IMG[(i, s)] = mv.image(i, size=s)
    return IMG[(i, s)]


AUG = T.Compose([T.ToImage(), T.RandomResizedCrop(INPUT_SIZE, scale=(0.6, 1.0), antialias=True), T.RandomHorizontalFlip(),
                 T.ColorJitter(0.3, 0.3, 0.3), T.ToDtype(torch.float32, scale=True),
                 T.Normalize(aitbc.IMAGENET_MEAN.tolist(), aitbc.IMAGENET_STD.tolist())])


def batches(idx, train, bs=64):
    # mini-batches (x, y): augmented training images or resized validation / test images
    for b in range(0, len(idx), bs):
        ii = idx[b:b + bs]
        if train and AUGMENT:
            x = torch.stack([AUG(image_for(i, int(INPUT_SIZE * 1.15))) for i in ii])
        else:
            x = torch.from_numpy(aitbc.imagenet_batch([image_for(i, INPUT_SIZE) for i in ii], INPUT_SIZE))
        yield x.to(DEVICE), label_of(ii).to(DEVICE)


parts44 = pd.DataFrame({"images": [len(TR_IDX), len(VA_IDX), len(TEST_IDX)],
                        "classes": [len(set(mv.table[col44].values[p])) for p in (TR_IDX, VA_IDX, TEST_IDX)]},
                       index=["training (augmented)" if AUGMENT else "training", "validation", "test (cleaned, block 4.1.2)"])
display(parts44.style.set_caption(f"{LEVEL_44}: {len(LABELS)} classes, input {INPUT_SIZE} × {INPUT_SIZE} px"))
i0 = TR_IDX[0]
fig, ax = plt.subplots(1, 8, figsize=(18, 2.6))
ax[0].imshow(image_for(i0, INPUT_SIZE))
ax[0].set_title(f"original: {mv.table[col44].values[i0]}", fontsize=8.5)
for k in range(1, 8):
    t_ = AUG(image_for(i0, int(INPUT_SIZE * 1.15))).numpy().transpose(1, 2, 0) * aitbc.IMAGENET_STD + aitbc.IMAGENET_MEAN
    ax[k].imshow(np.clip(t_, 0, 1))
    ax[k].set_title(f"augmented {k}", fontsize=8.5)
for a in ax:
    a.axis("off")
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "lesson44_augmentation.png"), dpi=80)
plt.show()

## Task 4.4.2. Stage 1: frozen network and a new output layer

**Method.** The output layer of MobileNetV3-small for the 1000 classes of ImageNet is replaced by a new one: dropout (a random 20 % of the 576 features are set to zero during the training) and a linear layer of $576 \cdot K + K$ parameters ($K$ — number of classes) with softmax. The convolutional part is frozen: its features are computed once for every training and validation image (without augmentation), after which an epoch of the training of the new layer takes a fraction of a second. The loss is the cross-entropy with label smoothing 0.1 (the target probability of the true class is 0.9 + 0.1/K instead of 1), the optimiser is AdamW.

**Parameters.** `EPOCHS_1` — epochs of the training of the new layer; `LR_HEAD` — its learning rate.

**Expected result.** The curves of the loss on the training part and of the accuracy on the validation part, the number of trained parameters, the accuracy of stage 1 on the validation part.

**What to observe.** Stage 1 trains 4039 parameters (576 · 7 + 7) of 2.5 million. The features of 3569 training and validation images are computed in 10–20 s on a processor, 30 epochs of the new layer take about 2 s. The validation accuracy reaches about 0.60 at 160 px and 0.64 at 224 px.

**Control question 8.** How many parameters are trained at stage 1 for the 7 groups, and why can they be trained on the features computed once?

In [ ]:
#@title 4.4.2 Stage 1: new output layer on the frozen network (code shown)
EPOCHS_1 = 30 #@param {type:"slider", min:5, max:60, step:5}
LR_HEAD = 0.001 #@param [0.0003, 0.001, 0.003] {type:"raw"}

from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from sklearn.metrics import accuracy_score, f1_score
torch.manual_seed(0)
NET = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)
NET.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(576, len(LABELS)))
NET = NET.to(DEVICE).eval()
for p in NET.features.parameters():
    p.requires_grad = False


def features_of(idx):
    with torch.no_grad():
        return torch.cat([NET.avgpool(NET.features(x)).flatten(1) for x, _ in batches(idx, train=False)])


t0 = time.time()
F_tr, F_va = features_of(TR_IDX), features_of(VA_IDX)
y_tr, y_va = label_of(TR_IDX).to(DEVICE), label_of(VA_IDX).to(DEVICE)
t_feat = time.time() - t0
opt = torch.optim.AdamW(NET.classifier.parameters(), lr=LR_HEAD, weight_decay=1e-4)
lossf = nn.CrossEntropyLoss(label_smoothing=0.1)
hist1 = []
t0 = time.time()
for ep in range(EPOCHS_1):
    NET.classifier.train()
    perm = torch.randperm(len(F_tr), device=DEVICE)
    tot = 0.0
    for b in range(0, len(F_tr), 64):
        ii = perm[b:b + 64]
        opt.zero_grad()
        loss = lossf(NET.classifier(F_tr[ii]), y_tr[ii])
        loss.backward()
        opt.step()
        tot += loss.item() * len(ii)
    NET.classifier.eval()
    with torch.no_grad():
        acc_va = (NET.classifier(F_va).argmax(1) == y_va).float().mean().item()
    hist1.append((tot / len(F_tr), acc_va))
t_head = time.time() - t0
HEAD_STAGE1 = {k: v.detach().clone() for k, v in NET.classifier.state_dict().items()}
h1 = np.array(hist1)
fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
ax[0].plot(range(1, EPOCHS_1 + 1), h1[:, 0])
ax[0].set_xlabel("epoch")
ax[0].set_ylabel("loss, training part")
ax[1].plot(range(1, EPOCHS_1 + 1), h1[:, 1], color="#2e7d32")
ax[1].set_xlabel("epoch")
ax[1].set_ylabel("accuracy, validation part")
for a in ax:
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()
n_head = sum(p.numel() for p in NET.classifier.parameters())
print(f"Features of {len(TR_IDX) + len(VA_IDX)} images: {t_feat:.0f} s on {DEVICE}; {EPOCHS_1} epochs of the new layer: {t_head:.1f} s; "
      f"trained parameters: {n_head} (576 · {len(LABELS)} + {len(LABELS)}); accuracy on the validation part: {h1[-1, 1]:.3f}")
STAGE1_VAL = float(h1[-1, 1])

## Task 4.4.3. Stage 2: fine-tuning of the last blocks

**Method.** The last `UNFREEZE_BLOCKS` blocks of the convolutional part are unfrozen and trained together with the output layer of stage 1 on augmented images. The learning rate of the blocks (`LR_BACKBONE`) is about ten times smaller than that of the output layer: the features of ImageNet must be adjusted, not destroyed. The learning rate decreases over the epochs by a cosine law; the batch normalisation layers of the frozen blocks keep the statistics of ImageNet (`eval` mode). After every epoch the accuracy on the validation part is measured, and the weights of the best epoch are kept.

**Parameters.** `UNFREEZE_BLOCKS` — 0 (only the output layer, now on augmented images) … 13 (the whole network); `EPOCHS_2`; `LR_BACKBONE`.

**Expected result.** The number of trained parameters, the time of an epoch and the estimated time of the stage (printed after the first epoch), the curve of the validation accuracy, the comparison with stage 1.

**What to observe.** With 4 unfrozen blocks 740 527 parameters are trained. On a processor of 16 cores an epoch takes 15 s at 160 px and 27 s at 224 px (in Colab without a GPU several times longer, with a T4 GPU a few seconds). After 3 epochs at 160 px the validation accuracy changes little (0.599 → 0.603); with 224 px and 6 epochs it rises from 0.644 to 0.688. Without augmentation and with all 13 blocks unfrozen on the 24 types the network fits the training images and the test accuracy does not improve (0.317 → 0.316).

**Control question 9.** Why are the last blocks of the network unfrozen and not the first ones, and what happens with a large learning rate of the unfrozen blocks?

In [ ]:
#@title 4.4.3 Stage 2: fine-tuning with augmentation (code shown)
UNFREEZE_BLOCKS = 4 #@param {type:"slider", min:0, max:13, step:1}
EPOCHS_2 = 3 #@param {type:"slider", min:1, max:10, step:1}
LR_BACKBONE = 0.0001 #@param [0.00003, 0.0001, 0.0003] {type:"raw"}

import copy
NET.classifier.load_state_dict(HEAD_STAGE1)
for p in NET.features.parameters():
    p.requires_grad = False
blocks = list(NET.features)[len(NET.features) - UNFREEZE_BLOCKS:] if UNFREEZE_BLOCKS else []
for blk in blocks:
    for p in blk.parameters():
        p.requires_grad = True
groups = [{"params": NET.classifier.parameters(), "lr": 3 * LR_BACKBONE if UNFREEZE_BLOCKS else LR_HEAD}]
if UNFREEZE_BLOCKS:
    groups.append({"params": [p for blk in blocks for p in blk.parameters()], "lr": LR_BACKBONE})
opt = torch.optim.AdamW(groups, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS_2)
n_train = sum(p.numel() for p in NET.parameters() if p.requires_grad)
rng44 = np.random.default_rng(0)
best = (STAGE1_VAL, copy.deepcopy(NET.state_dict()), 0)
hist2, t_ep = [], []
for ep in range(EPOCHS_2):
    t0 = time.time()
    NET.train()
    for m in NET.features.modules():                    # frozen batch normalisation keeps the statistics of ImageNet
        if isinstance(m, nn.BatchNorm2d) and not any(p.requires_grad for p in m.parameters()):
            m.eval()
    for x, yb in batches(rng44.permutation(TR_IDX), train=True):
        opt.zero_grad()
        lossf(NET(x), yb).backward()
        opt.step()
    sched.step()
    NET.eval()
    with torch.no_grad():
        pv = torch.cat([NET(x).argmax(1) for x, _ in batches(VA_IDX, train=False)])
    acc_va = (pv == y_va).float().mean().item()
    t_ep.append(time.time() - t0)
    hist2.append(acc_va)
    if acc_va > best[0]:
        best = (acc_va, copy.deepcopy(NET.state_dict()), ep + 1)
    if ep == 0:
        print(f"Epoch 1: {t_ep[0]:.0f} s on {DEVICE}; estimated time of the stage: {t_ep[0] * EPOCHS_2 / 60:.1f} min")
NET.load_state_dict(best[1])
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(range(0, EPOCHS_2 + 1), [STAGE1_VAL] + hist2, "o-")
ax.set_xlabel("epoch of stage 2 (0 — after stage 1)")
ax.set_ylabel("accuracy, validation part")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
display(pd.DataFrame({"value": [UNFREEZE_BLOCKS, f"{n_train:,}".replace(",", " "), f"{np.mean(t_ep):.0f} s ({DEVICE})",
                                f"{STAGE1_VAL:.3f}", f"{max(hist2):.3f} (epoch {int(np.argmax(hist2)) + 1})",
                                f"epoch {best[2]}" if best[2] else "stage 1 (fine-tuning did not improve)"]},
                     index=["unfrozen blocks", "trained parameters", "time of an epoch", "validation accuracy after stage 1",
                            "best validation accuracy of stage 2", "weights kept"]))
torch.save({"state_dict": NET.state_dict(), "labels": LABELS, "input_size": INPUT_SIZE, "level": LEVEL_44},
           os.path.join(MODEL_DIR, "military_vehicles_mobilenetv3_finetuned.pt"))

## Task 4.4.4. Evaluation of the model and Grad-CAM

**Evaluation.** The models of stage 1 and stage 2 are applied once to the test part; the accuracy, the macro F1 (Lesson 4.1), the metrics of every class and the confusion matrix are computed.

**Grad-CAM** (gradient-weighted class activation mapping) shows which regions of the image raised the score $z_c$ of the recognised class $c$. For the feature maps $A^k$ of the last block (5 × 5 or 7 × 7 cells):
$$\alpha_k = \frac{1}{Z}\sum_{i,j}\frac{\partial z_c}{\partial A^k_{ij}},\qquad L_c = \operatorname{ReLU}\Big(\sum_k \alpha_k A^k\Big),$$
the map $L_c$ is enlarged to the size of the image and shown over it. A correct decision based on the vehicle covers the hull, the turret, the launcher; a decision based on the background, a caption or a frame of a picture reveals a defect of the data.

**Parameters.** `N_CAM` — number of test images with Grad-CAM (half correct, half wrong).

**Expected result.** The table *stage 1 — stage 2* on the test part, the metrics of the classes, the confusion matrix and the Grad-CAM pictures.

**What to observe.** On the test part stage 1 gives 0.583 and stage 2 0.592 at 160 px with 3 epochs; at 224 px with 6 epochs 0.622 and 0.656 (macro F1 0.586 → 0.622), against 0.632 for the frozen network with logistic regression in block 4.2.5. The worst group is artillery: the self-propelled howitzer 2S19 is built on a tank chassis and is often recognised as a tank, the towed D-30 is small in the picture. The Grad-CAM of correct decisions covers the hull, the barrel or the launcher. Many errors fall on paper models, drawings and boxes of model kits; with the default parameters the gallery shows a drawing of the light tank T-80 of the Second World War under the label of the modern main battle tank T-80, recognised as an armoured personnel carrier: an error of the data, not of the network.

**Control question 10.** The Grad-CAM of a correctly recognised "missile system" covers a caption on the picture and not the launcher. Is this model good, and what must be done with the data?

In [ ]:
#@title 4.4.4 Test part, confusion matrix and Grad-CAM (code shown)
N_CAM = 8 #@param [4, 8, 12] {type:"raw"}

from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
y_te = label_of(TEST_IDX).numpy()
NET.eval()
with torch.no_grad():
    p2 = torch.cat([NET(x).argmax(1) for x, _ in batches(TEST_IDX, train=False)]).cpu().numpy()
# stage 1 model on the test part: the network with the weights of ImageNet and the output layer of stage 1
net1 = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)
net1.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(576, len(LABELS)))
net1.classifier.load_state_dict(HEAD_STAGE1)
net1 = net1.to(DEVICE).eval()
with torch.no_grad():
    p1 = torch.cat([net1(x).argmax(1) for x, _ in batches(TEST_IDX, train=False)]).cpu().numpy()
del net1
res = pd.DataFrame({"accuracy": [accuracy_score(y_te, p1), accuracy_score(y_te, p2)],
                    "F1 (macro)": [f1_score(y_te, p1, average="macro"), f1_score(y_te, p2, average="macro")]},
                   index=["stage 1: frozen network + new layer", f"stage 2: fine-tuning of {UNFREEZE_BLOCKS} blocks, {EPOCHS_2} epochs"])
display(res.style.format(precision=3).set_caption(f"Test part: {len(TEST_IDX)} images, {LEVEL_44}, input {INPUT_SIZE} px"))
P, R, F1, S = precision_recall_fscore_support(y_te, p2, labels=range(len(LABELS)), zero_division=0)
per = pd.DataFrame({"precision": P, "recall": R, "F1": F1, "test images": S}, index=LABELS).sort_values("F1")
display(per.style.format(precision=3).set_caption("Metrics of the classes (stage 2)") if len(per) <= 8
        else pd.concat([per.head(4), per.tail(4)]).style.format(precision=3).set_caption("4 worst and 4 best classes (stage 2)"))
cm = confusion_matrix(y_te, p2, labels=range(len(LABELS)))
cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(0.45 * len(LABELS) + 4, 0.4 * len(LABELS) + 3))
ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
if len(LABELS) <= 8:
    for i in range(len(LABELS)):
        for j in range(len(LABELS)):
            ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=8, color="w" if cmn[i, j] > 0.5 else "k")
ax.set_xticks(range(len(LABELS)))
ax.set_xticklabels(LABELS, rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(len(LABELS)))
ax.set_yticklabels(LABELS, fontsize=8)
ax.set_xlabel("recognised class")
ax.set_ylabel("true class")
ax.set_title(f"stage 2: accuracy {accuracy_score(y_te, p2):.3f}", fontsize=9.5)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "lesson44_confusion.png"), dpi=80)
plt.show()
ok, bad = np.where(p2 == y_te)[0], np.where(p2 != y_te)[0]
rng_cam = np.random.default_rng(1)                       # random test images of all classes
pick = list(rng_cam.choice(ok, min(N_CAM // 2, len(ok)), replace=False)) + list(rng_cam.choice(bad, min(N_CAM - N_CAM // 2, len(bad)), replace=False))
fig, ax = plt.subplots(2, len(pick), figsize=(2.3 * len(pick), 4.9))
for j, k in enumerate(pick):
    i = TEST_IDX[k]
    x = torch.from_numpy(aitbc.imagenet_batch([image_for(i, INPUT_SIZE)], INPUT_SIZE)).to(DEVICE)
    cam, c, prob = aitbc.grad_cam(NET, x, NET.features[-1])
    img = image_for(i, INPUT_SIZE)
    ax[0, j].imshow(img)
    ax[0, j].set_title(f"true: {LABELS[y_te[k]]}", fontsize=7.5)
    ax[1, j].imshow(img)
    ax[1, j].imshow(cam, cmap="jet", alpha=0.45)
    ax[1, j].set_title(f"recognised: {LABELS[c]} ({prob[c]:.2f})", fontsize=7.5, color="k" if c == y_te[k] else "#c62828")
for a in ax.ravel():
    a.axis("off")
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "lesson44_grad_cam.png"), dpi=80)
plt.show()
RESULT_44 = {"level": LEVEL_44, "input_size": INPUT_SIZE, "unfreeze_blocks": UNFREEZE_BLOCKS, "epochs_1": EPOCHS_1, "epochs_2": EPOCHS_2,
             "test_accuracy_stage1": round(float(accuracy_score(y_te, p1)), 3), "test_accuracy_stage2": round(float(accuracy_score(y_te, p2)), 3),
             "test_f1_stage2": round(float(f1_score(y_te, p2, average="macro")), 3)}

## Task 4.4.5. Export to ONNX and check of the inference

**Method.** The fine-tuned network with a softmax at the output is exported to ONNX by the classical exporter of PyTorch (opset 17; in Lesson 4.2 the file of the new exporter was not read by OpenCV 4.12 and gave wrong classes in OpenCV 5.0). The file is executed by ONNX Runtime and by OpenCV DNN on `N_CHECK` test images; the largest difference of the probabilities with PyTorch and the agreement of the classes are computed, and the time per image is measured for a batch of 1 image (a camera frame on the onboard computer) and of 16 images. The preprocessing is written in the file of labels.

**Parameters.** `N_CHECK` — test images of the check; `DYNAMIC_BATCH` — free number of images in the input.

**Expected result.** The ONNX file and its file of labels in `models`, the table of the check, the time per image.

**What to observe.** The ONNX file of 3.6 MB gives the same probabilities as PyTorch up to about 5 · 10⁻⁶ in ONNX Runtime and in OpenCV DNN, and the classes agree for all checked images. One image takes about 1 ms in ONNX Runtime on the processor (about 0.5 ms per image in a batch of 16).

**Control question 11.** The onboard computer gives the frames in BGR order (OpenCV) and without normalisation. What will the model recognise, and where must the correction be made?

In [ ]:
#@title 4.4.5 ONNX file of the fine-tuned model, ONNX Runtime and OpenCV DNN (code shown)
N_CHECK = 128 #@param {type:"slider", min:32, max:512, step:32}
DYNAMIC_BATCH = True #@param {type:"boolean"}

import importlib.util, subprocess
need = [p for p, mod in (("onnx", "onnx"), ("onnxruntime", "onnxruntime")) if importlib.util.find_spec(mod) is None]
if need:
    print("Installing:", ", ".join(need))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + need, check=True)
import onnx
import onnxruntime as ort


class WithSoftmax(nn.Module):
    def __init__(self, net):
        super().__init__()
        self.net = net

    def forward(self, image):
        return torch.softmax(self.net(image), dim=1)


model = WithSoftmax(copy.deepcopy(NET).cpu()).eval()
onnx_path = os.path.join(MODEL_DIR, "military_vehicles_mobilenetv3_finetuned.onnx")
kw = dict(input_names=["image"], output_names=["probabilities"], opset_version=17)
if DYNAMIC_BATCH:
    kw["dynamic_axes"] = {"image": {0: "batch"}, "probabilities": {0: "batch"}}
dummy = torch.from_numpy(aitbc.imagenet_batch([image_for(TEST_IDX[0], INPUT_SIZE)], INPUT_SIZE))
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    torch.onnx.export(model, (dummy,), onnx_path, dynamo=False, **kw)
onnx.checker.check_model(onnx.load(onnx_path))
LABELS_JSON = {"labels": LABELS, "level": LEVEL_44, "input": "image: N x 3 x S x S float32, RGB order", "size": INPUT_SIZE,
               "resize": "cv2.resize(..., (S, S), interpolation=cv2.INTER_AREA)", "scale": "x / 255",
               "mean": aitbc.IMAGENET_MEAN.tolist(), "std": aitbc.IMAGENET_STD.tolist(), "output": "probabilities N x K (softmax)",
               "model": f"MobileNetV3-small, ImageNet weights, new output layer + fine-tuning of {UNFREEZE_BLOCKS} blocks",
               "test_accuracy": RESULT_44["test_accuracy_stage2"], "opset": 17}
json.dump(LABELS_JSON, open(onnx_path.replace(".onnx", "_labels.json"), "w", encoding="utf-8"), indent=2, ensure_ascii=False)
pos_chk = np.sort(np.random.default_rng(0).choice(len(TEST_IDX), min(N_CHECK, len(TEST_IDX)), replace=False))   # random test images
idx = TEST_IDX[pos_chk]
xb = aitbc.imagenet_batch([image_for(i, INPUT_SIZE) for i in idx], INPUT_SIZE)
with torch.no_grad():
    p_t = model(torch.from_numpy(xb)).numpy()
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
p_o = sess.run(None, {"image": xb})[0] if DYNAMIC_BATCH else np.concatenate([sess.run(None, {"image": xb[k:k + 1]})[0] for k in range(len(xb))])


def time_ms(fn, n):
    t0 = time.time()
    fn()
    return (time.time() - t0) / n * 1000


t1 = time_ms(lambda: [sess.run(None, {"image": xb[k:k + 1]}) for k in range(32)], 32)
t16 = time_ms(lambda: [sess.run(None, {"image": xb[k:k + 16]}) for k in range(0, 64, 16)], 64) if DYNAMIC_BATCH else float("nan")
rows = {"file": f"{onnx_path} ({os.path.getsize(onnx_path) / 2 ** 20:.2f} MB)",
        "ONNX Runtime: largest difference of the probabilities": f"{np.abs(p_o - p_t).max():.2e}",
        "ONNX Runtime: agreement of the classes": f"{np.mean(p_o.argmax(1) == p_t.argmax(1)):.3f}",
        "ONNX Runtime (CPU): time per image, batch 1 / batch 16, ms": f"{t1:.1f} / {t16:.1f}"}
try:
    net_cv = cv2.dnn.readNetFromONNX(onnx_path)
    p_c = []
    for k in range(len(xb)):
        net_cv.setInput(xb[k:k + 1])
        p_c.append(net_cv.forward())
    p_c = np.concatenate(p_c)
    rows["OpenCV DNN: largest difference; agreement of the classes"] = f"{np.abs(p_c - p_t).max():.2e}; {np.mean(p_c.argmax(1) == p_t.argmax(1)):.3f}"
except Exception as e:
    rows["OpenCV DNN"] = f"not read by OpenCV {cv2.__version__}: {str(e).splitlines()[0][:120]}"
rows[f"accuracy of the ONNX model on {len(idx)} random test images"] = f"{np.mean(p_o.argmax(1) == y_te[pos_chk]):.3f}"
display(pd.DataFrame({"value": list(rows.values())}, index=list(rows)))
update_project_json("4.4", {"lesson_4_4": RESULT_44 | {"onnx": os.path.relpath(onnx_path, PROJECT_ROOT),
                                                       "max_difference_onnxruntime": float(np.abs(p_o - p_t).max())}})

## Task 4.4.6. Recognition of a new image by the ONNX model

**Method.** The ONNX file and its file of labels are loaded as on another computer, without the network in memory: the image is prepared exactly as written in the file of labels (RGB order, resize to $S \times S$, scale 1/255, normalisation by the mean and the standard deviation), then ONNX Runtime gives the probabilities of the classes.

**Parameters.** `SOURCE` — a test image or an own image (in Colab a file is uploaded from the computer); `IMAGE_NUMBER` — number of the test image.

**Expected result.** The image with the three most probable classes and their probabilities.

**Control question 12.** Which conditions of the use (distance, angle of view, thermal camera, camouflage) differ from the training data of the model, and how will this affect its decisions?

In [ ]:
#@title 4.4.6 Recognition by the ONNX file (code shown)
SOURCE = "test image" #@param ["test image", "own image (upload)"]
IMAGE_NUMBER = 5 #@param {type:"integer"}

meta44 = json.load(open(onnx_path.replace(".onnx", "_labels.json"), encoding="utf-8"))
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])


def prepare(rgb, meta):
    s = meta["size"]
    x = cv2.resize(rgb, (s, s), interpolation=cv2.INTER_AREA).astype(np.float32) / 255.0
    x = (x - np.array(meta["mean"], np.float32)) / np.array(meta["std"], np.float32)
    return x.transpose(2, 0, 1)[None].copy()


img, title = None, ""
if SOURCE.startswith("own") and IN_COLAB:
    from google.colab import files
    up = files.upload()
    if up:
        name, data = next(iter(up.items()))
        img = cv2.cvtColor(cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        title = name
if img is None:
    if SOURCE.startswith("own"):
        print("Uploading a file is possible in Google Colab; a test image is used.")
    i = TEST_IDX[IMAGE_NUMBER % len(TEST_IDX)]
    img = mv.image(i)
    title = f"test image {i}: true class {mv.table[col44].values[i]}"
prob = sess.run(None, {"image": prepare(img, meta44)})[0][0]
top = np.argsort(-prob)[:3]
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [1.2, 1]})
ax[0].imshow(img)
ax[0].set_title(title, fontsize=9)
ax[0].axis("off")
ax[1].barh([meta44["labels"][j] for j in top[::-1]], prob[top[::-1]], color="#1565c0")
ax[1].set_xlim(0, 1)
ax[1].set_xlabel("probability (ONNX Runtime)")
plt.tight_layout()
plt.show()
print("Three most probable classes:", "; ".join(f"{meta44['labels'][j]} {prob[j]:.2f}" for j in top))

### Report of Lesson 4.4

Double-click this cell and write the answers.

1. Parts of the data, input size, augmentation; answer to control question 7:
2. Stage 1: trained parameters, epochs, validation accuracy; answer to control question 8:
3. Stage 2: unfrozen blocks, trained parameters, time of an epoch, validation accuracy; answer to control question 9:
4. Test part: stage 1 and stage 2, worst classes, Grad-CAM of correct and wrong decisions; answer to control question 10:
5. ONNX file, differences of the probabilities, time per image; answer to control question 11:
6. Recognition of a new image; answer to control question 12:

---
## Self-check of Topic 4

1. Which data set is needed to train a verifier of thermal candidates, and why must the candidates of the training be produced by the same detector as in use?
2. How are the training, validation and test parts formed for frames of reconnaissance flights, and what is estimated on each of them?
3. Why is the average precision a better description of a verifier than the accuracy, and how is the working threshold chosen?
4. What do the two stages of transfer learning change in the network, and when is the fine-tuning worth its cost?
5. Which checks are made before an ONNX model is put on the onboard computer?

## Data sources and licences

The course library reads the files directly from the repositories of the data sets; the data are not redistributed with the notebook. When images or results are published, the data sets are cited.

1. **HIT-UAV.** Suo J., Wang T., Zhang X., Chen H., Zhou W., Shi W. HIT-UAV: A high-altitude infrared thermal dataset for Unmanned Aerial Vehicle-based object detection. *Scientific Data*, 10, 227 (2023). https://doi.org/10.1038/s41597-023-02066-6 · Data: https://github.com/suojiashun/HIT-UAV-Infrared-Thermal-Dataset · Licence CC BY 4.0.
2. **Military vehicles.** Kricheli J. S., Vo K., Datta A., Ozgur S., Shakarian P. Error Detection and Constraint Recovery in Hierarchical Multi-Label Classification without Prior Knowledge. arXiv:2407.15192 (2024) · Data: https://huggingface.co/datasets/leibnitz-lab/military_vehicles · Licence declared by the authors: MIT; the photographs were collected from web sites, the rights to the individual photographs are not stated: use them only for training and research.
3. **ImageNet weights of MobileNetV3-small** — torchvision (BSD-3-Clause); Howard A. et al. Searching for MobileNetV3. *ICCV 2019*, 1314–1324.
4. **HOG.** Dalal N., Triggs B. Histograms of Oriented Gradients for Human Detection. *CVPR 2005*, 886–893. **Grad-CAM.** Selvaraju R. R. et al. Grad-CAM: Visual Explanations from Deep Networks via Gradient-Based Localization. *ICCV 2017*, 618–626.

**Next notebook.** Notebook 08 — Topic 5, Lessons 5.1 (lecture) and 5.2 (group class): object detection by YOLO, the metrics IoU, confidence and mAP, small targets; segmentation of the traversable area for a ground robot, tracking of objects (ByteTrack), the video-analysis pipeline and the requirements of real time.